In [8]:
# updated, inclusing disoder 
import numpy as np
import scipy.linalg as la
from scipy.sparse import diags, block_diag, lil_matrix, csc_matrix, identity, issparse, eye as sparse_eye
from scipy.sparse.linalg import inv as sparse_inv
from scipy.sparse.linalg import spsolve, factorized, inv
import scipy.sparse as sp
from joblib import Parallel, delayed
import pandas as pd
import os
import time
from datetime import datetime
import logging
import json
import matplotlib.pyplot as plt
import hashlib
import functools
from collections import defaultdict

# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class HamiltonianBuilder:
    def __init__(self, params):
        # 验证disorder类型
        if 'disorder_type' in params:
            valid_disorders = ['gaussian', 'smooth', 'random_typeI', 'random_typeII', 'nonhermitian', 'none', 'from_file']
            if params['disorder_type'] not in valid_disorders:
                raise ValueError(f"non-effective model: {params['disorder_type']}. must be: {', '.join(valid_disorders)}")
        
        # 物理参数
        self.t = params['t']  # 跳跃强度 (meV)
        self.delta = params['delta']  # 超导能隙 (meV)
        self.mu = params['mu']  # 化学势 (meV)
        self.mu_lead = params['mu_lead']  # 引线化学势 (meV)
        self.B = params['B'] 
        self.alpha = params['alpha']  # 自旋轨道耦合 (meV)
             
        # 结参数
        self.N_SC = params['N_SC']  # 超导区长度
        self.N_junction = params['N_junction']  # 结区长度
        self.v_tau = params['v_tau']  # 隧穿强度
        
        # Disorder参数
        self.disorder_type = params.get('disorder_type', 'none')  # Disorder类型
        # Gaussian 
        self.V_gau = params.get('Vdis_gau', 0.0)  # Disorder强度
        self.X_gau = params.get('Xdis_gau', 50)  # Disorder中心
        self.decayL_gau = params.get('decayL_gau', 50)  # Disorder衰减长度     
        # smooth
        self.decayL_smooth = params.get('decayL_smooth', 50)  # Smooth disorder参数
        self.Vmax_smooth = params.get('Vdis_smooth', 0.3)  # Max disorder potential
        self.Vd_smooth = params.get('Vd_smooth', 0.0)  # Disorder深度   
        # random typeI
        self.Num_Rd1 = params.get('N_imp1', 52)  # Number of impurities
        self.lambda_Rd1 = params.get('lambda_imp1', 18.0)  # Characteristic length
        self.V0_Rd1 = params.get('V0_imp1', 0.0)  # Disorder amplitude (meV)  
        # random typeII
        self.Nd_Rd2 = params.get('Nd_imp2', 10.0)  # Impurity density (per micron)
        self.lambda_Rd2 = params.get('lambda_imp2', 20.0)  # Characteristic length (nm)
        self.V0_Rd2 = params.get('V0_imp2', 0.0)  # Disorder amplitude (meV)
        self.a0 = params.get('a0', 10.0)  # Lattice constant (nm)
        # Nonhermitian
        self.nonH_eta = params.get('nonH_eta', 1.5e-3)
        # From file
        self.disorder_file = params.get('disorder_file', 'disorder_data.txt')  # File to read disorder from
        self.disorder_strength = params.get('disorder_strength', 1.0)  
        
        # Floquet参数
        self.max_sidebands = params['max_sidebands']
        
        # 初始化Pauli矩阵
        self._init_pauli_matrices()
        
        # 计算系统总长度
        self.total_sites = 2 * self.N_SC + self.N_junction
        
        # 初始化无序分布
        self.disorder_distribution = self._calculate_disorder_distribution()
        
        # 稀疏矩阵配置
        self.use_sparse = params.get('use_sparse', True)  # 默认启用稀疏矩阵
        self.sparse_threshold = params.get('sparse_threshold', 0.3)  # 稀疏度阈值
        self.sparse_depth_limit = params.get('sparse_depth_limit', 5)  # 最大稀疏递归深度

        # 新增缓存变量
        self.prev_sidebands = None
        self.cached_g0_inv_FKs = None
        self.cached_hop_FKs = None
        
        # 增量扩建缓存
        self.incremental_cache = {}
        
        logging.info(f"HamiltonianBuilder initialized, disorder type: {self.disorder_type}, Sparse mode: {self.use_sparse}")

    def _init_pauli_matrices(self):
        """初始化Pauli矩阵"""
        # 自旋空间
        self.sigma_0 = np.eye(2)
        self.sigma_x = np.array([[0, 1], [1, 0]])
        self.sigma_y = np.array([[0, -1j], [1j, 0]])
        self.sigma_z = np.array([[1, 0], [0, -1]])
        
        # Nambu空间
        self.tau_0 = np.eye(2)
        self.tau_x = np.array([[0, 1], [1, 0]])
        self.tau_y = np.array([[0, -1j], [1j, 0]])
        self.tau_z = np.array([[1, 0], [0, -1]])
        self.tau_plus = (self.tau_x + 1j * self.tau_y) / 2
        self.tau_minus = (self.tau_x - 1j * self.tau_y) / 2
        self.tau_e = np.array([[1, 0], [0, 0]])  # 电子块
        self.tau_h = np.array([[0, 0], [0, 1]])  # 空穴块

    def _calculate_disorder_distribution(self):
        """计算无序分布"""
        # 初始化数组
        disorder_array = np.zeros(self.total_sites)
        
        # 根据disorder类型生成无序
        if self.disorder_type == 'smooth':
            # 平滑无序
            for i in range(self.total_sites):
                if i < self.N_SC: 
                    disorder_array[i] = self.Vmax_smooth * \
                        (1 - self.Vd_smooth * np.exp(i / self.decayL_smooth))
                elif i >= self.N_junction + self.N_SC:
                    disorder_array[i] = self.Vmax_smooth * \
                        (1 - self.Vd_smooth * np.exp(-(i - self.N_junction) / self.decayL_smooth))
                else: # inside junction 
                    disorder_array[i] = self.Vmax_smooth * (1 - self.Vd_smooth)
        
        elif self.disorder_type == 'gaussian':
            # 高斯无序
            for i in range(self.total_sites):
                disorder_array[i] = self.V_gau * np.exp(-(i - self.X_gau)**2 / self.decayL_gau**2)
        
        elif self.disorder_type == 'nonhermitian':
            # 非厄米无序
            for i in range(self.total_sites):
                disorder_array[i] = -1j * self.nonH_eta
        
        elif self.disorder_type == 'random_typeI':
            # 随机类型I
            disorder_array = self._generate_random_typeI_disorder(range(self.total_sites))
        
        elif self.disorder_type == 'random_typeII':
            # 随机类型II
            disorder_array = self._generate_random_typeII_disorder(range(self.total_sites))
        
        elif self.disorder_type == 'from_file':
            # 从文件读取无序
            disorder_array = self._load_disorder_from_file()
        elif self.disorder_type == 'none':
            for i in range(self.total_sites):
                disorder_array[i] = 0
            
        return disorder_array

    def _load_disorder_from_file(self):
        """从文件加载无序分布"""
        try:
            # 尝试从不同文件格式加载
            if self.disorder_file.endswith('.csv'):
                df = pd.read_csv(self.disorder_file)
                if 'disorder_value' in df.columns:
                    disorder_array = df['disorder_value'].values
                else:
                    # 假设第一列包含无序值
                    disorder_array = df.iloc[:, 0].values
            elif self.disorder_file.endswith('.txt') or self.disorder_file.endswith('.dat'):
                disorder_array = np.loadtxt(self.disorder_file)
            elif self.disorder_file.endswith('.npy'):
                disorder_array = np.load(self.disorder_file)
            else:
                # 尝试作为文本文件加载
                disorder_array = np.loadtxt(self.disorder_file)
            
            # 检查数组长度是否匹配总格点数
            if len(disorder_array) != self.total_sites:
                logging.warning(f"Disorder array length ({len(disorder_array)}) doesn't match total sites ({self.total_sites}). Truncating or padding.")
                if len(disorder_array) > self.total_sites:
                    disorder_array = disorder_array[:self.total_sites]
                else:
                    disorder_array = np.pad(disorder_array, (0, self.total_sites - len(disorder_array)), 'constant')
            
            logging.info(f"Loaded disorder from file: {self.disorder_file}")
            return disorder_array
            
        except Exception as e:
            logging.error(f"Failed to load disorder from file {self.disorder_file}: {str(e)}")
            return np.zeros(self.total_sites)

    def _generate_random_typeI_disorder(self, sites):
        """生成随机类型I无序"""
        disorder_array = np.zeros(self.total_sites)
        rng = np.random.default_rng()
        
        # 仅在选定区域生成杂质
        impurity_positions = rng.choice(sites, self.Num_Rd1)
        impurity_signs = np.array([(-1)**(n) for n in range(self.Num_Rd1)])
        
        # 计算未归一化势
        S = np.zeros(self.total_sites)
        for i in sites:
            for j in range(self.Num_Rd1):
                distance = abs(i - impurity_positions[j])
                S[i] += impurity_signs[j] * np.exp(-distance / self.lambda_Rd1)
        
        # 归一化
        S_avg = np.mean(S)
        S_centered = S - S_avg
        variance = np.mean(S_centered**2)
        N_V = np.sqrt(variance) / self.V0_Rd1 if variance > 0 else 1.0
        
        # 应用到无序数组
        for i in sites:
            disorder_array[i] = (self.V0_Rd1 / N_V) * S_centered[i]
        
        logging.info(f"Generated charge impurity disorder, impurities: {self.Num_Rd1}")
        return disorder_array

    def _generate_random_typeII_disorder(self, sites):
        """生成随机类型II无序"""
        disorder_array = np.zeros(self.total_sites)
        L_total_nm = (self.total_sites - 1) * self.a0
        
        # 计算选定区域中的杂质数量
        region_length = len(sites) * self.a0 / 1000  # in microns
        N_d = int(self.Nd_Rd2 * region_length)
        
        rng = np.random.default_rng()
        
        # 生成杂质位置
        min_pos = min(sites) * self.a0 if sites else 0
        max_pos = max(sites) * self.a0 if sites else 0
        impurity_positions = rng.uniform(min_pos, max_pos, N_d)
        impurity_amplitudes = rng.normal(0, 1, N_d)
        
        # 计算选定位点的无序
        for i in sites:
            x_i = i * self.a0  # 位点位置 (nm)
            potential_sum = 0.0
            for j in range(N_d):
                distance = abs(x_i - impurity_positions[j])
                potential_sum += impurity_amplitudes[j] * np.exp(-distance / self.lambda_Rd2)
            disorder_array[i] = self.V0_Rd2* potential_sum
        
        logging.info(f"Generated random amplitude disorder, impurities: {N_d}")
        return disorder_array

    def disorder_potential(self, site_index):
        """返回给定位点的无序势矩阵"""
        if 0 <= site_index < len(self.disorder_distribution):
            V_s = self.disorder_distribution[site_index]
            
            if self.disorder_type == 'nonhermitian':
                return V_s * np.kron(self.tau_0, self.sigma_0)
            else:
                return V_s * np.kron(self.tau_z, self.sigma_0)
        else:
            return np.zeros((4, 4))

    def block_hamiltonian(self, phi, region_type):
        """构建区域类型的哈密顿量块"""
        onsite = 2 * self.t - self.mu 
        H00 = onsite * np.kron(self.tau_z, self.sigma_0) + \
              self.B * np.kron(self.tau_z, self.sigma_x) * np.sqrt(self.mu_lead**2 + self.delta**2)
        H01 = -self.t * np.kron(self.tau_z, self.sigma_0) + 1j * self.alpha * np.kron(self.tau_z, self.sigma_y)
        
        if region_type == 'SC':
            H00 = H00 + self.delta * (np.exp(1j * phi) * np.kron(self.tau_plus, 1j*self.sigma_y) + 
                                 np.exp(-1j * phi) * np.kron(self.tau_minus, -1j*self.sigma_y))
            
        return H00, H01

    def construct_hamiltonian(self, phi):
        """构建SNS结的完整哈密顿量"""
        H00_L, H01_L = self.block_hamiltonian(0, 'SC')
        H00_R, H01_R = self.block_hamiltonian(phi, 'SC')
        H00_J, H01_J = self.block_hamiltonian(0, 'NM')
        
        return {
            'H00_L': H00_L, 'H01_L': H01_L,
            'H00_R': H00_R, 'H01_R': H01_R,
            'H00_J': H00_J, 'H01_J': H01_J
        }
    
    def _incremental_expand_slice_ham(self, Vbias, ham, new_max_sidebands):
        """增量扩建Floquet哈密顿量切片"""
        if self.prev_sidebands is None or self.cached_g0_inv_FKs is None or self.cached_hop_FKs is None:
            return self._full_build_slice_ham(Vbias, ham, new_max_sidebands)
            
        old_max_sidebands = self.prev_sidebands
        old_sideN = 2 * old_max_sidebands + 1
        new_sideN = 2 * new_max_sidebands + 1
        dim_old = 4 * old_sideN
        dim_new = 4 * new_sideN
        
        # 获取旧的矩阵列表
        old_g0_inv_FKs = self.cached_g0_inv_FKs
        old_hop_FKs = self.cached_hop_FKs
        
        # 新的矩阵列表
        g0_inv_FKs = []
        hop_FKs = []
        
        SNS_Len = self.N_SC * 2 + self.N_junction
        
        # 扩展每个位点的矩阵
        for site_i in range(SNS_Len):
            old_g0_inv = old_g0_inv_FKs[site_i]
            old_hop = old_hop_FKs[site_i]
            
            # 确定当前位点的哈密顿量块
            if self.N_SC <= site_i < self.N_SC + self.N_junction:
                Ham_onsite = ham['H00_J']
                Ham_hop = ham['H01_J']
                Hop_e = Ham_hop * np.kron(self.tau_e, np.ones((2, 2))) * self.v_tau
                Hop_h = -1 * Ham_hop * np.kron(self.tau_h, np.ones((2, 2))) * self.v_tau
                is_junction = True
            elif site_i < self.N_SC:
                Ham_onsite = ham['H00_L']
                Ham_hop = ham['H01_L']
                is_junction = False
            else:
                Ham_onsite = ham['H00_R']
                Ham_hop = ham['H01_R']
                is_junction = False
            
            # 扩展g0_inv_FK: 块对角矩阵，上下各增加 (new_max_sidebands - old_max_sidebands) 个块
            # 新矩阵的边带范围：[-new_max_sidebands, new_max_sidebands]
            # 旧矩阵的边带范围：[-old_max_sidebands, old_max_sidebands]
            num_bands_to_add = new_max_sidebands - old_max_sidebands
            diag_blocks = []
            
            # 上部新增的块（更负的边带）
            for band_i in range(-new_max_sidebands, -old_max_sidebands):
                diag_block = Ham_onsite - band_i * Vbias * np.eye(4)
                diag_blocks.append(diag_block)
            
            # 中间旧矩阵部分
            if sp.issparse(old_g0_inv):
                # 将旧的块对角矩阵转为块列表
                old_blocks = [old_g0_inv[i*4:(i+1)*4, i*4:(i+1)*4] for i in range(old_sideN)]
                diag_blocks.extend(old_blocks)
            else:
                for n in range(old_sideN):
                    idx = slice(n*4, (n+1)*4)
                    diag_blocks.append(old_g0_inv[idx, idx])
            
            # 下部新增的块（更正边带）
            for band_i in range(old_max_sidebands+1, new_max_sidebands+1):
                diag_block = Ham_onsite - band_i * Vbias * np.eye(4)
                diag_blocks.append(diag_block)
            
            # 构建新的块对角矩阵
            if self.use_sparse and dim_new > 20:
                g0_inv_new = block_diag(diag_blocks, format='csc')
            else:
                g0_inv_new = np.zeros((dim_new, dim_new), dtype=np.complex128)
                for i, block in enumerate(diag_blocks):
                    start_idx = i * 4
                    end_idx = (i+1) * 4
                    g0_inv_new[start_idx:end_idx, start_idx:end_idx] = block
            
            # 扩展hop_FK: 同样需要扩展
            if self.use_sparse and dim_new > 20:
                hop_new = lil_matrix((dim_new, dim_new), dtype=np.complex128)
            else:
                hop_new = np.zeros((dim_new, dim_new), dtype=np.complex128)
            
            # 将旧的跳跃矩阵放入新矩阵的中间部分
            old_start = num_bands_to_add * 4
            old_end = old_start + dim_old
            if sp.issparse(hop_new):
                hop_new[old_start:old_end, old_start:old_end] = old_hop
            else:
                hop_new[old_start:old_end, old_start:old_end] = old_hop
            
            # 对于特殊位点（结区中心），需要添加新的耦合项
            is_special_site = (site_i == (self.N_SC + self.N_junction // 2 - 1) or 
                              site_i == (self.N_SC + self.N_junction // 2))
            
            if is_special_site and is_junction:
                # 注意：新增的边带也需要耦合
                for n in range(new_sideN):
                    band_i = n - new_max_sidebands
                    start_idx = n * 4
                    end_idx = (n+1)*4
                    
                    # 上边带耦合（从当前边带到下一个边带）
                    if band_i != new_max_sidebands and Vbias >= 0:
                        next_idx = (n+1) * 4
                        if next_idx < dim_new:
                            if sp.issparse(hop_new):
                                hop_new[start_idx:end_idx, next_idx:next_idx+4] = Hop_e
                            else:
                                hop_new[start_idx:end_idx, next_idx:next_idx+4] = Hop_e
                    
                    # 下边带耦合（从当前边带到前一个边带）
                    if band_i != -new_max_sidebands and Vbias >= 0:
                        prev_idx = (n-1) * 4
                        if prev_idx >= 0:
                            if sp.issparse(hop_new):
                                hop_new[start_idx:end_idx, prev_idx:prev_idx+4] = Hop_h.conj().T
                            else:
                                hop_new[start_idx:end_idx, prev_idx:prev_idx+4] = Hop_h.conj().T
            else:
                # 非特殊位点，主对角块已经由旧矩阵填充
                pass
            
            if sp.issparse(hop_new):
                hop_new = hop_new.tocsc()
            
            g0_inv_FKs.append(g0_inv_new)
            hop_FKs.append(hop_new)
        
        return g0_inv_FKs, hop_FKs

    def build_slice_ham(self, Vbias, ham, max_sidebands):
        """构建Floquet哈密顿量切片，支持增量扩建"""
        # 检查是否可以使用增量扩建
        if self.prev_sidebands is not None and max_sidebands > self.prev_sidebands and \
                self.cached_g0_inv_FKs is not None and self.cached_hop_FKs is not None:
            logging.info(f"使用增量扩建: {self.prev_sidebands} -> {max_sidebands} 边带")
            g0_inv_FKs, hop_FKs = self._incremental_expand_slice_ham(Vbias, ham, max_sidebands)
        else:
            # 完整构建模式
            g0_inv_FKs, hop_FKs = self._full_build_slice_ham(Vbias, ham, max_sidebands)
        
        # 更新缓存
        self.prev_sidebands = max_sidebands
        self.cached_g0_inv_FKs = g0_inv_FKs
        self.cached_hop_FKs = hop_FKs
        
        return g0_inv_FKs, hop_FKs

    def _full_build_slice_ham(self, Vbias, ham, max_sidebands):
        """完整构建Floquet哈密顿量切片"""
        sideN = 2 * max_sidebands + 1
        dim = 4 * sideN
        g0_inv_FKs = []
        hop_FKs = []
        
        SNS_Len = self.N_SC * 2 + self.N_junction
        SN_Len = self.N_SC + self.N_junction
        
        for site_i in range(SNS_Len):        
            # 确定当前位点的哈密顿量块
            if self.N_SC <= site_i < SN_Len:
                Ham_onsite = ham['H00_J']
                Ham_hop = ham['H01_J']
                Hop_e = Ham_hop * np.kron(self.tau_e, np.ones((2, 2))) * self.v_tau
                Hop_h = -1 * Ham_hop * np.kron(self.tau_h, np.ones((2, 2))) * self.v_tau
                is_junction = True
            elif site_i < self.N_SC:
                Ham_onsite = ham['H00_L']
                Ham_hop = ham['H01_L']
                is_junction = False
            else:
                Ham_onsite = ham['H00_R']
                Ham_hop = ham['H01_R']
                is_junction = False
            
            # 添加无序势
            Ham_onsite = Ham_onsite + self.disorder_strength * self.disorder_potential(site_i)
            
            # 稀疏矩阵构建
            if self.use_sparse and dim > 20:
                # 构建对角块
                diag_blocks = []
                for n in range(sideN):
                    band_i = n - max_sidebands
                    diag_block = Ham_onsite - band_i * Vbias * np.eye(4)
                    diag_blocks.append(diag_block)
                
                # 创建块对角矩阵
                g0_inv_FK0 = block_diag(diag_blocks, format='csc')
                
                # 创建跳跃矩阵
                hop_FK0 = lil_matrix((dim, dim), dtype=np.complex128)
                
                # 填充主对角块
                for n in range(sideN):
                    band_i = n - max_sidebands
                    idx = slice(n * 4, (n + 1) * 4)
                    
                    # 特殊位点处理
                    is_special_site = (site_i == (self.N_SC + self.N_junction // 2 - 1) or 
                                      site_i == (self.N_SC + self.N_junction // 2))
                    
                    if is_special_site and is_junction:
                        # 上边带耦合
                        if band_i != max_sidebands and Vbias >= 0:
                            next_idx = slice((n + 1) * 4, (n + 2) * 4)
                            hop_FK0[idx, next_idx] = Hop_e
                        
                        # 下边带耦合
                        if band_i != -max_sidebands and Vbias >= 0:
                            prev_idx = slice((n - 1) * 4, n * 4)
                            hop_FK0[idx, prev_idx] = Hop_h.conj().T
                    else:
                        hop_FK0[idx, idx] = Ham_hop
                        
                hop_FK0 = hop_FK0.tocsc()
            else:
                # 稠密矩阵路径
                g0_inv_FK0 = np.zeros((dim, dim), dtype=np.complex128)
                hop_FK0 = np.zeros_like(g0_inv_FK0)
                
                for n in range(sideN):
                    band_i = n - max_sidebands
                    idx = slice(n * 4, (n + 1) * 4)
                    g0_inv_FK0[idx, idx] = Ham_onsite - band_i * Vbias * np.eye(4)
                    
                    # 特殊位点处理
                    is_special_site = (site_i == (self.N_SC + self.N_junction // 2 - 1) or 
                                      site_i == (self.N_SC + self.N_junction // 2))
                    
                    if is_special_site and is_junction:
                        if band_i != max_sidebands and Vbias >= 0:
                            next_idx = slice((n + 1) * 4, (n + 2) * 4)
                            hop_FK0[idx, next_idx] = Hop_e
                        if band_i != -max_sidebands and Vbias >= 0:
                            prev_idx = slice((n - 1) * 4, n * 4)
                            hop_FK0[idx, prev_idx] = Hop_h.conj().T
                    else:
                        hop_FK0[idx, idx] = Ham_hop
            
            g0_inv_FKs.append(g0_inv_FK0)
            hop_FKs.append(hop_FK0)
        
        return g0_inv_FKs, hop_FKs

class GreenFunctionCalculator:
    def __init__(self, params):
        # 计算参数
        self.recursion_depth = params['recursion_depth']
        self.eta = params['eta']  # 虚部无穷小
        self.mu_lead = params['mu_lead']  # 引线化学势
        self.delta = params['delta']  # 超导能隙
        self.unit_factor_nA = 1  #2 * 1.6e-19 * 1e-3 / (4.13e-15) * 1e9  # 单位转换因子 (nA) (2e/h) not (2e/hbar)
        
        # 稀疏计算配置
        self.use_sparse = params.get('use_sparse', True)
        self.sparse_threshold = params.get('sparse_threshold', 0.3)
        self.sparse_depth_limit = params.get('sparse_depth_limit', 5)
        
        # 预分解器缓存
        self.factorized_cache = {}
        
        logging.info(f"GreenFunctionCalculator initialized, Sparse mode: {self.use_sparse}") 

    def fermi_dirac(self, omega):
        """费米狄拉克分布"""
        kBT = 5e-3 * self.delta  # 温度 (meV)
        return 1.0 / (1.0 + np.exp((omega - self.mu_lead) / kBT))

    def surface_gf_sc(self, omega, H00, H01, direction='left'):
        """表面格林函数计算（超导引线）"""
        E0 = omega + 1j * self.eta
        dim = H00.shape[0]
        eps = eps_s = H00.copy()
        
        if direction == 'right':
            alpha = H01
            beta = H01.conj().T
        else:  # 'left'
            alpha = H01.conj().T
            beta = H01
        
        # 递归计算
        for _ in range(self.recursion_depth):
            g = la.inv(E0 * np.eye(dim) - eps)
            eps_s = eps_s + alpha @ g @ beta
            eps = eps + alpha @ g @ beta + beta @ g @ alpha
            
            alpha_new = alpha @ g @ alpha
            beta_new = beta @ g @ beta
            
            if np.linalg.norm(alpha_new) < 1e-12:
                break
            alpha, beta = alpha_new, beta_new
        
        g_surface = la.inv(E0 * np.eye(dim) - eps_s)
        hgh = H01 @ g_surface @ H01.conj().T if direction == 'right' else H01.conj().T @ g_surface @ H01
        
        return g_surface, hgh

    def recursive_sweep(self, omega, sE_initial, range_sets, hop_FKs, g0_inv_FKs):
        """递归扫描，完全兼容稀疏和稠密矩阵，优化稀疏矩阵操作"""
        range_1, range_2, offset_L = range_sets
        sE_L0, sE_R0, sE_Lf0, sE_Rf0 = sE_initial
        
        # 获取矩阵维度
        first_mat = g0_inv_FKs[0]
        dim = first_mat.shape[0]
        
        # 创建单位矩阵（兼容稀疏和稠密）
        if sp.issparse(g0_inv_FKs[0]):
            E0_I0 = (omega + 1j * self.eta) * sp.eye(dim, format='csc', dtype=np.complex128)
        else:
            E0_I0 = (omega + 1j * self.eta) * np.eye(dim, dtype=np.complex128)
        
        # 初始化列表
        g0s_invL = []
        sELs = []
        sERs = []
        sELs_less = []
        sERs_less = []
        
        # 初始化中间变量
        GL_i = None
        GR_i = None
        GL_less_i = None
        GR_less_i = None
        
        for recur_i in range(range_1, range_2):
            recur_i_R = range_2 - 1 - recur_i + offset_L
            
            # 获取当前矩阵
            g0_inv_i = g0_inv_FKs[recur_i]
            g0_inv_iR = g0_inv_FKs[recur_i_R]
            alpha_i = hop_FKs[recur_i]
            alpha_iR = hop_FKs[recur_i_R]
            
            # 初始自能处理
            if recur_i == range_1 or recur_i_R == range_2 - 1:
                sEL_i = sE_L0
                sER_i = sE_R0
                sEL_less_i = sE_Lf0.conj().T - sE_Lf0
                sER_less_i = sE_Rf0.conj().T - sE_Rf0
            else:
                # 使用稀疏矩阵乘法避免转换
                if sp.issparse(alpha_i) and sp.issparse(GL_i):
                    sEL_i = alpha_i.conj().T @ GL_i @ alpha_i
                else:
                    # 确保矩阵为稠密后再进行运算
                    if sp.issparse(GL_i):
                        GL_i = GL_i.toarray()
                    if sp.issparse(alpha_i):
                        alpha_i = alpha_i.toarray()
                    sEL_i = alpha_i.conj().T @ GL_i @ alpha_i
                
                if sp.issparse(alpha_iR) and sp.issparse(GR_i):
                    sER_i = alpha_iR @ GR_i @ alpha_iR.conj().T
                else:
                    if sp.issparse(GR_i):
                        GR_i = GR_i.toarray()
                    if sp.issparse(alpha_iR):
                        alpha_iR = alpha_iR.toarray()
                    sER_i = alpha_iR @ GR_i @ alpha_iR.conj().T
                
                # 同样处理lesser分量
                if sp.issparse(alpha_i) and sp.issparse(GL_less_i):
                    sEL_less_i = alpha_i.conj().T @ GL_less_i @ alpha_i
                else:
                    if sp.issparse(GL_less_i):
                        GL_less_i = GL_less_i.toarray()
                    if sp.issparse(alpha_i):
                        alpha_i = alpha_i.toarray()
                    sEL_less_i = alpha_i.conj().T @ GL_less_i @ alpha_i
                
                if sp.issparse(alpha_iR) and sp.issparse(GR_less_i):
                    sER_less_i = alpha_iR @ GR_less_i @ alpha_iR.conj().T
                else:
                    if sp.issparse(GR_less_i):
                        GR_less_i = GR_less_i.toarray()
                    if sp.issparse(alpha_iR):
                        alpha_iR = alpha_iR.toarray()
                    sER_less_i = alpha_iR @ GR_less_i @ alpha_iR.conj().T
            
            # 构建逆格林函数
            gi_inv_L = E0_I0 - g0_inv_i
            gi_inv_R = E0_I0 - g0_inv_iR
            
            # 预分解矩阵以提高性能
            cache_key_L = (id(gi_inv_L), id(sEL_i))
            if cache_key_L in self.factorized_cache:
                solve_L = self.factorized_cache[cache_key_L]
                GL_i = solve_L(sp.eye(dim, format='csc', dtype=np.complex128))
            else:
                mat_to_inv = gi_inv_L - sEL_i
                if sp.issparse(mat_to_inv):
                    solve_L = factorized(mat_to_inv.tocsc())
                    GL_i = solve_L(sp.eye(dim, format='csc', dtype=np.complex128))
                    self.factorized_cache[cache_key_L] = solve_L
                else:
                    GL_i = la.inv(mat_to_inv)
            
            cache_key_R = (id(gi_inv_R), id(sER_i))
            if cache_key_R in self.factorized_cache:
                solve_R = self.factorized_cache[cache_key_R]
                GR_i = solve_R(sp.eye(dim, format='csc', dtype=np.complex128))
            else:
                mat_to_inv = gi_inv_R - sER_i
                if sp.issparse(mat_to_inv):
                    solve_R = factorized(mat_to_inv.tocsc())
                    GR_i = solve_R(sp.eye(dim, format='csc', dtype=np.complex128))
                    self.factorized_cache[cache_key_R] = solve_R
                else:
                    GR_i = la.inv(mat_to_inv)
            
            # 计算lesser分量
            if sp.issparse(GL_i):
                GL_less_i = GL_i @ sEL_less_i @ GL_i.conj().T
            else:
                GL_less_i = GL_i @ sEL_less_i @ GL_i.conj().T
            
            if sp.issparse(GR_i):
                GR_less_i = GR_i @ sER_less_i @ GR_i.conj().T
            else:
                GR_less_i = GR_i @ sER_less_i @ GR_i.conj().T
            
            # 存储结果
            sELs.append(sEL_i)
            sERs.append(sER_i)
            sELs_less.append(sEL_less_i)
            sERs_less.append(sER_less_i)
            g0s_invL.append(gi_inv_L)
        
        return g0s_invL, [sELs, sERs, sELs_less, sERs_less]

    def compute_self_energies(self, omega, lead_type, ham, max_sidebands, Vbias, N_SC, N_junction, hop_FKs, g0_inv_FKs):
        """计算自能，完全兼容稀疏和稠密矩阵"""
        sideN = 2 * max_sidebands + 1
        dim = 4 * sideN
        
        # 创建稠密矩阵
        sEL = np.zeros((dim, dim), dtype=np.complex128)
        sER = np.zeros_like(sEL)
        sEL_f0 = np.zeros_like(sEL)
        sER_f0 = np.zeros_like(sEL)
        
        for n in range(sideN):
            band_i = n - max_sidebands
            omega_shifted = omega + band_i * Vbias
            f0 = self.fermi_dirac(omega_shifted)
            a = slice(n * 4, n * 4 + 4)
            
            # 左侧超导引线
            _, sEL_block = self.surface_gf_sc(omega_shifted, ham['H00_L'], ham['H01_L'], direction='left')
            sEL[a, a] = sEL_block
            sEL_f0[a, a] = sEL_block * f0
            
            # 右侧超导引线
            _, sER_block = self.surface_gf_sc(omega_shifted, ham['H00_R'], ham['H01_R'], direction='right')
            sER[a, a] = sER_block
            sER_f0[a, a] = sER_block * f0
        
        if lead_type == 'infinite':
            return [None, None], [sEL, sER, sEL_f0, sER_f0]
        
        # 处理有限引线
        sE_initial = [sEL, sER, sEL_f0, sER_f0]
        
        # 处理左侧引线区域
        g0_inv_Ld, sE_Ld = self.obtain_recursive_list(
            omega, 'left_lead', sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        # 处理右侧引线区域
        g0_inv_Rd, sE_Rd = self.obtain_recursive_list(
            omega, 'right_lead', sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        # 确保矩阵为稠密
        if sp.issparse(g0_inv_Ld[-1]):
            g0_inv_L = g0_inv_Ld[-1].toarray()
        else:
            g0_inv_L = g0_inv_Ld[-1]
        
        if sp.issparse(g0_inv_Rd[-1]):
            g0_inv_R = g0_inv_Rd[-1].toarray()
        else:
            g0_inv_R = g0_inv_Rd[-1]
        
        # 获取自能
        sEL_left = sE_Ld[0][-1] if not sp.issparse(sE_Ld[0][-1]) else sE_Ld[0][-1].toarray()
        sER_left = sE_Ld[1][0] if not sp.issparse(sE_Ld[1][0]) else sE_Ld[1][0].toarray()
        sEL_right = sE_Rd[0][0] if not sp.issparse(sE_Rd[0][0]) else sE_Rd[0][0].toarray()
        sER_right = sE_Rd[1][-1] if not sp.issparse(sE_Rd[1][-1]) else sE_Rd[1][-1].toarray()
        
        # 计算格林函数
        g_retard_L = la.inv(g0_inv_L - sEL_left - sER_left)
        g_retard_R = la.inv(g0_inv_R - sEL_right - sER_right)
        
        # 更新自能
        sEL = np.zeros_like(sEL)
        sER = np.zeros_like(sER)
        sEL_f0 = np.zeros_like(sEL_f0)
        sER_f0 = np.zeros_like(sER_f0)
        
        hop_LC = hop_FKs[N_SC - 1]
        hop_CR = hop_FKs[N_SC + N_junction]
        
        # 确保跳跃矩阵为稠密
        if sp.issparse(hop_LC):
            hop_LC = hop_LC.toarray()
        if sp.issparse(hop_CR):
            hop_CR = hop_CR.toarray()
        
        for n in range(sideN):
            band_i = n - max_sidebands
            omega_shifted = omega + band_i * Vbias
            f0 = self.fermi_dirac(omega_shifted)
            a = slice(n * 4, n * 4 + 4)
            
            sEL[a, a] = hop_LC[a, a].conj().T @ g_retard_L[a, a] @ hop_LC[a, a]
            sER[a, a] = hop_CR[a, a] @ g_retard_R[a, a] @ hop_CR[a, a].conj().T
            sEL_f0[a, a] = sEL[a, a] * f0
            sER_f0[a, a] = sER[a, a] * f0
        
        return [g_retard_L, g_retard_R], [sEL, sER, sEL_f0, sER_f0]

    def obtain_recursive_list(self, omega, region_type, sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs):
        """获取递归列表（保持不变）"""
        if region_type == 'left_lead':
            sE_initial = [sE * i0 for sE, i0 in zip(sE_initial, [1, 0, 1, 0])]
            range_sets = [0, N_SC, 0]
            recur_depth = N_SC
        elif region_type == 'right_lead':
            sE_initial = [sE * i0 for sE, i0 in zip(sE_initial, [0, 1, 0, 1])]
            range_sets = [N_SC + N_junction, 2 * N_SC + N_junction, N_SC + N_junction]
            recur_depth = N_SC
        elif region_type == 'junction':
            range_sets = [N_SC, N_SC + N_junction, N_SC]
            recur_depth = N_junction
        
        return self.recursive_sweep(omega, sE_initial, range_sets, hop_FKs, g0_inv_FKs)

    def _compute_current_at_omega(self, omega, lead_type, region_type, ham_builder, max_sidebands, Vbias, N_SC, N_junction, g0_inv_FKs, hop_FKs):
        """电流计算，使用传入的哈密顿量切片"""
        # 计算自能
        Gs, sE_initial = self.compute_self_energies(
            omega, lead_type, ham_builder.construct_hamiltonian(0), max_sidebands, Vbias, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        # 获取递归列表
        g0_inv, sE4s_leads = self.obtain_recursive_list(
            omega, region_type, sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        recur_depth = N_junction
        
        # 准备tau_z矩阵
        sideN = 2 * max_sidebands + 1
        tau_z_matrix = np.kron(np.eye(sideN), np.kron(ham_builder.tau_z, np.eye(2)))
        
        trace0 = 0
        trace_set = []
        for recur_i in range(recur_depth):
            recur_i_R = recur_depth - 1 - recur_i
            
            # 获取当前矩阵并确保为稠密
            gi_inv = g0_inv[recur_i]
            if sp.issparse(gi_inv):
                gi_inv = gi_inv.toarray()
            
            sEL_i = sE4s_leads[0][recur_i]
            if sp.issparse(sEL_i):
                sEL_i = sEL_i.toarray()
            
            sER_i = sE4s_leads[1][recur_i_R]
            if sp.issparse(sER_i):
                sER_i = sER_i.toarray()
            
            sEL_less_i = sE4s_leads[2][recur_i]
            if sp.issparse(sEL_less_i):
                sEL_less_i = sEL_less_i.toarray()
            
            sER_less_i = sE4s_leads[3][recur_i_R]
            if sp.issparse(sER_less_i):
                sER_less_i = sER_less_i.toarray()
            
            # 使用稠密矩阵计算
            G_nn_ret = la.inv(gi_inv - sEL_i - sER_i)
            G_nn_adv = G_nn_ret.conj().T
            
            sEL_nn_less = sEL_less_i
            sE_nn_less = sEL_less_i + sER_less_i
            sEL_nn = sEL_i
            
            G_nn_less = G_nn_ret @ sE_nn_less @ G_nn_adv
            temp = (G_nn_ret @ sEL_nn_less + G_nn_less @ sEL_nn.conj().T) @ tau_z_matrix
            
            trace_val = np.trace(temp).real
            trace0 += trace_val * self.unit_factor_nA
            
            trace_set.append(trace_val * self.unit_factor_nA)
            
        return np.array(trace_set) #, trace0

class JosephsonJunctionSolver:
    def __init__(self, params, path_manager):
        # 添加稀疏配置参数
        params.setdefault('use_sparse', True)
        params.setdefault('sparse_threshold', 0.3)
        params.setdefault('sparse_depth_limit', 5)
        
        self.params = params
        self.path_manager = path_manager
        self.output_dir = path_manager.get_raw_data_dir()
        
        # 系统参数
        self.params = params
        self.ham_builder = HamiltonianBuilder(params)
        self.gf_calculator = GreenFunctionCalculator(params)
        
        # 计算参数
        self.max_sidebands = params['max_sidebands']
        self.omega_points = params['omega_points']
        self.Vbias_points = params['Vbias_points']
        self.B_points = params.get('B_points', 0)
        self.job_parallel = params['job_parallel']
        
        # 创建输出目录
        os.makedirs(self.output_dir, exist_ok=True)
        
        # 单位转换
        self.unit_factor_nA = 2 * 1.6e-19 * 1e-3 / (4.13e-15) * 1e9  # nA
        # 自适应计算参数
        self.adaptive_iv = params.get('adaptive_iv', False)
        self.adaptive_tol = params.get('adaptive_tol', 0.01)
        self.adaptive_max_N = params.get('adaptive_max_N', 20)
        self.adaptive_method = params.get('adaptive_method', 'basic')  # 'basic', 'dynamic', 'step'

        # 哈密顿量切片缓存
        self.ham_slice_cache = {}
        
        # 生成公共时间戳
        self.common_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        logging.info("JosephsonJunctionSolver initialized")
        logging.info(f"Superconductor: {params['N_SC']} sites, Junction: {params['N_junction']} sites")
        logging.info(f"Recursion depth: {params['recursion_depth']}, Max sidebands: {params['max_sidebands']}")

    def _get_base_metadata(self):
        """返回所有计算的通用元数据参数"""
        # 基础元数据，始终包含
        metadata = {
            "N_SC": self.params['N_SC'],
            "N_junction": self.params['N_junction'],
            "delta": self.params['delta'],
            "alpha": self.params['alpha'],
            "v_tau": self.params['v_tau'],
            "B": self.params['B'],
            "Vdis":self.params['disorder_strength'],
            "max_sidebands": self.params['max_sidebands'],
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "common_timestamp": self.common_timestamp  # 添加公共时间戳
        }
        
        # 添加disorder类型
        disorder_type = self.params.get('disorder_type', 'none')
        metadata["disorder_type"] = disorder_type
        
        # 根据disorder类型添加相关参数
        if disorder_type != 'none':
            if disorder_type == 'gaussian':
                metadata.update({
                    "Vdis": self.params.get('Vdis_gau', 0.0),
                    "decayL": self.params.get('decayL_gau', 50),
                    "Xdis": self.params.get('Xdis_gau', 150)
                })
            elif disorder_type == 'smooth':
                metadata.update({
                    "decayL": self.params.get('decayL_smooth', 50),
                    "Vdis": self.params.get('Vdis_smooth', 0.0),
                    "Vd": self.params.get('Vd_smooth', 0.8)
                })
            elif disorder_type in ['random_typeI', 'random_typeII']:
                metadata.update({
                    "N_imp": self.params.get('N_imp1', 52),
                    "lambda_imp": self.params.get('lambda_imp1', 18.0),
                    "V0": self.params.get('V0_imp1', 0.0),
                    "n_d": self.params.get('Nd_imp2', 10.0),
                    "random_lambda_imp": self.params.get('lambda_imp2', 20.0),
                    "random_V0": self.params.get('V0_imp2', 0.0),
                    "a0": self.params.get('a0', 10.0)
                })
            elif disorder_type == 'nonhermitian':
                metadata.update({
                    "charge_V0": self.params.get('charge_V0', 0.5),
                    "n_d": self.params.get('n_d', 10.0)
                })
            elif disorder_type == 'from_file':
                metadata.update({
                    "disorder_file": self.params.get('disorder_file', 'disorder_data.txt')
                })
        
        # 添加自适应参数
        if self.params.get('adaptive_iv', False):
            metadata.update({
                "adaptive_method": self.params.get('adaptive_method', 'basic'),
                "adaptive_tol": self.params.get('adaptive_tol', 0.01),
                "adaptive_max_N": self.params.get('adaptive_max_N', 20)
            })
        
        return metadata

    def save_disorder_data(self):
        """保存无序数据（如果disorder类型为'from_file'则跳过）"""
        # 如果disorder类型为'from_file'，则不保存
        if self.params.get('disorder_type', 'none') == 'from_file':
            logging.info("Disorder type is 'from_file', skipping disorder data saving")
            return {
                "csv_file": None,
                "plot_file": None,
                "metadata_file": None
            }
            
        disorder_dir = os.path.join(self.output_dir, "disorder_data")
        os.makedirs(disorder_dir, exist_ok=True)
        
        # 保存图表
        plot_file = self._plot_disorder_distribution(disorder_dir)
        
        # 保存CSV
        csv_file = self._save_disorder_csv(disorder_dir)
        
        # 保存元数据
        metadata_file = self._save_disorder_metadata(disorder_dir)
        
        return {
            "csv_file": csv_file,
            "plot_file": plot_file,
            "metadata_file": metadata_file
        }

    def _plot_disorder_distribution(self, output_dir):
        """绘制无序分布并保存为PNG"""
        # 使用公共时间戳
        plot_filename = os.path.join(output_dir, f"disorder_plot_{self.common_timestamp}.png")
        
        # 创建图表
        plt.figure(figsize=(12, 6))
        plt.plot(self.ham_builder.disorder_distribution, 'b-', linewidth=1.5)
        
        # 添加区域标记
        plt.axvline(x=self.params['N_SC'], color='r', linestyle='--', alpha=0.5)
        plt.axvline(x=self.params['N_SC'] + self.params['N_junction'], color='r', linestyle='--', alpha=0.5)
        
        # 标签和标题
        plt.title(f'Disorder Potential ({self.params.get("disorder_type", "none")})', fontsize=14)
        plt.xlabel('Site Index', fontsize=12)
        plt.ylabel('Potential (meV)', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.7)
        
        # 保存图表
        plt.savefig(plot_filename, dpi=300)
        plt.close()
        logging.info(f"Disorder plot saved: {plot_filename}")
        
        return plot_filename

    def _save_disorder_csv(self, output_dir):
        """保存无序分布到CSV文件"""
        # 使用公共时间戳
        csv_filename = os.path.join(output_dir, f"disorder_data_{self.common_timestamp}.csv")
        
        # 创建CSV文件
        df = pd.DataFrame({
            'site_index': np.arange(len(self.ham_builder.disorder_distribution)),
            'disorder_value': self.ham_builder.disorder_distribution
        })
        df.to_csv(csv_filename, index=False)
        logging.info(f"Disorder data saved: {csv_filename}")
        
        return csv_filename

    def _save_disorder_metadata(self, output_dir):
        """保存无序元数据到JSON文件，仅相关参数"""
        disorder_type = self.params.get('disorder_type', 'none')
        
        # 基础元数据
        metadata = {
            "disorder_type": disorder_type,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "common_timestamp": self.common_timestamp  # 添加公共时间戳
        }
        
        # 根据disorder类型添加相关参数
        if disorder_type != 'none':
            if disorder_type == 'gaussian':
                metadata.update({
                    "Vdis": self.params.get('Vdis_gau', 0.0),
                    "decayL": self.params.get('decayL_gau', 50),
                    "Xdis": self.params.get('Xdis_gau', 150)
                })
            elif disorder_type == 'smooth':
                metadata.update({
                    "decayL": self.params.get('decayL_smooth', 50),
                    "Vdis": self.params.get('Vdis_smooth', 0.0),
                    "Vd": self.params.get('Vd_smooth', 0.8)
                })
            elif disorder_type in ['random_typeI', 'random_typeII']:
                metadata.update({
                    "N_imp": self.params.get('N_imp1', 52),
                    "lambda_imp": self.params.get('lambda_imp1', 18.0),
                    "V0": self.params.get('V0_imp1', 0.0),
                    "n_d": self.params.get('Nd_imp2', 10.0),
                    "random_lambda_imp": self.params.get('lambda_imp2', 20.0),
                    "random_V0": self.params.get('V0_imp2', 0.0),
                    "a0": self.params.get('a0', 10.0)
                })
            elif disorder_type == 'nonhermitian':
                metadata.update({
                    "charge_V0": self.params.get('charge_V0', 0.5),
                    "n_d": self.params.get('n_d', 10.0)
                })
            elif disorder_type == 'from_file':
                metadata.update({
                    "disorder_file": self.params.get('disorder_file', 'disorder_data.txt')
                })
        
        # 使用公共时间戳
        metadata_file = os.path.join(output_dir, f"disorder_metadata_{self.common_timestamp}.json")
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=4)
        
        logging.info(f"Disorder metadata saved: {metadata_file}")
        
        return metadata_file

    def run_calculation(self, calc_type):
        """运行指定计算"""
        # 保存无序数据（如果disorder类型不是'from_file'）
        if self.params.get('disorder_type', 'none') != 'none' and self.params.get('disorder_type', 'none') != 'from_file':
            self.save_disorder_data()
        
        # 保存任务元数据
        self.path_manager.save_task_metadata(self.params)
        
        # 运行计算
        if calc_type == "DC_IV":
            metadata, *data = self.compute_dc_iv()
        elif calc_type == "DC_IV_Bsweep":
            metadata, *data = self.compute_dc_iv_Bsweep()
        else:
            raise ValueError("Invalid calculation type")
        
        # 保存结果
        filepath = self.save_results(metadata, data)
        
        # 绘图
        plot_path = JJPlotter.plot_single_result(metadata, data, self.path_manager.get_plots_dir())
        
        return filepath, plot_path
    
    def save_results(self, metadata, data):
        """保存计算结果到文件"""
        calc_type = metadata["calculation"]
        # 使用公共时间戳
        common_timestamp = metadata.get("common_timestamp", datetime.now().strftime("%Y%m%d_%H%M%S"))
        filename = f"{calc_type}_{common_timestamp}.csv"
        filepath = os.path.join(self.output_dir, filename)
        
        metadata_lines = ["# Calculation Metadata"]
        for key, value in metadata.items():
            metadata_lines.append(f"# {key}: {value}")
        
        # 根据计算类型创建数据框
        if calc_type == "DC_IV":
            if len(data) == 3:  # 包含边带数
                Vbias, current, sideband = data
                df = pd.DataFrame({
                    'Bias Voltage (meV)': Vbias,
                    'Current (nA)': current,
                    'SidebandN': sideband
                })
            else:
                Vbias, current = data
                df = pd.DataFrame({
                    'Bias Voltage (meV)': Vbias,
                    'Current (nA)': current
                })
        elif calc_type == "DC_IV_Bsweep":
            B, current, sideband = data
            df = pd.DataFrame({
                'Magnetic Field': B,
                'Current (nA)': current,
                'sidebandN':sideband,
                'Fixed Bias (meV)': metadata['fixed_Vbias']
            })
            
        # 写入文件
        with open(filepath, 'w') as f:
            f.write("\n".join(metadata_lines))
            f.write("\n")
            df.to_csv(f, index=False)
        
        return filepath
    
    def tics(self, stamp=False):
        """Start timer """
        if stamp:
            print(datetime.now().strftime("Starting at %d/%m/%Y %H:%M:%S"))
        return time.time()

    def toc(self, tic, disp=True, nice_format=False, stamp=False):
        """Stop timer and display elapsed time"""
        elapsed_time = time.time() - tic
        if stamp:
            print(datetime.now().strftime("Finished at %d/%m/%Y %H:%M:%S"))
        if disp:
            if (not nice_format) or (elapsed_time < 60):
                print(f"Elapsed time: {np.round(elapsed_time,3)} seconds")
            else:
                hrs, rem = divmod(elapsed_time, 3600)
                mins, secs = divmod(rem, 60)
                print(f"Elapsed time: {int(hrs):02d}:{int(mins):02d}:{int(secs):02d}")
        return elapsed_time
        
    @staticmethod
    def generate_nonuniform_grid(
        start: float,           # 整个区域起始点
        end: float,             # 整个区域结束点
        dense_start: float,     # 期望的dense区域起点
        dense_end: float,       # 期望的dense区域终点
        space_non_dense: float = 0.05,  # 非dense区域间距 (默认0.05)
        space_ratio: float = 0.4,       # dense区域间距比例 (默认0.4)
        max_points: int = 300           # 最大总点数 (默认300)
        ) ->np.ndarray:
        """ """
        space_dense = space_non_dense * space_ratio
        
        # 调整dense区域边界
        n_left_segments = max(0, int(np.floor((dense_start - start) / space_non_dense)))
        adjusted_dense_start = start + n_left_segments * space_non_dense

        max_dense_end = end - space_non_dense  
        
        if max_dense_end > adjusted_dense_start:   # dense区域右侧至少有一个点
            min_segments = max(0, int(np.ceil((dense_end - adjusted_dense_start) / space_dense)))
            max_segments = int(np.floor((max_dense_end - adjusted_dense_start) / space_dense))
            n_dense_segments = min(min_segments, max_segments) if min_segments > 0 else max_segments
            adjusted_dense_end = adjusted_dense_start + n_dense_segments * space_dense
        else:
            n_dense_segments = 0
            adjusted_dense_end = adjusted_dense_start
        
        # 计算各区域点数
        n_left = n_left_segments
        n_dense = n_dense_segments + 1 if n_dense_segments > 0 else 0
        n_right_segments = max(0, int(np.floor((end - adjusted_dense_end) / space_non_dense)))
        n_right = n_right_segments
        
        points = []
        
        # 左侧非dense区域
        if n_left > 0:
            left_points = np.linspace(start, adjusted_dense_start, n_left + 1, endpoint=False)
            points.extend(left_points)
        
        # dense区域
        if n_dense > 0:
            dense_points = np.linspace(adjusted_dense_start, adjusted_dense_end, n_dense)
            points.extend(dense_points)
        
        # 右侧非dense区域
        if n_right > 0:
            right_start = adjusted_dense_end + space_non_dense
            right_points = np.linspace(
                right_start, 
                right_start + n_right * space_non_dense, 
                n_right + 1, 
                endpoint=True
            )
            # 确保不超过end
            right_points = right_points[right_points <= end]
            points.extend(right_points)
        
        points = np.array(points)
        points.sort()
        
        # 确保包含起点和终点
        if abs(points[0] - start) > 1e-10:
            points = np.insert(points, 0, start)
        if abs(points[-1] - end) > 1e-10:
            points = np.append(points, end)
        
        # 点数限制
        if len(points) > max_points:
            logging.warning(f"网格点数({len(points)})超过最大限制({max_points})，进行截断")
            points = points[:max_points]
        
        return points

    @staticmethod
    def generate_uniform_grid(
        start: float,           # 区域起始点
        end: float,             # 区域结束点
        spacing: float = 0.05,  # 点间距 (默认0.05)
        max_points: int = 300   # 最大总点数 (默认300)
        ) -> np.ndarray:
        """ """
        num_segments = (end - start) / spacing
        n_points = int(np.floor(num_segments)) + 1
        
        # 调整间距以满足最大点数限制
        if n_points > max_points:
            n_points = max_points
            spacing = (end - start) / (n_points - 1) if n_points > 1 else 0
        
        points = np.linspace(start, end, n_points)
        return points
        
    def generate_Vbias_vals(self):
        """Generate Vbias values using the same logic as compute_dc_iv"""
        # delta = self.params['delta']
        # Config. [0.03,3,0.03,2.1,0.03,0.4,300]
        # Config. [0.03,3,0.12,1.2,0.04,0.4,300]
        
        Vbias_max_ratio = self.params.get('Vbias_max_ratio', 2.0)
        
        if Vbias_max_ratio <= 2.2:
            return self.generate_uniform_grid(0.01, (Vbias_max_ratio+0.01),
                                              0.01,self.params['Vbias_points'])  
        else:
            # compute_dc_iv 
            return self.generate_nonuniform_grid(
                start = 0.01,   ## previous, set as 0.001
                end = Vbias_max_ratio ,
                dense_start= 0.12,
                dense_end = 1.2,
                space_non_dense =0.04,
                space_ratio= 0.25,
                max_points = self.params['Vbias_points']
            )   
            
    def compute_dc_iv(self):
        """Compute DC I-V characteristic, returns (metadata, Vbias_vals, currents, sideband_Ns)"""
        start_time = time.time()
        delta = self.params['delta']
        
        Vbias_vals= delta*self.generate_Vbias_vals()
        
        logging.info("Starting DC IV calculation")
        results = Parallel(n_jobs=self.job_parallel[0])(
            delayed(self.compute_parallel_aux)(x0, "DC_IV") for x0 in Vbias_vals #[::-1]
        )
        
        # 分离电流和边带数
        currents = [res[0] for res in results]
        sideband_Ns = [res[1] for res in results]
        
        # Prepare metadata
        metadata = self._get_base_metadata()
        metadata.update({
            "calculation": "DC_IV",
            "duration": time.time() - start_time,
            "Vbias_points": self.Vbias_points, })
        
        logging.info(f"DC IV calculation completed, duration: {metadata['duration']:.2f} sec")
        return metadata, Vbias_vals, currents, sideband_Ns
    

    def compute_parallel_aux(self, Vari_input, calc_type):
        """并行计算辅助函数（使用稀疏优化）"""
        if calc_type == "DC_IV":
            Vbias = Vari_input
            if self.params['adaptive_iv']:
                if self.params['adaptive_method'] == 'advanced':
                    return self.adaptive_current_advanced(Vbias)
            else:
                current = self.compute_current_at_bias(Vbias)
                return current, self.params['max_sidebands']
                
        elif calc_type == "DC_IV_Bsweep":
            B, Vbias = Vari_input  # 解包磁场和偏压
            # 更新磁场值
            self.ham_builder.B = B
            
            # 计算当前磁场和偏压下的电流
            if self.params['adaptive_iv']:
                current, sideband = self.compute_parallel_aux(Vbias, "DC_IV")
            else:
                current = self.compute_current_at_bias(Vbias)
                sideband = self.params['max_sidebands']
            
            return current, sideband
        
    def compute_dc_iv_Bsweep(self):
        """在固定偏压下扫描磁场计算直流电流"""
        start_time = time.time()
        Vbias = self.params['fixed_Vbias']  # 获取固定偏压值
        B_vals = self.generate_uniform_grid(
            start=0,
            end=self.params['B_max'],
            spacing=0.02,
            max_points=self.B_points
        )
        
        logging.info(f"Starting DC IV with B sweep @ Vbias={Vbias} meV")
        results = Parallel(n_jobs=self.job_parallel[0])(
            delayed(self.compute_parallel_aux)((B, Vbias), "DC_IV_Bsweep") 
            for B in B_vals
        )

        # 分离电流和边带数
        currents = [res[0] for res in results]
        sideband_Ns = [res[1] for res in results]

        
        # 准备元数据
        metadata = self._get_base_metadata()
        metadata.update({
            "calculation": "DC_IV_Bsweep",
            "duration": time.time() - start_time,
            "fixed_Vbias": Vbias,
            "B_points": self.B_points,
            "B_max": self.params['B_max']
        })
        
        logging.info(f"DC IV with B sweep completed, duration: {metadata['duration']:.2f} sec")
        
        return metadata, B_vals, currents, sideband_Ns
    
    def adaptive_current_advanced(self, Vbias):
        """改进的自适应策略（增量版本）"""
        # 获取自适应配置
        init_N, step_size, max_N = self._get_adaptive_config(Vbias)
        
        # 重置哈密顿量缓存
        self.ham_builder.prev_sidebands = None
        self.ham_builder.cached_g0_inv_FKs = None
        self.ham_builder.cached_hop_FKs = None
        
        # 初始计算
        N = init_N
        I_current = self.compute_current_at_bias(Vbias, N)
        
        # 记录收敛历史
        history = [(N, I_current)]
        
        # 自适应迭代
        for _ in range(15):  # 增加迭代次数限制
            # 检查是否达到最大边带数
            if N >= max_N:
                logging.warning(f"达到最大边带数 {max_N} @ Vbias={Vbias:.4f}Δ")
                # 返回历史中最接近收敛的结果
                return self._select_best_result(history)
            
            # 增加边带数
            new_N = N + step_size
            logging.debug(f"扩展边带数: {N} → {new_N} @ Vbias={Vbias:.4f}Δ")
            
            # 使用增量方法计算新电流
            I_next = self.compute_current_at_bias(Vbias, new_N)
            history.append((new_N, I_next))
            
            # 计算变化量
            delta_I = abs(I_next - I_current)
            rel_delta = delta_I / (abs(I_current) + 1e-6)
            
            logging.debug(f"边带数 {new_N}: 电流={I_next:.6f}, ΔI={delta_I:.6f}, 相对Δ={rel_delta:.4f}")
            
            # 收敛检查 - 同时考虑相对和绝对误差
            if rel_delta < self.params['rel_tol'] or delta_I < self.params['abs_tol']:
                logging.info(f"收敛于 {new_N} 边带 @ Vbias={Vbias:.4f}Δ, 电流={I_next:.6f}nA")
                return I_next, new_N
            
            I_current = I_next
            N = new_N
        
        # 未收敛时返回最佳结果
        logging.warning(f"达到最大迭代次数 @ Vbias={Vbias:.4f}Δ")
        return self._select_best_result(history)
    
    def compute_current_at_bias(self, Vbias, max_sidebands=None):
        """计算给定偏压下的电流（支持增量哈密顿量和缓存）"""
        if max_sidebands is None:
            max_sidebands = self.max_sidebands
            
        # 使用缓存：对于相同的Vbias和max_sidebands，避免重复构建哈密顿量切片
        cache_key = (Vbias, max_sidebands)
        if cache_key in self.ham_slice_cache:
            g0_inv_FKs, hop_FKs = self.ham_slice_cache[cache_key]
        else:
            ham = self.ham_builder.construct_hamiltonian(0)
            g0_inv_FKs, hop_FKs = self.ham_builder.build_slice_ham(Vbias, ham, max_sidebands)
            self.ham_slice_cache[cache_key] = (g0_inv_FKs, hop_FKs)
        
        N_SC = self.params['N_SC']
        N_junction = self.params['N_junction']
        
        # 生成能量积分网格
        if Vbias < 0.12:
            max_points_ = 21
        else:
            max_points_ = self.omega_points
            
        omega_values = self.generate_uniform_grid(
            start = 0 * Vbias,
            end = 1 * Vbias,
            spacing = 0.0025 * Vbias,  # 
            max_points =  max_points_
        )

        # 使用增量计算
        # 使用部分函数固定参数
        compute_func = functools.partial(
            self.gf_calculator._compute_current_at_omega,
            lead_type='infinite',
            region_type='junction',
            ham_builder=self.ham_builder,
            max_sidebands=max_sidebands,
            Vbias=Vbias,
            N_SC=N_SC,
            N_junction=N_junction,
            g0_inv_FKs=g0_inv_FKs,
            hop_FKs=hop_FKs
        )
        
        # 并行计算
        results = Parallel(n_jobs=self.job_parallel[1])(
            delayed(compute_func)(omega) for omega in omega_values)
        
        current_mat = np.real(np.array(results)).reshape([len(omega_values),-1])
        
        return np.trapz(current_mat, omega_values,axis=0)/0.3  #/ (2 * np.pi)
    
    def _get_adaptive_config(self, Vbias):
        """获取自适应配置（添加默认配置）"""
        delta = self.params['delta']
        ratio = Vbias / delta
        
        # 查找匹配的区域配置
        for config in self.params['adaptive_regions']:
            if config[0] <= ratio < config[1]:
                return config[2], config[3], config[4]
        
        # 默认配置
        return 10, 4, 50
    
    def _select_best_result(self, history):
        """从未收敛的历史中选择最佳结果"""
        if len(history) < 2:
            return history[-1][1], history[-1][0]
        
        # 计算所有连续点之间的变化
        changes = []
        for i in range(1, len(history)):
            prev_N, prev_I = history[i-1]
            curr_N, curr_I = history[i]
            delta = abs(curr_I - prev_I)
            rel_delta = delta / (abs(prev_I) + 1e-6)
            changes.append((rel_delta, delta, i))
        
        # 找到变化最小的点
        best_idx = min(changes, key=lambda x: x[0])[2]
        best_N, best_I = history[best_idx]
        
        logging.info(f"选择最佳结果: {best_N} 边带, 电流={best_I:.6f}nA")
        return best_I, best_N

class JJPlotter:
    @staticmethod
    def plot_single_result(metadata, data, output_dir):
        """绘制单次计算结果"""
        calc_type = metadata["calculation"]
        fig = plt.figure(figsize=(10, 8))
        
        if calc_type == "DC_IV":
            if len(data) == 3:  # 包含边带数信息
                Vbias, current, sideband = data
                # 双面板图
                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10), sharex=True)
                
                # 电流-电压曲线
                ax1.plot(Vbias, current, 'r-', lw=2)
                ax1.set_ylabel('Current (nA)', fontsize=14)
                ax1.set_title('DC Current-Voltage Characteristic', fontsize=16)
                ax1.grid(True)
                
                # 边带数-电压曲线
                ax2.plot(Vbias, sideband, 'b-', lw=2)
                ax2.set_xlabel('Bias Voltage (meV)', fontsize=14)
                ax2.set_ylabel('Sideband Number', fontsize=14)
                ax2.set_title('Converged Sideband Numbers', fontsize=16)
                ax2.grid(True)
            else:
                Vbias, current = data
                plt.plot(Vbias, current, '-', lw=2, color='r')
                plt.xlabel('Bias Voltage (meV)', fontsize=14)
                plt.ylabel('Current (nA)', fontsize=14)
                plt.title('DC Current-Voltage Characteristic', fontsize=16)
                plt.grid(True)
        
        elif calc_type == "DC_IV_Bsweep":
            if len(data) == 3:  # 包含边带数信息
                B, current, sideband = data
                # 双面板图
                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10), sharex=True)
                
                # 电流-电压曲线
                ax1.plot(B, current, 'r-', lw=2, marker='.', markersize =16)
                ax1.set_ylabel('Current (nA)', fontsize=14)
                ax1.set_title('DC Current-Voltage Characteristic', fontsize=16)
                ax1.grid(True)
                
                # 边带数-电压曲线
                ax2.plot(B, sideband, 'b-', lw=2)
                ax2.set_xlabel('zeeman (meV)', fontsize=14)
                ax2.set_ylabel('Sideband Number', fontsize=14)
                ax2.set_title('Converged Sideband Numbers', fontsize=16)
                ax2.grid(True)
            else:
                B, current= data
                plt.plot(B, current, 'b-o', lw=2)
                plt.xlabel('Magnetic Field', fontsize=14)
                plt.ylabel('Current (nA)', fontsize=14)
                plt.title(f'DC Current vs Magnetic Field @ Vbias={metadata["fixed_Vbias"]} meV', fontsize=16)
                plt.grid(True)
        
        # 创建文件名并保存
        common_timestamp = metadata.get("common_timestamp", datetime.now().strftime("%Y%m%d_%H%M%S"))
        filename = f"{calc_type}_plot_{common_timestamp}.png"
        filepath = os.path.join(output_dir, filename)
        
        plt.tight_layout()
        plt.savefig(filepath, dpi=300)
        plt.close()
        
        return filepath

class PathManager:
    def __init__(self, base_dir="results", task_name=None):
        self.base_dir = base_dir
        self.task_name = task_name or datetime.now().strftime("%Y%m%d_%H%M%S")
        self.task_dir = os.path.join(self.base_dir, self.task_name)
        
        # 创建必要的目录
        self._create_directories()
    
    def _create_directories(self):
        """创建所有必要的目录"""
        os.makedirs(self.task_dir, exist_ok=True)
        os.makedirs(self.get_raw_data_dir(), exist_ok=True)
        os.makedirs(self.get_plots_dir(), exist_ok=True)
    
    def get_raw_data_dir(self):
        return os.path.join(self.task_dir, "raw_data")
    
    def get_plots_dir(self):
        return os.path.join(self.task_dir, "plots")
    
    def get_task_metadata_path(self):
        return os.path.join(self.task_dir, "task_metadata.json")
    
    def save_task_metadata(self, params):
        """保存任务元数据"""
        metadata = {
            "task_name": self.task_name,
            "start_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "parameters": self._filter_metadata(params)
        }
        
        with open(self.get_task_metadata_path(), 'w') as f:
            json.dump(metadata, f, indent=4)
        
        return self.get_task_metadata_path()
    
    def _filter_metadata(self, params):
        """过滤元数据，只保留相关参数"""
        # 创建副本
        filtered = params.copy()
        
        # 移除大数组
        if 'disorder_distribution' in filtered:
            del filtered['disorder_distribution']
        
        return filtered

def update_parameters(user_params=None):
    """确保参数正确更新"""
    # get default
    params = get_default_parameters()
    
    # get a copy
    from copy import deepcopy
    updated_params = deepcopy(params)
    
    # update it then
    if user_params is not None:
        for key, value in user_params.items():
            # 特殊处理嵌套字典
            if key == 'adaptive_regions' and isinstance(value, list):
                updated_params[key] = deepcopy(value)
            if key == 'disorder_type':
                valid_regions = ['gaussian', 'smooth', 'random_typeI', 'random_typeII', 'nonhermitian', 'none', 'from_file']
                if value not in valid_regions:
                    raise ValueError(f"Invalid disorder_type: {value}. Must be one of {valid_regions}")
            elif key in updated_params:
                updated_params[key] = value
            else:
                logging.warning(f"Ignoring unknown parameter: {key}")
    
    # recording ...
    logging.info("Final simulation parameters:")
    
    for key, value in updated_params.items():
        logging.info(f"  {key}: {value}")
    
    return updated_params

def get_default_parameters():
    """ """
    return {
        'recursion_depth': 100,
        'eta': 1e-3,
        
        'N_SC': 150,
        'N_junction': 2,
        't': 12.7,
        'delta': 0.3,
        'mu': 0.0,
        'mu_lead': 0.0,
        'B': 0,
        'alpha': 4.0,
        
        'v_tau': 0.842,        
        'max_sidebands': 10,
        'Vbias_max_ratio': 5.0,
        'fixed_Vbias': 0.1,  # 固定偏压值 (meV)
        'B_max': 0.0,       # 最大磁场强度
        
        # can set them to be the largest allowed
        'omega_points': 2000,
        'Vbias_points': 300,
        'B_points': 300,

        # disorder参数
        'disorder_type': 'none',
        'disorder_file': 'disorder_data_test_Nd=50_ld=10_v0=2Delta.csv',
        'disorder_strength':0.0,
        
        # adaptive params 
        'adaptive_iv': True,
        'rel_tol': 0.02,
        'abs_tol':0.01,
        'adaptive_max_N': 100,
        'adaptive_method': 'advanced', # 'dynamic'
        'adaptive_regions': [
            (0.0, 0.025, 50, 10, 220),
            (0.025, 0.05, 40, 2, 80),
            (0.05, 0.1, 38, 2, 70),    # low vbias：start_N=30, increase_N = 4, end_Nmax = 80
            (0.1, 0.2, 20, 2, 50),    
            (0.2, 0.4, 10, 2, 30),     
            (0.4, 0.7, 8, 1, 20),     
            (0.7, 1.0, 6, 1, 12),     
            (1.0, 6.0, 4, 1, 10)      
        ],   # all in ratio to self. delta
        
        'job_parallel': [101, 1],
        'output_dir': 'results'     ## should be updated every time
    }

def run_dc_iv_test(use_sparse=True, max_sidebands=5):
    """运行DC_IV测试"""
    # 获取默认参数
    params = get_default_parameters()
    
    # 设置关键参数
    params.update({
        'N_SC': 150,           # 超导区长度
        'N_junction': 2,       # 结区长度
        'delta': 0.3,          # 超导能隙 (meV)
        'alpha': 4.0,          # 自旋轨道耦合强度
        'v_tau': 0.882, #0.934, #1.0, #0.882,         # 隧穿强度
        'B': 0,             # 磁场强度  # 0.25, 0.75, 1.25, 1.75
        'mu_lead': 0.0,        # 引线化学势
        'mu': 0.0,             # 化学势
        'max_sidebands': max_sidebands,  # 最大边带数
        'omega_points': 501,    # 能量点数（减少以加速测试）
        'Vbias_points': 1000,    # 偏压点数（减少以加速测试）
        'disorder_strength':0.5,
        
        'job_parallel': [161,1],  # 并行设置 [外层并行, 内层并行]
        'use_sparse': use_sparse,  # 是否使用稀疏矩阵
        'sparse_threshold': 0.3,  # 稀疏度阈值
        'sparse_depth_limit': 5,  # 最大稀疏递归深度
        'adaptive_iv': True,  # 禁用自适应计算
        'recursion_depth': 50, # 减少递归深度以加速测试
        'fixed_Vbias': 0.01*0.3,  # 固定偏压 0.15 meV
        'B_max': 2.0,         # 扫描磁场范围 0-1.5
        'B_points': 11,       # 磁场点数
    })
    # data set order: vtau = 0.842, B=0, 0.5, 2.0; vtau = 1.00, B=2.0, 0.5, 0.0
    
    # 创建路径管理器
    path_manager = PathManager(base_dir="26_results", task_name="dc_iv_test_v0")
    
    # 创建求解器
    solver = JosephsonJunctionSolver(params, path_manager)
    
    try:
        # 运行DC_IV计算
        start_time = time.time()
        solver.run_calculation("DC_IV") #("DC_IV") #("DC_IV_Bsweep")
        compute_time = time.time() - start_time
        
        # 打印结果摘要
        print("\n" + "="*50)
        print(f"DC_IV计算完成，耗时: {compute_time:.2f}秒")

    except Exception as e:
        logging.error(f"DC_IV计算失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

if __name__ == "__main__":
    # 测试单个DC_IV计算
    print("运行DC_IV计算测试...")
    run_dc_iv_test(use_sparse=True, max_sidebands=15)

2026-01-21 19:46:56,679 - INFO - HamiltonianBuilder initialized, disorder type: none, Sparse mode: True
2026-01-21 19:46:56,681 - INFO - GreenFunctionCalculator initialized, Sparse mode: True
2026-01-21 19:46:56,682 - INFO - JosephsonJunctionSolver initialized
2026-01-21 19:46:56,682 - INFO - Superconductor: 150 sites, Junction: 2 sites
2026-01-21 19:46:56,683 - INFO - Recursion depth: 50, Max sidebands: 15
2026-01-21 19:46:56,684 - INFO - Starting DC IV calculation


运行DC_IV计算测试...


2026-01-21 19:47:29,645 - INFO - DC IV calculation completed, duration: 32.96 sec



DC_IV计算完成，耗时: 33.41秒


<Figure size 1000x800 with 0 Axes>

/tmp/ipykernel_324996/2689089103.py:557: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/3449587957.py:557: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2689089103.py:557: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2689089103.py:557: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/3449587957.py:557: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2689089103.py:557: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/3449587957.py:557: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2689089103.py:557: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2689089103.py:557: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/3449587957.py:557: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2689089103.py:557: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/3449587957.py:557: RuntimeWarning: overflow

In [39]:
# updated, inclusing disoder 
import numpy as np
import scipy.linalg as la
from scipy.sparse import diags, block_diag, lil_matrix, csc_matrix, identity, issparse, eye as sparse_eye
from scipy.sparse.linalg import inv as sparse_inv
from scipy.sparse.linalg import spsolve, factorized, inv
import scipy.sparse as sp
from joblib import Parallel, delayed
import pandas as pd
import os
import time
from datetime import datetime
import logging
import json
import matplotlib.pyplot as plt
import hashlib
import functools
from collections import defaultdict

# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class HamiltonianBuilder:
    def __init__(self, params):
        # 验证disorder类型
        if 'disorder_type' in params:
            valid_disorders = ['gaussian', 'smooth', 'random_typeI', 'random_typeII', 'nonhermitian', 'none', 'from_file']
            if params['disorder_type'] not in valid_disorders:
                raise ValueError(f"non-effective model: {params['disorder_type']}. must be: {', '.join(valid_disorders)}")
        
        # 物理参数
        self.t = params['t']  # 跳跃强度 (meV)
        self.delta = params['delta']  # 超导能隙 (meV)
        self.mu = params['mu']  # 化学势 (meV)
        self.mu_lead = params['mu_lead']  # 引线化学势 (meV)
        self.B = params['B'] 
        self.alpha = params['alpha']  # 自旋轨道耦合 (meV)
             
        # 结参数
        self.N_SC = params['N_SC']  # 超导区长度
        self.N_junction = params['N_junction']  # 结区长度
        self.v_tau = params['v_tau']  # 隧穿强度
        
        # Disorder参数
        self.disorder_type = params.get('disorder_type', 'none')  # Disorder类型
        # Gaussian 
        self.V_gau = params.get('Vdis_gau', 0.0)  # Disorder强度
        self.X_gau = params.get('Xdis_gau', 50)  # Disorder中心
        self.decayL_gau = params.get('decayL_gau', 50)  # Disorder衰减长度     
        # smooth
        self.decayL_smooth = params.get('decayL_smooth', 50)  # Smooth disorder参数
        self.Vmax_smooth = params.get('Vdis_smooth', 0.3)  # Max disorder potential
        self.Vd_smooth = params.get('Vd_smooth', 0.0)  # Disorder深度   
        # random typeI
        self.Num_Rd1 = params.get('N_imp1', 52)  # Number of impurities
        self.lambda_Rd1 = params.get('lambda_imp1', 18.0)  # Characteristic length
        self.V0_Rd1 = params.get('V0_imp1', 0.0)  # Disorder amplitude (meV)  
        # random typeII
        self.Nd_Rd2 = params.get('Nd_imp2', 10.0)  # Impurity density (per micron)
        self.lambda_Rd2 = params.get('lambda_imp2', 20.0)  # Characteristic length (nm)
        self.V0_Rd2 = params.get('V0_imp2', 0.0)  # Disorder amplitude (meV)
        self.a0 = params.get('a0', 10.0)  # Lattice constant (nm)
        # Nonhermitian
        self.nonH_eta = params.get('nonH_eta', 1.5e-3)
        # From file
        self.disorder_file = params.get('disorder_file', 'disorder_data.txt')  # File to read disorder from
        self.disorder_strength = params.get('disorder_strength', 1.0)  
        
        # Floquet参数
        self.max_sidebands = params['max_sidebands']
        
        # 初始化Pauli矩阵
        self._init_pauli_matrices()
        
        # 计算系统总长度
        self.total_sites = 2 * self.N_SC + self.N_junction
        self.select_mid_i = params['mid_site_i']
        
        # 初始化无序分布
        self.disorder_distribution = self._calculate_disorder_distribution()
        
        # 稀疏矩阵配置
        self.use_sparse = params.get('use_sparse', True)  # 默认启用稀疏矩阵
        self.sparse_threshold = params.get('sparse_threshold', 0.3)  # 稀疏度阈值
        self.sparse_depth_limit = params.get('sparse_depth_limit', 5)  # 最大稀疏递归深度

        # 新增缓存变量
        self.prev_sidebands = None
        self.cached_g0_inv_FKs = None
        self.cached_hop_FKs = None
        
        # 增量扩建缓存
        self.incremental_cache = {}
        
        logging.info(f"HamiltonianBuilder initialized, disorder type: {self.disorder_type}, Sparse mode: {self.use_sparse}")

    def _init_pauli_matrices(self):
        """初始化Pauli矩阵"""
        # 自旋空间
        self.sigma_0 = np.eye(2)
        self.sigma_x = np.array([[0, 1], [1, 0]])
        self.sigma_y = np.array([[0, -1j], [1j, 0]])
        self.sigma_z = np.array([[1, 0], [0, -1]])
        
        # Nambu空间
        self.tau_0 = np.eye(2)
        self.tau_x = np.array([[0, 1], [1, 0]])
        self.tau_y = np.array([[0, -1j], [1j, 0]])
        self.tau_z = np.array([[1, 0], [0, -1]])
        self.tau_plus = (self.tau_x + 1j * self.tau_y) / 2
        self.tau_minus = (self.tau_x - 1j * self.tau_y) / 2
        self.tau_e = np.array([[1, 0], [0, 0]])  # 电子块
        self.tau_h = np.array([[0, 0], [0, 1]])  # 空穴块

    def _calculate_disorder_distribution(self):
        """计算无序分布"""
        # 初始化数组
        disorder_array = np.zeros(self.total_sites)
        
        # 根据disorder类型生成无序
        if self.disorder_type == 'smooth':
            # 平滑无序
            for i in range(self.total_sites):
                if i < self.N_SC: 
                    disorder_array[i] = self.Vmax_smooth * \
                        (1 - self.Vd_smooth * np.exp(i / self.decayL_smooth))
                elif i >= self.N_junction + self.N_SC:
                    disorder_array[i] = self.Vmax_smooth * \
                        (1 - self.Vd_smooth * np.exp(-(i - self.N_junction) / self.decayL_smooth))
                else: # inside junction 
                    disorder_array[i] = self.Vmax_smooth * (1 - self.Vd_smooth)
        
        elif self.disorder_type == 'gaussian':
            # 高斯无序
            for i in range(self.total_sites):
                disorder_array[i] = self.V_gau * np.exp(-(i - self.X_gau)**2 / self.decayL_gau**2)
        
        elif self.disorder_type == 'nonhermitian':
            # 非厄米无序
            for i in range(self.total_sites):
                disorder_array[i] = -1j * self.nonH_eta
        
        elif self.disorder_type == 'random_typeI':
            # 随机类型I
            disorder_array = self._generate_random_typeI_disorder(range(self.total_sites))
        
        elif self.disorder_type == 'random_typeII':
            # 随机类型II
            disorder_array = self._generate_random_typeII_disorder(range(self.total_sites))
        
        elif self.disorder_type == 'from_file':
            # 从文件读取无序
            disorder_array = self._load_disorder_from_file()
        elif self.disorder_type == 'none':
            for i in range(self.total_sites):
                disorder_array[i] = 0
            
        return disorder_array

    def _load_disorder_from_file(self):
        """从文件加载无序分布"""
        try:
            # 尝试从不同文件格式加载
            if self.disorder_file.endswith('.csv'):
                df = pd.read_csv(self.disorder_file)
                if 'disorder_value' in df.columns:
                    disorder_array = df['disorder_value'].values
                else:
                    # 假设第一列包含无序值
                    disorder_array = df.iloc[:, 0].values
            elif self.disorder_file.endswith('.txt') or self.disorder_file.endswith('.dat'):
                disorder_array = np.loadtxt(self.disorder_file)
            elif self.disorder_file.endswith('.npy'):
                disorder_array = np.load(self.disorder_file)
            else:
                # 尝试作为文本文件加载
                disorder_array = np.loadtxt(self.disorder_file)
            
            # 检查数组长度是否匹配总格点数
            if len(disorder_array) != self.total_sites:
                logging.warning(f"Disorder array length ({len(disorder_array)}) doesn't match total sites ({self.total_sites}). Truncating or padding.")
                if len(disorder_array) > self.total_sites:
                    disorder_array = disorder_array[:self.total_sites]
                else:
                    disorder_array = np.pad(disorder_array, (0, self.total_sites - len(disorder_array)), 'constant')
            
            logging.info(f"Loaded disorder from file: {self.disorder_file}")
            return disorder_array
            
        except Exception as e:
            logging.error(f"Failed to load disorder from file {self.disorder_file}: {str(e)}")
            return np.zeros(self.total_sites)

    def _generate_random_typeI_disorder(self, sites):
        """生成随机类型I无序"""
        disorder_array = np.zeros(self.total_sites)
        rng = np.random.default_rng()
        
        # 仅在选定区域生成杂质
        impurity_positions = rng.choice(sites, self.Num_Rd1)
        impurity_signs = np.array([(-1)**(n) for n in range(self.Num_Rd1)])
        
        # 计算未归一化势
        S = np.zeros(self.total_sites)
        for i in sites:
            for j in range(self.Num_Rd1):
                distance = abs(i - impurity_positions[j])
                S[i] += impurity_signs[j] * np.exp(-distance / self.lambda_Rd1)
        
        # 归一化
        S_avg = np.mean(S)
        S_centered = S - S_avg
        variance = np.mean(S_centered**2)
        N_V = np.sqrt(variance) / self.V0_Rd1 if variance > 0 else 1.0
        
        # 应用到无序数组
        for i in sites:
            disorder_array[i] = (self.V0_Rd1 / N_V) * S_centered[i]
        
        logging.info(f"Generated charge impurity disorder, impurities: {self.Num_Rd1}")
        return disorder_array

    def _generate_random_typeII_disorder(self, sites):
        """生成随机类型II无序"""
        disorder_array = np.zeros(self.total_sites)
        L_total_nm = (self.total_sites - 1) * self.a0
        
        # 计算选定区域中的杂质数量
        region_length = len(sites) * self.a0 / 1000  # in microns
        N_d = int(self.Nd_Rd2 * region_length)
        
        rng = np.random.default_rng()
        
        # 生成杂质位置
        min_pos = min(sites) * self.a0 if sites else 0
        max_pos = max(sites) * self.a0 if sites else 0
        impurity_positions = rng.uniform(min_pos, max_pos, N_d)
        impurity_amplitudes = rng.normal(0, 1, N_d)
        
        # 计算选定位点的无序
        for i in sites:
            x_i = i * self.a0  # 位点位置 (nm)
            potential_sum = 0.0
            for j in range(N_d):
                distance = abs(x_i - impurity_positions[j])
                potential_sum += impurity_amplitudes[j] * np.exp(-distance / self.lambda_Rd2)
            disorder_array[i] = self.V0_Rd2* potential_sum
        
        logging.info(f"Generated random amplitude disorder, impurities: {N_d}")
        return disorder_array

    def disorder_potential(self, site_index):
        """返回给定位点的无序势矩阵"""
        if 0 <= site_index < len(self.disorder_distribution):
            V_s = self.disorder_distribution[site_index]
            
            if self.disorder_type == 'nonhermitian':
                return V_s * np.kron(self.tau_0, self.sigma_0)
            else:
                return V_s * np.kron(self.tau_z, self.sigma_0)
        else:
            return np.zeros((4, 4))

    def block_hamiltonian(self, phi, region_type):
        """构建区域类型的哈密顿量块"""
        onsite = 2 * self.t - self.mu 
        H00 = onsite * np.kron(self.tau_z, self.sigma_0) + \
              self.B * np.kron(self.tau_z, self.sigma_x) * np.sqrt(self.mu_lead**2 + self.delta**2)
        H01 = -self.t * np.kron(self.tau_z, self.sigma_0) + 1j * self.alpha * np.kron(self.tau_z, self.sigma_y)
        
        if region_type == 'SC':
            H00 = H00 + self.delta * (np.exp(1j * phi) * np.kron(self.tau_plus, 1j*self.sigma_y) + 
                                 np.exp(-1j * phi) * np.kron(self.tau_minus, -1j*self.sigma_y))
            
        return H00, H01

    def construct_hamiltonian(self, phi):
        """构建SNS结的完整哈密顿量"""
        H00_L, H01_L = self.block_hamiltonian(0, 'SC')
        H00_R, H01_R = self.block_hamiltonian(phi, 'SC')
        H00_J, H01_J = self.block_hamiltonian(0, 'NM')
        
        return {
            'H00_L': H00_L, 'H01_L': H01_L,
            'H00_R': H00_R, 'H01_R': H01_R,
            'H00_J': H00_J, 'H01_J': H01_J
        }
    
    
    def build_slice_ham(self, Vbias, ham, max_sidebands):
        """构建Floquet哈密顿量切片，支持增量扩建"""
        # 检查是否可以使用增量扩建
        can_increment = (
            self.prev_sidebands is not None and 
            max_sidebands > self.prev_sidebands and
            self.cached_g0_inv_FKs is not None and 
            self.cached_hop_FKs is not None
        )
        
        if can_increment:
            logging.info(f"使用增量扩建: {self.prev_sidebands} -> {max_sidebands} 边带")
            g0_inv_FKs, hop_FKs = self._incremental_expand_slice_ham(Vbias, ham, max_sidebands)
        else:
            # 完整构建模式
            logging.info(f"完整构建模式: {max_sidebands} 边带")
            g0_inv_FKs, hop_FKs = self._full_build_slice_ham(Vbias, ham, max_sidebands)
        
        # 更新缓存
        self.prev_sidebands = max_sidebands
        self.cached_g0_inv_FKs = g0_inv_FKs
        self.cached_hop_FKs = hop_FKs
        
        return g0_inv_FKs, hop_FKs

    def _incremental_expand_slice_ham(self, Vbias, ham, new_max_sidebands):
        """增量扩建Floquet哈密顿量切片"""
        # 检查缓存是否可用
        if (self.prev_sidebands is None or 
            self.cached_g0_inv_FKs is None or 
            self.cached_hop_FKs is None):
            logging.info("缓存不可用，回退到完整构建")
            return self._full_build_slice_ham(Vbias, ham, new_max_sidebands)
            
        old_max_sidebands = self.prev_sidebands
        old_sideN = 2 * old_max_sidebands + 1
        new_sideN = 2 * new_max_sidebands + 1
        dim_old = 4 * old_sideN
        dim_new = 4 * new_sideN
        
        # 获取旧的矩阵列表
        old_g0_inv_FKs = self.cached_g0_inv_FKs
        old_hop_FKs = self.cached_hop_FKs
        
        # 新的矩阵列表
        g0_inv_FKs = []
        hop_FKs = []
        
        SNS_Len = self.N_SC * 2 + self.N_junction
        
        select_mid_i = self.select_mid_i   # self.N_SC + self.N_junction // 2 

        
        for site_i in range(SNS_Len):
            old_g0_inv = old_g0_inv_FKs[site_i]
            old_hop = old_hop_FKs[site_i]
            
            # 确定当前位点的哈密顿量块
            if self.N_SC <= site_i < self.N_SC + self.N_junction:
                ''' make sure, the hop strength is added in this reagion '''
                Ham_onsite = ham['H00_J']
                Ham_hop = ham['H01_J']
                Hop_e = Ham_hop * np.kron(self.tau_e, np.ones((2, 2))) * self.v_tau
                Hop_h = -1 * Ham_hop * np.kron(self.tau_h, np.ones((2, 2))) * self.v_tau
                is_junction = True
            elif site_i < self.N_SC:
                Ham_onsite = ham['H00_L']
                Ham_hop = ham['H01_L']
                is_junction = False
            else:
                Ham_onsite = ham['H00_R']
                Ham_hop = ham['H01_R']
                is_junction = False
            
            # 扩展g0_inv_FK: 块对角矩阵，上下各增加 (new_max_sidebands - old_max_sidebands) 个块
            num_bands_to_add = new_max_sidebands - old_max_sidebands
            diag_blocks = []
            
            # 上部新增的块（更负的边带）
            for band_i in range(-new_max_sidebands, -old_max_sidebands):
                diag_block = Ham_onsite - band_i * Vbias * np.eye(4)
                diag_blocks.append(diag_block)
            
            # 中间旧矩阵部分
            if sp.issparse(old_g0_inv):
                # 将旧的块对角矩阵转为块列表, from sparese to array 
                old_blocks = []
                for i in range(old_sideN):
                    start = i * 4
                    end = (i + 1) * 4
                    old_blocks.append(old_g0_inv[start:end, start:end].toarray())
                diag_blocks.extend(old_blocks)
            else:
                for n in range(old_sideN):
                    idx = slice(n*4, (n+1)*4)
                    diag_blocks.append(old_g0_inv[idx, idx])
            
            # 下部新增的块（更正边带）
            for band_i in range(old_max_sidebands+1, new_max_sidebands+1):
                diag_block = Ham_onsite - band_i * Vbias * np.eye(4)
                diag_blocks.append(diag_block)
            
            # 构建新的块对角矩阵
            if self.use_sparse and dim_new > 20:
                g0_inv_new = block_diag(diag_blocks, format='csc')
            else:
                g0_inv_new = np.zeros((dim_new, dim_new), dtype=np.complex128)
                for i, block in enumerate(diag_blocks):
                    start_idx = i * 4
                    end_idx = (i+1) * 4
                    g0_inv_new[start_idx:end_idx, start_idx:end_idx] = block
            
            # 扩展hop_FK: 同样需要扩展
            if self.use_sparse and dim_new > 20:
                hop_new = lil_matrix((dim_new, dim_new), dtype=np.complex128)
            else:
                hop_new = np.zeros((dim_new, dim_new), dtype=np.complex128)
            
            # 将旧的跳跃矩阵放入新矩阵的中间部分
            old_start = num_bands_to_add * 4
            old_end = old_start + dim_old
            if sp.issparse(hop_new):
                hop_new[old_start:old_end, old_start:old_end] = old_hop
            else:
                hop_new[old_start:old_end, old_start:old_end] = old_hop
            
            # 对于特殊位点（结区中心），需要添加新的耦合项
            is_special_site = (site_i == select_mid_i) # or site_i == select_mid_i -1 
            
            if is_special_site and is_junction:
                # 注意：新增的边带也需要耦合
                for n in range(new_sideN):
                    band_i = n - new_max_sidebands
                    start_idx = n * 4
                    end_idx = (n+1)*4
                    
                    # 上边带耦合（从当前边带到下一个边带）
                    if band_i != new_max_sidebands and Vbias >= 0:
                        next_idx = (n+1) * 4
                        if next_idx < dim_new:
                            if sp.issparse(hop_new):
                                hop_new[start_idx:end_idx, next_idx:next_idx+4] = Hop_e
                            else:
                                hop_new[start_idx:end_idx, next_idx:next_idx+4] = Hop_e
                    
                    # 下边带耦合（从当前边带到前一个边带）
                    if band_i != -new_max_sidebands and Vbias >= 0:
                        prev_idx = (n-1) * 4
                        if prev_idx >= 0:
                            if sp.issparse(hop_new):
                                hop_new[start_idx:end_idx, prev_idx:prev_idx+4] = Hop_h.conj().T
                            else:
                                hop_new[start_idx:end_idx, prev_idx:prev_idx+4] = Hop_h.conj().T
            
            if sp.issparse(hop_new):
                hop_new = hop_new.tocsc()
            
            g0_inv_FKs.append(g0_inv_new)
            hop_FKs.append(hop_new)
    
        return g0_inv_FKs, hop_FKs
    
    def _full_build_slice_ham(self, Vbias, ham, max_sidebands):
        """完整构建Floquet哈密顿量切片"""
        sideN = 2 * max_sidebands + 1
        dim = 4 * sideN
        g0_inv_FKs = []
        hop_FKs = []
        
        SNS_Len = self.N_SC * 2 + self.N_junction
        SN_Len = self.N_SC + self.N_junction
        
        select_mid_i = self.select_mid_i
        
        for site_i in range(SNS_Len):        
            # 确定当前位点的哈密顿量块
            if self.N_SC <= site_i < SN_Len:
                Ham_onsite = ham['H00_J']
                Ham_hop = ham['H01_J']
                Hop_e = Ham_hop * np.kron(self.tau_e, np.ones((2, 2))) * self.v_tau
                Hop_h = -1 * Ham_hop * np.kron(self.tau_h, np.ones((2, 2))) * self.v_tau
                is_junction = True
            elif site_i < self.N_SC:
                Ham_onsite = ham['H00_L']
                Ham_hop = ham['H01_L']
                is_junction = False
            else:
                Ham_onsite = ham['H00_R']
                Ham_hop = ham['H01_R']
                is_junction = False
            
            # 添加无序势
            Ham_onsite = Ham_onsite + self.disorder_strength * self.disorder_potential(site_i)
            
            # 稀疏矩阵构建
            if self.use_sparse and dim > 20:
                # 构建对角块
                diag_blocks = []
                for n in range(sideN):
                    band_i = n - max_sidebands
                    diag_block = Ham_onsite - band_i * Vbias * np.eye(4)
                    diag_blocks.append(diag_block)
                
                # 创建块对角矩阵
                g0_inv_FK0 = block_diag(diag_blocks, format='csc')
                
                # 创建跳跃矩阵
                hop_FK0 = lil_matrix((dim, dim), dtype=np.complex128)
                
                # 填充主对角块
                for n in range(sideN):
                    band_i = n - max_sidebands
                    idx = slice(n * 4, (n + 1) * 4)
                    
                    # 特殊位点处理
                    is_special_site = (site_i == select_mid_i ) # or site_i == select_mid_i-1
                    
                    # is_special_site = (site_i == (self.N_SC + self.N_junction // 2) or 
                    #                    site_i == (self.N_SC + self.N_junction // 2))
                    
                    if is_special_site and is_junction:
                        # 上边带耦合
                        if band_i != max_sidebands and Vbias >= 0:
                            next_idx = slice((n + 1) * 4, (n + 2) * 4)
                            hop_FK0[idx, next_idx] = Hop_e
                        
                        # 下边带耦合
                        if band_i != -max_sidebands and Vbias >= 0:
                            prev_idx = slice((n - 1) * 4, n * 4)
                            hop_FK0[idx, prev_idx] = Hop_h.conj().T
                    else:
                        hop_FK0[idx, idx] = Ham_hop
                        
                hop_FK0 = hop_FK0.tocsc()
            else:
                # 稠密矩阵路径
                g0_inv_FK0 = np.zeros((dim, dim), dtype=np.complex128)
                hop_FK0 = np.zeros_like(g0_inv_FK0)
                
                for n in range(sideN):
                    band_i = n - max_sidebands
                    idx = slice(n * 4, (n + 1) * 4)
                    g0_inv_FK0[idx, idx] = Ham_onsite - band_i * Vbias * np.eye(4)
                    
                    # 特殊位点处理
                    is_special_site = (site_i == select_mid_i ) # or site_i == select_mid_i-1
                    
                    # is_special_site = (site_i == (self.N_SC) or 
                    #                    site_i == (self.N_SC + self.N_junction // 2))
                    
                    if is_special_site and is_junction:
                        if band_i != max_sidebands and Vbias >= 0:
                            next_idx = slice((n + 1) * 4, (n + 2) * 4)
                            hop_FK0[idx, next_idx] = Hop_e
                        if band_i != -max_sidebands and Vbias >= 0:
                            prev_idx = slice((n - 1) * 4, n * 4)
                            hop_FK0[idx, prev_idx] = Hop_h.conj().T
                    else:
                        hop_FK0[idx, idx] = Ham_hop
            
            g0_inv_FKs.append(g0_inv_FK0)
            hop_FKs.append(hop_FK0)
        
        return g0_inv_FKs, hop_FKs

class GreenFunctionCalculator:
    def __init__(self, params):
        # 计算参数
        self.recursion_depth = params['recursion_depth']
        self.eta = params['eta']  # 虚部无穷小
        self.mu_lead = params['mu_lead']  # 引线化学势
        self.delta = params['delta']  # 超导能隙
        self.unit_factor_nA = 1  #2 * 1.6e-19 * 1e-3 / (4.13e-15) * 1e9  # 单位转换因子 (nA) (2e/h) not (2e/hbar)
        
        # 稀疏计算配置
        self.use_sparse = params.get('use_sparse', True)
        self.sparse_threshold = params.get('sparse_threshold', 0.3)
        self.sparse_depth_limit = params.get('sparse_depth_limit', 5)
        
        # 预分解器缓存
        self.factorized_cache = {}
        
        logging.info(f"GreenFunctionCalculator initialized, Sparse mode: {self.use_sparse}") 

    def fermi_dirac(self, omega):
        """费米狄拉克分布"""
        kBT = 5e-3 * self.delta  # 温度 (meV)
        return 1.0 / (1.0 + np.exp((omega - self.mu_lead) / kBT))

    def surface_gf_sc(self, omega, H00, H01, direction='left'):
        """表面格林函数计算（超导引线）"""
        E0 = omega + 1j * self.eta
        dim = H00.shape[0]
        eps = eps_s = H00.copy()
        
        if direction == 'right':
            alpha = H01
            beta = H01.conj().T
        else:  # 'left'
            alpha = H01.conj().T
            beta = H01
        
        # 递归计算
        for _ in range(self.recursion_depth):
            g = la.inv(E0 * np.eye(dim) - eps)
            eps_s = eps_s + alpha @ g @ beta
            eps = eps + alpha @ g @ beta + beta @ g @ alpha
            
            alpha_new = alpha @ g @ alpha
            beta_new = beta @ g @ beta
            
            if np.linalg.norm(alpha_new) < 1e-12:
                break
            alpha, beta = alpha_new, beta_new
        
        g_surface = la.inv(E0 * np.eye(dim) - eps_s)
        hgh = H01 @ g_surface @ H01.conj().T if direction == 'right' else H01.conj().T @ g_surface @ H01
        
        return g_surface, hgh

    def recursive_sweep(self, omega, sE_initial, range_sets, hop_FKs, g0_inv_FKs):
        """递归扫描，完全兼容稀疏和稠密矩阵，优化稀疏矩阵操作"""
        range_1, range_2, offset_L = range_sets
        sE_L0, sE_R0, sE_Lf0, sE_Rf0 = sE_initial
        
        # 获取矩阵维度
        first_mat = g0_inv_FKs[0]
        dim = first_mat.shape[0]
        
        # 创建单位矩阵（兼容稀疏和稠密）
        if sp.issparse(g0_inv_FKs[0]):
            E0_I0 = (omega + 1j * self.eta) * sp.eye(dim, format='csc', dtype=np.complex128)
        else:
            E0_I0 = (omega + 1j * self.eta) * np.eye(dim, dtype=np.complex128)
        
        # 初始化列表
        g0s_invL = []
        sELs = []
        sERs = []
        sELs_less = []
        sERs_less = []
        
        # 初始化中间变量
        GL_i = None
        GR_i = None
        GL_less_i = None
        GR_less_i = None
        
        for recur_i in range(range_1, range_2):
            recur_i_R = range_2 - 1 - recur_i + offset_L
            
            # 获取当前矩阵
            g0_inv_i = g0_inv_FKs[recur_i]
            g0_inv_iR = g0_inv_FKs[recur_i_R]
            alpha_i = hop_FKs[recur_i]
            alpha_iR = hop_FKs[recur_i_R]
            
            # 初始自能处理
            if recur_i == range_1 or recur_i_R == range_2 - 1:
                sEL_i = sE_L0
                sER_i = sE_R0
                sEL_less_i = sE_Lf0.conj().T - sE_Lf0
                sER_less_i = sE_Rf0.conj().T - sE_Rf0
            else:
                # 使用稀疏矩阵乘法避免转换
                if sp.issparse(alpha_i) and sp.issparse(GL_i):
                    sEL_i = alpha_i.conj().T @ GL_i @ alpha_i
                else:
                    # 确保矩阵为稠密后再进行运算
                    if sp.issparse(GL_i):
                        GL_i = GL_i.toarray()
                    if sp.issparse(alpha_i):
                        alpha_i = alpha_i.toarray()
                    sEL_i = alpha_i.conj().T @ GL_i @ alpha_i
                
                if sp.issparse(alpha_iR) and sp.issparse(GR_i):
                    sER_i = alpha_iR @ GR_i @ alpha_iR.conj().T
                else:
                    if sp.issparse(GR_i):
                        GR_i = GR_i.toarray()
                    if sp.issparse(alpha_iR):
                        alpha_iR = alpha_iR.toarray()
                    sER_i = alpha_iR @ GR_i @ alpha_iR.conj().T
                
                # 同样处理lesser分量
                if sp.issparse(alpha_i) and sp.issparse(GL_less_i):
                    sEL_less_i = alpha_i.conj().T @ GL_less_i @ alpha_i
                else:
                    if sp.issparse(GL_less_i):
                        GL_less_i = GL_less_i.toarray()
                    if sp.issparse(alpha_i):
                        alpha_i = alpha_i.toarray()
                    sEL_less_i = alpha_i.conj().T @ GL_less_i @ alpha_i
                
                if sp.issparse(alpha_iR) and sp.issparse(GR_less_i):
                    sER_less_i = alpha_iR @ GR_less_i @ alpha_iR.conj().T
                else:
                    if sp.issparse(GR_less_i):
                        GR_less_i = GR_less_i.toarray()
                    if sp.issparse(alpha_iR):
                        alpha_iR = alpha_iR.toarray()
                    sER_less_i = alpha_iR @ GR_less_i @ alpha_iR.conj().T
            
            # 构建逆格林函数
            gi_inv_L = E0_I0 - g0_inv_i
            gi_inv_R = E0_I0 - g0_inv_iR
            
            # 预分解矩阵以提高性能
            cache_key_L = (id(gi_inv_L), id(sEL_i))
            if cache_key_L in self.factorized_cache:
                solve_L = self.factorized_cache[cache_key_L]
                GL_i = solve_L(sp.eye(dim, format='csc', dtype=np.complex128))
            else:
                mat_to_inv = gi_inv_L - sEL_i
                if sp.issparse(mat_to_inv):
                    solve_L = factorized(mat_to_inv.tocsc())
                    GL_i = solve_L(sp.eye(dim, format='csc', dtype=np.complex128))
                    self.factorized_cache[cache_key_L] = solve_L
                else:
                    GL_i = la.inv(mat_to_inv)
            
            cache_key_R = (id(gi_inv_R), id(sER_i))
            if cache_key_R in self.factorized_cache:
                solve_R = self.factorized_cache[cache_key_R]
                GR_i = solve_R(sp.eye(dim, format='csc', dtype=np.complex128))
            else:
                mat_to_inv = gi_inv_R - sER_i
                if sp.issparse(mat_to_inv):
                    solve_R = factorized(mat_to_inv.tocsc())
                    GR_i = solve_R(sp.eye(dim, format='csc', dtype=np.complex128))
                    self.factorized_cache[cache_key_R] = solve_R
                else:
                    GR_i = la.inv(mat_to_inv)
            
            # 计算lesser分量
            if sp.issparse(GL_i):
                GL_less_i = GL_i @ sEL_less_i @ GL_i.conj().T
            else:
                GL_less_i = GL_i @ sEL_less_i @ GL_i.conj().T
            
            if sp.issparse(GR_i):
                GR_less_i = GR_i @ sER_less_i @ GR_i.conj().T
            else:
                GR_less_i = GR_i @ sER_less_i @ GR_i.conj().T
            
            # 存储结果
            sELs.append(sEL_i)
            sERs.append(sER_i)
            sELs_less.append(sEL_less_i)
            sERs_less.append(sER_less_i)
            g0s_invL.append(gi_inv_L)
        
        return g0s_invL, [sELs, sERs, sELs_less, sERs_less]

    def compute_self_energies(self, omega, lead_type, ham, max_sidebands, Vbias, N_SC, N_junction, hop_FKs, g0_inv_FKs):
        """计算自能，完全兼容稀疏和稠密矩阵"""
        sideN = 2 * max_sidebands + 1
        dim = 4 * sideN
        
        # 创建稠密矩阵
        sEL = np.zeros((dim, dim), dtype=np.complex128)
        sER = np.zeros_like(sEL)
        sEL_f0 = np.zeros_like(sEL)
        sER_f0 = np.zeros_like(sEL)
        
        for n in range(sideN):
            band_i = n - max_sidebands
            omega_shifted = omega + band_i * Vbias
            f0 = self.fermi_dirac(omega_shifted)
            a = slice(n * 4, n * 4 + 4)
            
            # 左侧超导引线
            _, sEL_block = self.surface_gf_sc(omega_shifted, ham['H00_L'], ham['H01_L'], direction='left')
            sEL[a, a] = sEL_block
            sEL_f0[a, a] = sEL_block * f0
            
            # 右侧超导引线
            _, sER_block = self.surface_gf_sc(omega_shifted, ham['H00_R'], ham['H01_R'], direction='right')
            sER[a, a] = sER_block
            sER_f0[a, a] = sER_block * f0
        
        if lead_type == 'infinite':
            return [None, None], [sEL, sER, sEL_f0, sER_f0]
        
        # 处理有限引线
        sE_initial = [sEL, sER, sEL_f0, sER_f0]
        
        # 处理左侧引线区域
        g0_inv_Ld, sE_Ld = self.obtain_recursive_list(
            omega, 'left_lead', sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        # 处理右侧引线区域
        g0_inv_Rd, sE_Rd = self.obtain_recursive_list(
            omega, 'right_lead', sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        # 确保矩阵为稠密
        if sp.issparse(g0_inv_Ld[-1]):
            g0_inv_L = g0_inv_Ld[-1].toarray()
        else:
            g0_inv_L = g0_inv_Ld[-1]
        
        if sp.issparse(g0_inv_Rd[-1]):
            g0_inv_R = g0_inv_Rd[-1].toarray()
        else:
            g0_inv_R = g0_inv_Rd[-1]
        
        # 获取自能
        sEL_left = sE_Ld[0][-1] if not sp.issparse(sE_Ld[0][-1]) else sE_Ld[0][-1].toarray()
        sER_left = sE_Ld[1][0] if not sp.issparse(sE_Ld[1][0]) else sE_Ld[1][0].toarray()
        sEL_right = sE_Rd[0][0] if not sp.issparse(sE_Rd[0][0]) else sE_Rd[0][0].toarray()
        sER_right = sE_Rd[1][-1] if not sp.issparse(sE_Rd[1][-1]) else sE_Rd[1][-1].toarray()
        
        # 计算格林函数
        g_retard_L = la.inv(g0_inv_L - sEL_left - sER_left)
        g_retard_R = la.inv(g0_inv_R - sEL_right - sER_right)
        
        # 更新自能
        sEL = np.zeros_like(sEL)
        sER = np.zeros_like(sER)
        sEL_f0 = np.zeros_like(sEL_f0)
        sER_f0 = np.zeros_like(sER_f0)
        
        hop_LC = hop_FKs[N_SC - 1]
        hop_CR = hop_FKs[N_SC + N_junction]
        
        # 确保跳跃矩阵为稠密
        if sp.issparse(hop_LC):
            hop_LC = hop_LC.toarray()
        if sp.issparse(hop_CR):
            hop_CR = hop_CR.toarray()
        
        for n in range(sideN):
            band_i = n - max_sidebands
            omega_shifted = omega + band_i * Vbias
            f0 = self.fermi_dirac(omega_shifted)
            a = slice(n * 4, n * 4 + 4)
            
            sEL[a, a] = hop_LC[a, a].conj().T @ g_retard_L[a, a] @ hop_LC[a, a]
            sER[a, a] = hop_CR[a, a] @ g_retard_R[a, a] @ hop_CR[a, a].conj().T
            sEL_f0[a, a] = sEL[a, a] * f0
            sER_f0[a, a] = sER[a, a] * f0
        
        return [g_retard_L, g_retard_R], [sEL, sER, sEL_f0, sER_f0]

    def obtain_recursive_list(self, omega, region_type, sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs):
        """获取递归列表（保持不变）"""
        if region_type == 'left_lead':
            sE_initial = [sE * i0 for sE, i0 in zip(sE_initial, [1, 0, 1, 0])]
            range_sets = [0, N_SC, 0]
            recur_depth = N_SC
        elif region_type == 'right_lead':
            sE_initial = [sE * i0 for sE, i0 in zip(sE_initial, [0, 1, 0, 1])]
            range_sets = [N_SC + N_junction, 2 * N_SC + N_junction, N_SC + N_junction]
            recur_depth = N_SC
        elif region_type == 'junction':
            range_sets = [N_SC, N_SC + N_junction, N_SC]
            recur_depth = N_junction
        
        return self.recursive_sweep(omega, sE_initial, range_sets, hop_FKs, g0_inv_FKs)

    def _compute_current_at_omega(self, omega, lead_type, region_type, ham_builder, max_sidebands, Vbias, N_SC, N_junction, g0_inv_FKs, hop_FKs):
        """电流计算，使用传入的哈密顿量切片"""
        # 计算自能
        Gs, sE_initial = self.compute_self_energies(
            omega, lead_type, ham_builder.construct_hamiltonian(0), max_sidebands, Vbias, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        # 获取递归列表
        g0_inv, sE4s_leads = self.obtain_recursive_list(
            omega, region_type, sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        recur_depth = N_junction
        
        # 准备tau_z矩阵
        sideN = 2 * max_sidebands + 1
        tau_z_matrix = np.kron(np.eye(sideN), np.kron(ham_builder.tau_z, np.eye(2)))
        
        trace0 = 0
        trace_set = []
        for recur_i in range(recur_depth):
            recur_i_R = recur_depth - 1 - recur_i
            
            # 获取当前矩阵并确保为稠密
            gi_inv = g0_inv[recur_i]
            if sp.issparse(gi_inv):
                gi_inv = gi_inv.toarray()
            
            sEL_i = sE4s_leads[0][recur_i]
            if sp.issparse(sEL_i):
                sEL_i = sEL_i.toarray()
            
            sER_i = sE4s_leads[1][recur_i_R]
            if sp.issparse(sER_i):
                sER_i = sER_i.toarray()
            
            sEL_less_i = sE4s_leads[2][recur_i]
            if sp.issparse(sEL_less_i):
                sEL_less_i = sEL_less_i.toarray()
            
            sER_less_i = sE4s_leads[3][recur_i_R]
            if sp.issparse(sER_less_i):
                sER_less_i = sER_less_i.toarray()
            
            # 使用稠密矩阵计算
            G_nn_ret = la.inv(gi_inv - sEL_i - sER_i)
            G_nn_adv = G_nn_ret.conj().T
            
            sEL_nn_less = sEL_less_i
            sE_nn_less = sEL_less_i + sER_less_i
            sEL_nn = sEL_i
            
            G_nn_less = G_nn_ret @ sE_nn_less @ G_nn_adv
            temp = (G_nn_ret @ sEL_nn_less + G_nn_less @ sEL_nn.conj().T) @ tau_z_matrix
            
            trace_val = np.trace(temp).real
            trace0 += trace_val * self.unit_factor_nA
            
            trace_set.append(trace_val * self.unit_factor_nA)
            
        return np.array(trace_set) #, trace0

class JosephsonJunctionSolver:
    def __init__(self, params, path_manager):
        # 添加稀疏配置参数
        params.setdefault('use_sparse', True)
        params.setdefault('sparse_threshold', 0.3)
        params.setdefault('sparse_depth_limit', 5)
        
        self.params = params
        self.path_manager = path_manager
        self.output_dir = path_manager.get_raw_data_dir()
        
        # 系统参数
        self.params = params
        self.ham_builder = HamiltonianBuilder(params)
        self.gf_calculator = GreenFunctionCalculator(params)
        
        # 计算参数
        self.max_sidebands = params['max_sidebands']
        self.omega_points = params['omega_points']
        self.Vbias_points = params['Vbias_points']
        self.B_points = params.get('B_points', 0)
        self.job_parallel = params['job_parallel']
        
        # 创建输出目录
        os.makedirs(self.output_dir, exist_ok=True)
        
        # 单位转换
        self.unit_factor_nA = 2 * 1.6e-19 * 1e-3 / (4.13e-15) * 1e9  # nA
        # 自适应计算参数
        self.adaptive_iv = params.get('adaptive_iv', False)
        self.adaptive_tol = params.get('adaptive_tol', 0.01)
        self.adaptive_max_N = params.get('adaptive_max_N', 20)
        self.adaptive_method = params.get('adaptive_method', 'basic')  # 'basic', 'dynamic', 'step'

        # 哈密顿量切片缓存
        self.ham_slice_cache = {}
        
        # 生成公共时间戳
        self.common_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        logging.info("JosephsonJunctionSolver initialized")
        logging.info(f"Superconductor: {params['N_SC']} sites, Junction: {params['N_junction']} sites")
        logging.info(f"Recursion depth: {params['recursion_depth']}, Max sidebands: {params['max_sidebands']}")

    def _get_base_metadata(self):
        """返回所有计算的通用元数据参数"""
        # 基础元数据，始终包含
        metadata = {
            "N_SC": self.params['N_SC'],
            "N_junction": self.params['N_junction'],
            "delta": self.params['delta'],
            "alpha": self.params['alpha'],
            "v_tau": self.params['v_tau'],
            "B": self.params['B'],
            "Vdis":self.params['disorder_strength'],
            "max_sidebands": self.params['max_sidebands'],
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "common_timestamp": self.common_timestamp  # 添加公共时间戳
        }
        
        # 添加disorder类型
        disorder_type = self.params.get('disorder_type', 'none')
        metadata["disorder_type"] = disorder_type
        
        # 根据disorder类型添加相关参数
        if disorder_type != 'none':
            if disorder_type == 'gaussian':
                metadata.update({
                    "Vdis": self.params.get('Vdis_gau', 0.0),
                    "decayL": self.params.get('decayL_gau', 50),
                    "Xdis": self.params.get('Xdis_gau', 150)
                })
            elif disorder_type == 'smooth':
                metadata.update({
                    "decayL": self.params.get('decayL_smooth', 50),
                    "Vdis": self.params.get('Vdis_smooth', 0.0),
                    "Vd": self.params.get('Vd_smooth', 0.8)
                })
            elif disorder_type in ['random_typeI', 'random_typeII']:
                metadata.update({
                    "N_imp": self.params.get('N_imp1', 52),
                    "lambda_imp": self.params.get('lambda_imp1', 18.0),
                    "V0": self.params.get('V0_imp1', 0.0),
                    "n_d": self.params.get('Nd_imp2', 10.0),
                    "random_lambda_imp": self.params.get('lambda_imp2', 20.0),
                    "random_V0": self.params.get('V0_imp2', 0.0),
                    "a0": self.params.get('a0', 10.0)
                })
            elif disorder_type == 'nonhermitian':
                metadata.update({
                    "charge_V0": self.params.get('charge_V0', 0.5),
                    "n_d": self.params.get('n_d', 10.0)
                })
            elif disorder_type == 'from_file':
                metadata.update({
                    "disorder_file": self.params.get('disorder_file', 'disorder_data.txt')
                })
        
        # 添加自适应参数
        if self.params.get('adaptive_iv', False):
            metadata.update({
                "adaptive_method": self.params.get('adaptive_method', 'basic'),
                "adaptive_tol": self.params.get('adaptive_tol', 0.01),
                "adaptive_max_N": self.params.get('adaptive_max_N', 20)
            })
        
        return metadata

    def save_disorder_data(self):
        """保存无序数据（如果disorder类型为'from_file'则跳过）"""
        # 如果disorder类型为'from_file'，则不保存
        if self.params.get('disorder_type', 'none') == 'from_file':
            logging.info("Disorder type is 'from_file', skipping disorder data saving")
            return {
                "csv_file": None,
                "plot_file": None,
                "metadata_file": None
            }
            
        disorder_dir = os.path.join(self.output_dir, "disorder_data")
        os.makedirs(disorder_dir, exist_ok=True)
        
        # 保存图表
        plot_file = self._plot_disorder_distribution(disorder_dir)
        
        # 保存CSV
        csv_file = self._save_disorder_csv(disorder_dir)
        
        # 保存元数据
        metadata_file = self._save_disorder_metadata(disorder_dir)
        
        return {
            "csv_file": csv_file,
            "plot_file": plot_file,
            "metadata_file": metadata_file
        }

    def _plot_disorder_distribution(self, output_dir):
        """绘制无序分布并保存为PNG"""
        # 使用公共时间戳
        plot_filename = os.path.join(output_dir, f"disorder_plot_{self.common_timestamp}.png")
        
        # 创建图表
        plt.figure(figsize=(12, 6))
        plt.plot(self.ham_builder.disorder_distribution, 'b-', linewidth=1.5)
        
        # 添加区域标记
        plt.axvline(x=self.params['N_SC'], color='r', linestyle='--', alpha=0.5)
        plt.axvline(x=self.params['N_SC'] + self.params['N_junction'], color='r', linestyle='--', alpha=0.5)
        
        # 标签和标题
        plt.title(f'Disorder Potential ({self.params.get("disorder_type", "none")})', fontsize=14)
        plt.xlabel('Site Index', fontsize=12)
        plt.ylabel('Potential (meV)', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.7)
        
        # 保存图表
        plt.savefig(plot_filename, dpi=300)
        plt.close()
        logging.info(f"Disorder plot saved: {plot_filename}")
        
        return plot_filename

    def _save_disorder_csv(self, output_dir):
        """保存无序分布到CSV文件"""
        # 使用公共时间戳
        csv_filename = os.path.join(output_dir, f"disorder_data_{self.common_timestamp}.csv")
        
        # 创建CSV文件
        df = pd.DataFrame({
            'site_index': np.arange(len(self.ham_builder.disorder_distribution)),
            'disorder_value': self.ham_builder.disorder_distribution
        })
        df.to_csv(csv_filename, index=False)
        logging.info(f"Disorder data saved: {csv_filename}")
        
        return csv_filename

    def _save_disorder_metadata(self, output_dir):
        """保存无序元数据到JSON文件，仅相关参数"""
        disorder_type = self.params.get('disorder_type', 'none')
        
        # 基础元数据
        metadata = {
            "disorder_type": disorder_type,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "common_timestamp": self.common_timestamp  # 添加公共时间戳
        }
        
        # 根据disorder类型添加相关参数
        if disorder_type != 'none':
            if disorder_type == 'gaussian':
                metadata.update({
                    "Vdis": self.params.get('Vdis_gau', 0.0),
                    "decayL": self.params.get('decayL_gau', 50),
                    "Xdis": self.params.get('Xdis_gau', 150)
                })
            elif disorder_type == 'smooth':
                metadata.update({
                    "decayL": self.params.get('decayL_smooth', 50),
                    "Vdis": self.params.get('Vdis_smooth', 0.0),
                    "Vd": self.params.get('Vd_smooth', 0.8)
                })
            elif disorder_type in ['random_typeI', 'random_typeII']:
                metadata.update({
                    "N_imp": self.params.get('N_imp1', 52),
                    "lambda_imp": self.params.get('lambda_imp1', 18.0),
                    "V0": self.params.get('V0_imp1', 0.0),
                    "n_d": self.params.get('Nd_imp2', 10.0),
                    "random_lambda_imp": self.params.get('lambda_imp2', 20.0),
                    "random_V0": self.params.get('V0_imp2', 0.0),
                    "a0": self.params.get('a0', 10.0)
                })
            elif disorder_type == 'nonhermitian':
                metadata.update({
                    "charge_V0": self.params.get('charge_V0', 0.5),
                    "n_d": self.params.get('n_d', 10.0)
                })
            elif disorder_type == 'from_file':
                metadata.update({
                    "disorder_file": self.params.get('disorder_file', 'disorder_data.txt')
                })
        
        # 使用公共时间戳
        metadata_file = os.path.join(output_dir, f"disorder_metadata_{self.common_timestamp}.json")
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=4)
        
        logging.info(f"Disorder metadata saved: {metadata_file}")
        
        return metadata_file

    def run_calculation(self, calc_type):
        """运行指定计算"""
        # 保存无序数据（如果disorder类型不是'from_file'）
        if self.params.get('disorder_type', 'none') != 'none' and self.params.get('disorder_type', 'none') != 'from_file':
            self.save_disorder_data()
        
        # 保存任务元数据
        self.path_manager.save_task_metadata(self.params)
        
        # 运行计算
        if calc_type == "DC_IV":
            metadata, *data = self.compute_dc_iv()
        elif calc_type == "DC_IV_Bsweep":
            metadata, *data = self.compute_dc_iv_Bsweep()
        else:
            raise ValueError("Invalid calculation type")
        
        # 保存结果
        filepath = self.save_results(metadata, data)
        
        # 绘图
        plot_path = JJPlotter.plot_single_result(metadata, data, self.path_manager.get_plots_dir())
        
        return filepath, plot_path
    
    def save_results(self, metadata, data):
        """保存计算结果到文件，自动处理一维和二维电流数据"""
        calc_type = metadata["calculation"]
        # 使用公共时间戳
        common_timestamp = metadata.get("common_timestamp", datetime.now().strftime("%Y%m%d_%H%M%S"))
        filename = f"{calc_type}_{common_timestamp}.csv"
        filepath = os.path.join(self.output_dir, filename)
        
        metadata_lines = ["# Calculation Metadata"]
        for key, value in metadata.items():
            metadata_lines.append(f"# {key}: {value}")
        
        # 根据计算类型创建数据框
        if calc_type == "DC_IV":
            Vbias_vals = data[0]
            currents = data[1]  # 可能是标量或数组
            sideband = data[2] if len(data) > 2 else None
            
            # 检查电流数据的维度
            currents_array = np.array(currents)
            
            if currents_array.ndim == 1:
                # 一维：总电流或单个site的电流
                df_data = {
                    'Bias Voltage (meV)': Vbias_vals,
                    'Current (nA)': currents
                }
                if sideband is not None:
                    df_data['SidebandN'] = sideband
                df = pd.DataFrame(df_data)
                
            elif currents_array.ndim == 2:
                # 二维：每个site的电流贡献 (Vbias点数 × site数)
                n_sites = currents_array.shape[1]
                
                # 创建数据字典
                df_data = {'Bias Voltage (meV)': Vbias_vals}
                
                # 添加每个site的电流列
                for site_idx in range(n_sites):
                    df_data[f'Current_site_{site_idx}'] = currents_array[:, site_idx]
                
                # 添加总电流列（所有site之和）
                df_data['Current_total'] = currents_array.sum(axis=1)
                
                if sideband is not None:
                    # 边带数可能是数组或标量
                    if hasattr(sideband, '__len__') and len(sideband) == len(Vbias_vals):
                        df_data['SidebandN'] = sideband
                    else:
                        df_data['SidebandN'] = sideband
                
                df = pd.DataFrame(df_data)
            
            else:
                raise ValueError(f"Unexpected current data dimension: {currents_array.ndim}")
                
        elif calc_type == "DC_IV_Bsweep":
            B_vals = data[0]
            currents = data[1]  # 可能是标量或数组
            sideband = data[2] if len(data) > 2 else None
            
            # 检查电流数据的维度
            currents_array = np.array(currents)
            
            if currents_array.ndim == 1:
                # 一维：总电流
                df_data = {
                    'Magnetic Field': B_vals,
                    'Current (nA)': currents,
                    'Fixed Bias (meV)': metadata['fixed_Vbias']
                }
                if sideband is not None:
                    df_data['SidebandN'] = sideband
                df = pd.DataFrame(df_data)
                
            elif currents_array.ndim == 2:
                # 二维：每个site的电流贡献
                n_sites = currents_array.shape[1]
                
                # 创建数据字典
                df_data = {
                    'Magnetic Field': B_vals,
                    'Fixed Bias (meV)': metadata['fixed_Vbias']
                }
                
                # 添加每个site的电流列
                for site_idx in range(n_sites):
                    df_data[f'Current_site_{site_idx}'] = currents_array[:, site_idx]
                
                # 添加总电流列
                df_data['Current_total'] = currents_array.sum(axis=1)
                
                if sideband is not None:
                    # 边带数可能是数组或标量
                    if hasattr(sideband, '__len__') and len(sideband) == len(B_vals):
                        df_data['SidebandN'] = sideband
                    else:
                        df_data['SidebandN'] = sideband
                
                df = pd.DataFrame(df_data)
            
            else:
                raise ValueError(f"Unexpected current data dimension: {currents_array.ndim}")
        
        # 写入文件
        with open(filepath, 'w') as f:
            f.write("\n".join(metadata_lines))
            f.write("\n")
            df.to_csv(f, index=False)
        
        logging.info(f"Results saved to: {filepath}")
        return filepath
    
    def tics(self, stamp=False):
        """Start timer """
        if stamp:
            print(datetime.now().strftime("Starting at %d/%m/%Y %H:%M:%S"))
        return time.time()

    def toc(self, tic, disp=True, nice_format=False, stamp=False):
        """Stop timer and display elapsed time"""
        elapsed_time = time.time() - tic
        if stamp:
            print(datetime.now().strftime("Finished at %d/%m/%Y %H:%M:%S"))
        if disp:
            if (not nice_format) or (elapsed_time < 60):
                print(f"Elapsed time: {np.round(elapsed_time,3)} seconds")
            else:
                hrs, rem = divmod(elapsed_time, 3600)
                mins, secs = divmod(rem, 60)
                print(f"Elapsed time: {int(hrs):02d}:{int(mins):02d}:{int(secs):02d}")
        return elapsed_time
        
    @staticmethod
    def generate_nonuniform_grid(
        start: float,           # 整个区域起始点
        end: float,             # 整个区域结束点
        dense_start: float,     # 期望的dense区域起点
        dense_end: float,       # 期望的dense区域终点
        space_non_dense: float = 0.05,  # 非dense区域间距 (默认0.05)
        space_ratio: float = 0.4,       # dense区域间距比例 (默认0.4)
        max_points: int = 300           # 最大总点数 (默认300)
        ) ->np.ndarray:
        """ """
        space_dense = space_non_dense * space_ratio
        
        # 调整dense区域边界
        n_left_segments = max(0, int(np.floor((dense_start - start) / space_non_dense)))
        adjusted_dense_start = start + n_left_segments * space_non_dense

        max_dense_end = end - space_non_dense  
        
        if max_dense_end > adjusted_dense_start:   # dense区域右侧至少有一个点
            min_segments = max(0, int(np.ceil((dense_end - adjusted_dense_start) / space_dense)))
            max_segments = int(np.floor((max_dense_end - adjusted_dense_start) / space_dense))
            n_dense_segments = min(min_segments, max_segments) if min_segments > 0 else max_segments
            adjusted_dense_end = adjusted_dense_start + n_dense_segments * space_dense
        else:
            n_dense_segments = 0
            adjusted_dense_end = adjusted_dense_start
        
        # 计算各区域点数
        n_left = n_left_segments
        n_dense = n_dense_segments + 1 if n_dense_segments > 0 else 0
        n_right_segments = max(0, int(np.floor((end - adjusted_dense_end) / space_non_dense)))
        n_right = n_right_segments
        
        points = []
        
        # 左侧非dense区域
        if n_left > 0:
            left_points = np.linspace(start, adjusted_dense_start, n_left + 1, endpoint=False)
            points.extend(left_points)
        
        # dense区域
        if n_dense > 0:
            dense_points = np.linspace(adjusted_dense_start, adjusted_dense_end, n_dense)
            points.extend(dense_points)
        
        # 右侧非dense区域
        if n_right > 0:
            right_start = adjusted_dense_end + space_non_dense
            right_points = np.linspace(
                right_start, 
                right_start + n_right * space_non_dense, 
                n_right + 1, 
                endpoint=True
            )
            # 确保不超过end
            right_points = right_points[right_points <= end]
            points.extend(right_points)
        
        points = np.array(points)
        points.sort()
        
        # 确保包含起点和终点
        if abs(points[0] - start) > 1e-10:
            points = np.insert(points, 0, start)
        if abs(points[-1] - end) > 1e-10:
            points = np.append(points, end)
        
        # 点数限制
        if len(points) > max_points:
            logging.warning(f"网格点数({len(points)})超过最大限制({max_points})，进行截断")
            points = points[:max_points]
        
        return points

    @staticmethod
    def generate_uniform_grid(
        start: float,           # 区域起始点
        end: float,             # 区域结束点
        spacing: float = 0.05,  # 点间距 (默认0.05)
        max_points: int = 300   # 最大总点数 (默认300)
        ) -> np.ndarray:
        """ """
        num_segments = (end - start) / spacing
        n_points = int(np.floor(num_segments)) + 1
        
        # 调整间距以满足最大点数限制
        if n_points > max_points:
            n_points = max_points
            spacing = (end - start) / (n_points - 1) if n_points > 1 else 0
        
        points = np.linspace(start, end, n_points)
        return points
        
    def generate_Vbias_vals(self):
        """Generate Vbias values using the same logic as compute_dc_iv"""
        # delta = self.params['delta']
        # Config. [0.03,3,0.03,2.1,0.03,0.4,300]
        # Config. [0.03,3,0.12,1.2,0.04,0.4,300]
        
        Vbias_max_ratio = self.params.get('Vbias_max_ratio', 2.0)
        
        if Vbias_max_ratio <= 2.2:
            return self.generate_uniform_grid(0.01, (Vbias_max_ratio+0.01),
                                              0.01,self.params['Vbias_points'])  
        else:
            # compute_dc_iv 
            return self.generate_nonuniform_grid(
                start = 0.01,   ## previous, set as 0.001
                end = Vbias_max_ratio ,
                dense_start= 0.12,
                dense_end = 1.2,
                space_non_dense =0.04,
                space_ratio= 0.25,
                max_points = self.params['Vbias_points']
            )   
            
    def compute_dc_iv(self):
        """Compute DC I-V characteristic, returns (metadata, Vbias_vals, currents, sideband_Ns)"""
        start_time = time.time()
        delta = self.params['delta']
        
        Vbias_vals= delta*self.generate_Vbias_vals()
        
        logging.info("Starting DC IV calculation")
        results = Parallel(n_jobs=self.job_parallel[0])(
            delayed(self.compute_parallel_aux)(x0, "DC_IV") for x0 in Vbias_vals #[::-1]
        )
        
        # 分离电流和边带数
        currents_list = [res[0] for res in results]  # 每个元素是一个数组（总电流或每个site的电流）
        sideband_Ns = [res[1] for res in results]
        
        # 检查电流数据的维度
        first_current = currents_list[0]
        if isinstance(first_current, (int, float, complex, np.number)):
            # 标量电流：转换为数组
            currents = np.array(currents_list)
        else:
            # 数组电流：确保形状一致
            currents = np.array(currents_list)
            logging.info(f"Current data shape: {currents.shape}, each site's contribution")
        
        # Prepare metadata
        metadata = self._get_base_metadata()
        metadata.update({
            "calculation": "DC_IV",
            "duration": time.time() - start_time,
            "Vbias_points": self.Vbias_points, 
            "current_data_type": "scalar" if currents.ndim == 1 else f"array_{currents.shape[1]}_sites"
        })
        
        logging.info(f"DC IV calculation completed, duration: {metadata['duration']:.2f} sec")
        logging.info(f"Current data: {metadata['current_data_type']}")
        
        return metadata, Vbias_vals, currents, sideband_Ns
    

    def compute_parallel_aux(self, Vari_input, calc_type):
        """并行计算辅助函数（使用稀疏优化）"""
        if calc_type == "DC_IV":
            Vbias = Vari_input
            if self.params['adaptive_iv']:
                if self.params['adaptive_method'] == 'advanced':
                    return self.adaptive_current_advanced(Vbias)
            else:
                current = self.compute_current_at_bias(Vbias)
                return current, self.params['max_sidebands']
                
        elif calc_type == "DC_IV_Bsweep":
            B, Vbias = Vari_input  # 解包磁场和偏压
            # 更新磁场值
            self.ham_builder.B = B
            
            # 计算当前磁场和偏压下的电流
            if self.params['adaptive_iv']:
                current, sideband = self.compute_parallel_aux(Vbias, "DC_IV")
            else:
                current = self.compute_current_at_bias(Vbias)
                sideband = self.params['max_sidebands']
            
            return current, sideband
        
    def compute_dc_iv_Bsweep(self):
        """在固定偏压下扫描磁场计算直流电流"""
        start_time = time.time()
        Vbias = self.params['fixed_Vbias']  # 获取固定偏压值
        B_vals = self.generate_uniform_grid(
            start=0,
            end=self.params['B_max'],
            spacing=0.02,
            max_points=self.B_points
        )
        
        logging.info(f"Starting DC IV with B sweep @ Vbias={Vbias} meV")
        results = Parallel(n_jobs=self.job_parallel[0])(
            delayed(self.compute_parallel_aux)((B, Vbias), "DC_IV_Bsweep") 
            for B in B_vals
        )

        # 分离电流和边带数
        currents_list = [res[0] for res in results]
        sideband_Ns = [res[1] for res in results]
        
        # 检查电流数据的维度
        first_current = currents_list[0]
        if isinstance(first_current, (int, float, complex, np.number)):
            # 标量电流：转换为数组
            currents = np.array(currents_list)
        else:
            # 数组电流：确保形状一致
            currents = np.array(currents_list)
            logging.info(f"Current data shape: {currents.shape}, each site's contribution")

        # 准备元数据
        metadata = self._get_base_metadata()
        metadata.update({
            "calculation": "DC_IV_Bsweep",
            "duration": time.time() - start_time,
            "fixed_Vbias": Vbias,
            "B_points": self.B_points,
            "B_max": self.params['B_max'],
            "current_data_type": "scalar" if currents.ndim == 1 else f"array_{currents.shape[1]}_sites"
        })
        
        logging.info(f"DC IV with B sweep completed, duration: {metadata['duration']:.2f} sec")
        
        return metadata, B_vals, currents, sideband_Ns
    
    def adaptive_current_advanced(self, Vbias):
        """改进的自适应策略（增量版本）"""
        # 获取自适应配置
        init_N, step_size, max_N = self._get_adaptive_config(Vbias)
        
        # 重置哈密顿量缓存
        self.ham_builder.prev_sidebands = None
        self.ham_builder.cached_g0_inv_FKs = None
        self.ham_builder.cached_hop_FKs = None
        
        # 初始计算
        N = init_N
        I_current = self.compute_current_at_bias(Vbias, N)
        
        # 记录收敛历史
        history = [(N, I_current)]
        
        # 自适应迭代
        for iteration in range(15):  # 增加迭代次数限制
            # 检查是否达到最大边带数
            if N >= max_N:
                logging.warning(f"达到最大边带数 {max_N} @ Vbias={Vbias:.4f}Δ")
                # 返回历史中最接近收敛的结果
                return self._select_best_result(history)
            
            # 增加边带数
            new_N = N + step_size
            logging.debug(f"迭代 {iteration}: 扩展边带数: {N} → {new_N} @ Vbias={Vbias:.4f}Δ")
            
            # 使用增量方法计算新电流
            I_next = self.compute_current_at_bias(Vbias, new_N)
            history.append((new_N, I_next))
            
            # 计算变化量
            if isinstance(I_current, np.ndarray):
                # 数组电流：计算范数变化
                delta_I = np.linalg.norm(I_next - I_current)
                norm_current = np.linalg.norm(I_current) + 1e-6
                rel_delta = delta_I / norm_current
            else:
                # 标量电流
                delta_I = abs(I_next - I_current)
                rel_delta = delta_I / (abs(I_current) + 1e-6)
            
            logging.debug(f"边带数 {new_N}: 电流={I_next}, ΔI={delta_I:.6f}, 相对Δ={rel_delta:.4f}")
            
            # 收敛检查 - 同时考虑相对和绝对误差
            if rel_delta < self.params['rel_tol'] or delta_I < self.params['abs_tol']:
                logging.info(f"收敛于 {new_N} 边带 @ Vbias={Vbias:.4f}Δ")
                return I_next, new_N
            
            I_current = I_next
            N = new_N
        
        # 未收敛时返回最佳结果
        logging.warning(f"达到最大迭代次数 @ Vbias={Vbias:.4f}Δ")
        return self._select_best_result(history)


    
    def adaptive_current_advanced_old(self, Vbias):
        """改进的自适应策略（增量版本）"""
        # 获取自适应配置
        init_N, step_size, max_N = self._get_adaptive_config(Vbias)
        
        # 重置哈密顿量缓存
        self.ham_builder.prev_sidebands = None
        self.ham_builder.cached_g0_inv_FKs = None
        self.ham_builder.cached_hop_FKs = None
        
        # 初始计算
        N = init_N
        I_current = self.compute_current_at_bias(Vbias, N)
        
        # 记录收敛历史
        history = [(N, I_current)]
        
        # 自适应迭代
        for _ in range(15):  # 增加迭代次数限制
            # 检查是否达到最大边带数
            if N >= max_N:
                logging.warning(f"达到最大边带数 {max_N} @ Vbias={Vbias:.4f}Δ")
                # 返回历史中最接近收敛的结果
                return self._select_best_result(history)
            
            # 增加边带数
            new_N = N + step_size
            logging.debug(f"扩展边带数: {N} → {new_N} @ Vbias={Vbias:.4f}Δ")
            
            # 使用增量方法计算新电流
            I_next = self.compute_current_at_bias(Vbias, new_N)
            history.append((new_N, I_next))
            
            # 计算变化量
            if isinstance(I_current, np.ndarray):
                # 数组电流：计算范数变化
                delta_I = np.linalg.norm(I_next - I_current)
                norm_current = np.linalg.norm(I_current) + 1e-6
                rel_delta = delta_I / norm_current
            else:
                # 标量电流
                delta_I = abs(I_next - I_current)
                rel_delta = delta_I / (abs(I_current) + 1e-6)
            
            logging.debug(f"边带数 {new_N}: 电流={I_next}, ΔI={delta_I:.6f}, 相对Δ={rel_delta:.4f}")
            
            # 收敛检查 - 同时考虑相对和绝对误差
            if rel_delta < self.params['rel_tol'] or delta_I < self.params['abs_tol']:
                logging.info(f"收敛于 {new_N} 边带 @ Vbias={Vbias:.4f}Δ")
                return I_next, new_N
            
            I_current = I_next
            N = new_N
        
        # 未收敛时返回最佳结果
        logging.warning(f"达到最大迭代次数 @ Vbias={Vbias:.4f}Δ")
        return self._select_best_result(history)
    
    def compute_current_at_bias(self, Vbias, max_sidebands=None):
        """计算给定偏压下的电流（支持增量哈密顿量和缓存）"""
        if max_sidebands is None:
            max_sidebands = self.max_sidebands
            
        # 使用缓存：对于相同的Vbias和max_sidebands，避免重复构建哈密顿量切片
        cache_key = (Vbias, max_sidebands)
        if cache_key in self.ham_slice_cache:
            g0_inv_FKs, hop_FKs = self.ham_slice_cache[cache_key]
        else:
            ham = self.ham_builder.construct_hamiltonian(0)
            g0_inv_FKs, hop_FKs = self.ham_builder.build_slice_ham(Vbias, ham, max_sidebands)
            self.ham_slice_cache[cache_key] = (g0_inv_FKs, hop_FKs)
        
        N_SC = self.params['N_SC']
        N_junction = self.params['N_junction']
        
        # 生成能量积分网格
        if Vbias < 0.12:
            max_points_ = 21
        else:
            max_points_ = self.omega_points
            
        omega_values = self.generate_uniform_grid(
            start = 0 * Vbias,
            end = 1 * Vbias,
            spacing = 0.0025 * Vbias,  # 
            max_points =  max_points_
        )

        # 使用增量计算
        # 使用部分函数固定参数
        compute_func = functools.partial(
            self.gf_calculator._compute_current_at_omega,
            lead_type='infinite',
            region_type='junction',
            ham_builder=self.ham_builder,
            max_sidebands=max_sidebands,
            Vbias=Vbias,
            N_SC=N_SC,
            N_junction=N_junction,
            g0_inv_FKs=g0_inv_FKs,
            hop_FKs=hop_FKs
        )
        
        # 并行计算
        results = Parallel(n_jobs=self.job_parallel[1])(
            delayed(compute_func)(omega) for omega in omega_values)
        
        current_mat = np.real(np.array(results)).reshape([len(omega_values),-1])
        
        # 对能量积分，返回每个site的电流
        site_currents = np.trapz(current_mat, omega_values, axis=0) / 0.3  #/ (2 * np.pi)
        
        return site_currents
    
    def _get_adaptive_config_old(self, Vbias):
        """获取自适应配置（添加默认配置）"""
        delta = self.params['delta']
        ratio = Vbias / delta
        
        # 查找匹配的区域配置
        for config in self.params['adaptive_regions']:
            if config[0] <= ratio < config[1]:
                return config[2], config[3], config[4]
        
        # 默认配置
        return 10, 4, 50

    def _get_adaptive_config(self, Vbias):
        """获取自适应配置（添加默认配置）"""
        delta = self.params['delta']
        ratio = Vbias / delta if delta != 0 else 0
        
        # 查找匹配的区域配置
        for config in self.params['adaptive_regions']:
            if config[0] <= ratio < config[1]:
                return config[2], config[3], config[4]
        
        # 默认配置（处理边界情况）
        if ratio >= self.params['adaptive_regions'][-1][1]:
            # 超过最大ratio，使用最后一个配置
            last_config = self.params['adaptive_regions'][-1]
            return last_config[2], last_config[3], last_config[4]
        else:
            # 默认配置
            return 10, 4, 50
    
    def _select_best_result(self, history):
        """从未收敛的历史中选择最佳结果"""
        if len(history) < 2:
            return history[-1][1], history[-1][0]
        
        # 计算所有连续点之间的变化
        changes = []
        for i in range(1, len(history)):
            prev_N, prev_I = history[i-1]
            curr_N, curr_I = history[i]
            
            if isinstance(prev_I, np.ndarray) and isinstance(curr_I, np.ndarray):
                # 数组电流：计算范数变化
                delta = np.linalg.norm(curr_I - prev_I)
                rel_delta = delta / (np.linalg.norm(prev_I) + 1e-6)
            else:
                # 标量电流
                delta = abs(curr_I - prev_I)
                rel_delta = delta / (abs(prev_I) + 1e-6)
            
            changes.append((rel_delta, delta, i))
        
        # 找到变化最小的点
        best_idx = min(changes, key=lambda x: x[0])[2]
        best_N, best_I = history[best_idx]
        
        logging.info(f"选择最佳结果: {best_N} 边带")
        return best_I, best_N

class JJPlotter:
    @staticmethod
    def plot_single_result(metadata, data, output_dir):
        """绘制单次计算结果，自动处理一维和二维电流数据"""
        calc_type = metadata["calculation"]
        current_data_type = metadata.get("current_data_type", "scalar")
        
        if calc_type == "DC_IV":
            Vbias = data[0]
            currents = data[1]
            sideband = data[2] if len(data) > 2 else None
            
            # 检查电流数据的维度
            currents_array = np.array(currents)
            
            if currents_array.ndim == 1 or (currents_array.ndim == 2 and currents_array.shape[1] <= 1):
                # 一维数据或只有一列：绘制标准IV曲线
                fig = plt.figure(figsize=(10, 8))
                
                if sideband is not None:
                    # 双面板图：电流和边带数
                    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10), sharex=True)
                    
                    # 电流-电压曲线
                    ax1.plot(Vbias, currents, 'r-', lw=2)
                    ax1.set_ylabel('Current (nA)', fontsize=14)
                    ax1.set_title('DC Current-Voltage Characteristic', fontsize=16)
                    ax1.grid(True)
                    
                    # 边带数-电压曲线
                    ax2.plot(Vbias, sideband, 'b-', lw=2)
                    ax2.set_xlabel('Bias Voltage (meV)', fontsize=14)
                    ax2.set_ylabel('Sideband Number', fontsize=14)
                    ax2.set_title('Converged Sideband Numbers', fontsize=16)
                    ax2.grid(True)
                else:
                    # 单面板图：只有电流
                    plt.plot(Vbias, currents, '-', lw=2, color='r')
                    plt.xlabel('Bias Voltage (meV)', fontsize=14)
                    plt.ylabel('Current (nA)', fontsize=14)
                    plt.title('DC Current-Voltage Characteristic', fontsize=16)
                    plt.grid(True)
                    
            else:
                # 二维数据：每个site的电流贡献
                n_sites = currents_array.shape[1]
                
                # 创建三面板图
                fig, axes = plt.subplots(3, 1, figsize=(5, 8), sharex=True)
                
                # 1. 总电流
                total_current = currents_array.sum(axis=1)
                axes[0].plot(Vbias, total_current, 'r-', lw=2)
                axes[0].set_ylabel('Total Current (nA)', fontsize=14)
                axes[0].set_title('Total Current vs Voltage', fontsize=14)
                axes[0].grid(True)
                
                # 2. 每个site的电流（热图）
                # 创建位置网格
                X, Y = np.meshgrid(Vbias, np.arange(n_sites))
                pc = axes[1].pcolormesh(X, Y, currents_array.T, shading='auto', cmap='viridis')
                axes[1].set_ylabel('Site Index', fontsize=14)
                axes[1].set_title('Current Distribution Across Sites', fontsize=14)
                plt.colorbar(pc, ax=axes[1], label='Current (nA)')
                
                # 3. 几个代表性site的电流
                # 选择几个site绘制详细曲线
                selected_sites = np.linspace(0, n_sites-1, min(5, n_sites), dtype=int)
                for site_idx in selected_sites:
                    axes[2].plot(Vbias, currents_array[:, site_idx], '-', 
                                label=f'Site {site_idx}', lw=1.5)
                axes[2].set_xlabel('Bias Voltage (meV)', fontsize=14)
                axes[2].set_ylabel('Current (nA)', fontsize=14)
                axes[2].set_title('Current at Selected Sites', fontsize=14)
                axes[2].legend()
                axes[2].grid(True)
                
                if sideband is not None:
                    # 在总电流图中添加边带数信息
                    ax_twin = axes[0].twinx()
                    ax_twin.plot(Vbias, sideband, 'b--', alpha=0.7, lw=1.5)
                    ax_twin.set_ylabel('Sideband Number', fontsize=12, color='b')
                    ax_twin.tick_params(axis='y', labelcolor='b')
                    
        elif calc_type == "DC_IV_Bsweep":
            B = data[0]
            currents = data[1]
            sideband = data[2] if len(data) > 2 else None
            
            # 检查电流数据的维度
            currents_array = np.array(currents)
            
            if currents_array.ndim == 1 or (currents_array.ndim == 2 and currents_array.shape[1] <= 1):
                # 一维数据或只有一列
                fig = plt.figure(figsize=(10, 8))
                
                if sideband is not None:
                    # 双面板图
                    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10), sharex=True)
                    
                    # 电流-磁场曲线
                    ax1.plot(B, currents, 'r-', lw=2, marker='.', markersize=16)
                    ax1.set_ylabel('Current (nA)', fontsize=14)
                    ax1.set_title(f'DC Current vs Magnetic Field @ Vbias={metadata["fixed_Vbias"]} meV', fontsize=16)
                    ax1.grid(True)
                    
                    # 边带数-磁场曲线
                    ax2.plot(B, sideband, 'b-', lw=2)
                    ax2.set_xlabel('Zeeman (meV)', fontsize=14)
                    ax2.set_ylabel('Sideband Number', fontsize=14)
                    ax2.set_title('Converged Sideband Numbers', fontsize=16)
                    ax2.grid(True)
                else:
                    # 单面板图
                    plt.plot(B, currents, 'b-o', lw=2)
                    plt.xlabel('Magnetic Field', fontsize=14)
                    plt.ylabel('Current (nA)', fontsize=14)
                    plt.title(f'DC Current vs Magnetic Field @ Vbias={metadata["fixed_Vbias"]} meV', fontsize=16)
                    plt.grid(True)
                    
            else:
                # 二维数据：每个site的电流贡献
                n_sites = currents_array.shape[1]
                
                # 创建三面板图
                fig, axes = plt.subplots(3, 1, figsize=(12, 15), sharex=True)
                
                # 1. 总电流
                total_current = currents_array.sum(axis=1)
                axes[0].plot(B, total_current, 'r-', lw=2, marker='.', markersize=10)
                axes[0].set_ylabel('Total Current (nA)', fontsize=14)
                axes[0].set_title(f'Total Current vs Magnetic Field @ Vbias={metadata["fixed_Vbias"]} meV', fontsize=14)
                axes[0].grid(True)
                
                # 2. 每个site的电流（热图）
                X, Y = np.meshgrid(B, np.arange(n_sites))
                pc = axes[1].pcolormesh(X, Y, currents_array.T, shading='auto', cmap='viridis')
                axes[1].set_ylabel('Site Index', fontsize=14)
                axes[1].set_title('Current Distribution Across Sites', fontsize=14)
                plt.colorbar(pc, ax=axes[1], label='Current (nA)')
                
                # 3. 几个代表性site的电流
                selected_sites = np.linspace(0, n_sites-1, min(5, n_sites), dtype=int)
                for site_idx in selected_sites:
                    axes[2].plot(B, currents_array[:, site_idx], '-o', 
                                label=f'Site {site_idx}', lw=1.5, markersize=6)
                axes[2].set_xlabel('Zeeman (meV)', fontsize=14)
                axes[2].set_ylabel('Current (nA)', fontsize=14)
                axes[2].set_title('Current at Selected Sites', fontsize=14)
                axes[2].legend()
                axes[2].grid(True)
                
                if sideband is not None:
                    # 在总电流图中添加边带数信息
                    ax_twin = axes[0].twinx()
                    ax_twin.plot(B, sideband, 'b--', alpha=0.7, lw=1.5)
                    ax_twin.set_ylabel('Sideband Number', fontsize=12, color='b')
                    ax_twin.tick_params(axis='y', labelcolor='b')
        
        # 创建文件名并保存
        common_timestamp = metadata.get("common_timestamp", datetime.now().strftime("%Y%m%d_%H%M%S"))
        filename = f"{calc_type}_plot_{common_timestamp}.png"
        filepath = os.path.join(output_dir, filename)
        
        plt.tight_layout()
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()
        
        logging.info(f"Plot saved: {filepath}")
        return filepath

class PathManager:
    def __init__(self, base_dir="results", task_name=None):
        self.base_dir = base_dir
        self.task_name = task_name or datetime.now().strftime("%Y%m%d_%H%M%S")
        self.task_dir = os.path.join(self.base_dir, self.task_name)
        
        # 创建必要的目录
        self._create_directories()
    
    def _create_directories(self):
        """创建所有必要的目录"""
        os.makedirs(self.task_dir, exist_ok=True)
        os.makedirs(self.get_raw_data_dir(), exist_ok=True)
        os.makedirs(self.get_plots_dir(), exist_ok=True)
    
    def get_raw_data_dir(self):
        return os.path.join(self.task_dir, "raw_data")
    
    def get_plots_dir(self):
        return os.path.join(self.task_dir, "plots")
    
    def get_task_metadata_path(self):
        return os.path.join(self.task_dir, "task_metadata.json")
    
    def save_task_metadata(self, params):
        """保存任务元数据"""
        metadata = {
            "task_name": self.task_name,
            "start_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "parameters": self._filter_metadata(params)
        }
        
        with open(self.get_task_metadata_path(), 'w') as f:
            json.dump(metadata, f, indent=4)
        
        return self.get_task_metadata_path()
    
    def _filter_metadata(self, params):
        """过滤元数据，只保留相关参数"""
        # 创建副本
        filtered = params.copy()
        
        # 移除大数组
        if 'disorder_distribution' in filtered:
            del filtered['disorder_distribution']
        
        return filtered

def update_parameters(user_params=None):
    """确保参数正确更新"""
    # get default
    params = get_default_parameters()
    
    # get a copy
    from copy import deepcopy
    updated_params = deepcopy(params)
    
    # update it then
    if user_params is not None:
        for key, value in user_params.items():
            # 特殊处理嵌套字典
            if key == 'adaptive_regions' and isinstance(value, list):
                updated_params[key] = deepcopy(value)
            if key == 'disorder_type':
                valid_regions = ['gaussian', 'smooth', 'random_typeI', 'random_typeII', 'nonhermitian', 'none', 'from_file']
                if value not in valid_regions:
                    raise ValueError(f"Invalid disorder_type: {value}. Must be one of {valid_regions}")
            elif key in updated_params:
                updated_params[key] = value
            else:
                logging.warning(f"Ignoring unknown parameter: {key}")
    
    # recording ...
    logging.info("Final simulation parameters:")
    
    for key, value in updated_params.items():
        logging.info(f"  {key}: {value}")
    
    return updated_params

def get_default_parameters():
    """ """
    return {
        'recursion_depth': 100,
        'eta': 1e-3,
        
        'N_SC': 150,
        'N_junction': 2,
        't': 12.7,
        'delta': 0.3,
        'mu': 0.0,
        'mu_lead': 0.0,
        'B': 0,
        'alpha': 4.0,
        
        'v_tau': 0.842,        
        'max_sidebands': 10,
        'Vbias_max_ratio': 5.0,
        'fixed_Vbias': 0.1,  # 固定偏压值 (meV)
        'B_max': 0.0,       # 最大磁场强度
        
        # can set them to be the largest allowed
        'omega_points': 2000,
        'Vbias_points': 300,
        'B_points': 300,

        # disorder参数
        'disorder_type': 'none',
        'disorder_file': 'disorder_data_test_Nd=50_ld=10_v0=2Delta.csv',
        'disorder_strength':0.0,
        
        # adaptive params 
        'adaptive_iv': True,
        'rel_tol': 0.02,
        'abs_tol':0.01,
        'adaptive_max_N': 100,
        'adaptive_method': 'advanced', # 'dynamic'
        'adaptive_regions': [
            (0.0, 0.025, 50, 10, 220),
            (0.025, 0.05, 40, 2, 80),
            (0.05, 0.1, 38, 2, 70),    # low vbias：start_N=30, increase_N = 4, end_Nmax = 80
            (0.1, 0.2, 20, 2, 50),    
            (0.2, 0.4, 10, 2, 30),     
            (0.4, 0.7, 8, 1, 20),     
            (0.7, 1.0, 6, 1, 12),     
            (1.0, 6.0, 4, 1, 10)      
        ],   # all in ratio to self. delta
        
        'job_parallel': [101, 1],
        'output_dir': 'results'     ## should be updated every time
    }

def run_dc_iv_test(use_sparse=True, max_sidebands=5):
    """运行DC_IV测试"""
    # 获取默认参数
    params = get_default_parameters()
    
    # 设置关键参数
    params.update({
        'N_SC': 150,           # 超导区长度
        'N_junction': 5,       # 结区长度
        'delta': 0.3,          # 超导能隙 (meV)
        'alpha': 4.0,          # 自旋轨道耦合强度
        'v_tau': 0.882, #0.934, #1.0, #0.882,         # 隧穿强度
        'B': 0,             # 磁场强度  # 0.25, 0.75, 1.25, 1.75
        'mu_lead': 0.0,        # 引线化学势
        'mu': 0.0,             # 化学势
        'max_sidebands': max_sidebands,  # 最大边带数
        'omega_points': 501,    # 能量点数（减少以加速测试）
        'Vbias_points': 1000,    # 偏压点数（减少以加速测试）
        'disorder_strength':0.0,
        
        'job_parallel': [161,1],  # 并行设置 [外层并行, 内层并行]
        'use_sparse': use_sparse,  # 是否使用稀疏矩阵
        'sparse_threshold': 0.3,  # 稀疏度阈值
        'sparse_depth_limit': 5,  # 最大稀疏递归深度
        'adaptive_iv': True,  # 禁用自适应计算
        'recursion_depth': 50, # 减少递归深度以加速测试
        'fixed_Vbias': 0.01*0.3,  # 固定偏压 0.15 meV
        'B_max': 2.0,         # 扫描磁场范围 0-1.5
        'B_points': 11,       # 磁场点数
        'mid_site_i': 150      # 'N_sc' + 'N_junction'//2 
    })
    # data set order: vtau = 0.842, B=0, 0.5, 2.0; vtau = 1.00, B=2.0, 0.5, 0.0
    
    # 创建路径管理器
    path_manager = PathManager(base_dir="26_results", task_name="dc_iv_test_v0")
    
    # 创建求解器
    solver = JosephsonJunctionSolver(params, path_manager)
    
    try:
        # 运行DC_IV计算
        start_time = time.time()
        solver.run_calculation("DC_IV") #("DC_IV") #("DC_IV_Bsweep")
        compute_time = time.time() - start_time
        
        # 打印结果摘要
        print("\n" + "="*50)
        print(f"DC_IV计算完成，耗时: {compute_time:.2f}秒")

    except Exception as e:
        logging.error(f"DC_IV计算失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

if __name__ == "__main__":
    # 测试单个DC_IV计算
    print("运行DC_IV计算测试...")
    run_dc_iv_test(use_sparse=True, max_sidebands=15)

2026-01-23 18:05:34,459 - INFO - HamiltonianBuilder initialized, disorder type: none, Sparse mode: True
2026-01-23 18:05:34,462 - INFO - GreenFunctionCalculator initialized, Sparse mode: True
2026-01-23 18:05:34,463 - INFO - JosephsonJunctionSolver initialized
2026-01-23 18:05:34,463 - INFO - Superconductor: 150 sites, Junction: 5 sites
2026-01-23 18:05:34,464 - INFO - Recursion depth: 50, Max sidebands: 15
2026-01-23 18:05:34,465 - INFO - Starting DC IV calculation


运行DC_IV计算测试...


2026-01-23 18:06:31,039 - INFO - Current data shape: (210, 5), each site's contribution
2026-01-23 18:06:31,042 - INFO - DC IV calculation completed, duration: 56.58 sec
2026-01-23 18:06:31,043 - INFO - Current data: array_5_sites
2026-01-23 18:06:31,048 - INFO - Results saved to: 26_results/dc_iv_test_v0/raw_data/DC_IV_20260123_180534.csv
2026-01-23 18:06:31,669 - INFO - Plot saved: 26_results/dc_iv_test_v0/plots/DC_IV_plot_20260123_180534.png



DC_IV计算完成，耗时: 57.21秒


/tmp/ipykernel_324996/2614878438.py:577: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2046980332.py:577: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/581279195.py:577: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2614878438.py:577: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2046980332.py:577: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/581279195.py:577: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2614878438.py:577: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2046980332.py:577: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/581279195.py:577: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2614878438.py:577: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/2046980332.py:577: RuntimeWarning: overflow encountered in exp
/tmp/ipykernel_324996/581279195.py:577: RuntimeWarning: overflow enc

In [15]:
#  added on Dec 23 th

# updated, inclusing disoder 
import numpy as np
import scipy.linalg as la
from scipy.sparse import diags, block_diag, lil_matrix, csc_matrix, identity, issparse, eye as sparse_eye
from scipy.sparse.linalg import inv as sparse_inv
from scipy.sparse.linalg import spsolve, factorized, inv
import scipy.sparse as sp
from joblib import Parallel, delayed
import pandas as pd
import os
import time
from datetime import datetime
import logging
import json
import matplotlib.pyplot as plt
import hashlib
import functools
from collections import defaultdict

# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class HamiltonianBuilder:
    def __init__(self, params):
        # 验证disorder类型
        if 'disorder_type' in params:
            valid_disorders = ['gaussian', 'smooth', 'random_typeI', 'random_typeII', 'nonhermitian', 'none', 'from_file']
            if params['disorder_type'] not in valid_disorders:
                raise ValueError(f"non-effective model: {params['disorder_type']}. must be: {', '.join(valid_disorders)}")
        
        # 物理参数
        self.t = params['t']  # 跳跃强度 (meV)
        self.delta = params['delta']  # 超导能隙 (meV)
        self.mu = params['mu']  # 化学势 (meV)
        self.mu_lead = params['mu_lead']  # 引线化学势 (meV)
        self.B = params['B'] 
        self.alpha = params['alpha']  # 自旋轨道耦合 (meV)
             
        # 结参数
        self.N_SC = params['N_SC']  # 超导区长度
        self.N_junction = params['N_junction']  # 结区长度
        self.v_tau = params['v_tau']  # 隧穿强度
        
        # Disorder参数
        self.disorder_type = params.get('disorder_type', 'none')  # Disorder类型
        # Gaussian 
        self.V_gau = params.get('Vdis_gau', 0.0)  # Disorder强度
        self.X_gau = params.get('Xdis_gau', 50)  # Disorder中心
        self.decayL_gau = params.get('decayL_gau', 50)  # Disorder衰减长度     
        # smooth
        self.decayL_smooth = params.get('decayL_smooth', 50)  # Smooth disorder参数
        self.Vmax_smooth = params.get('Vdis_smooth', 0.3)  # Max disorder potential
        self.Vd_smooth = params.get('Vd_smooth', 0.0)  # Disorder深度   
        # random typeI
        self.Num_Rd1 = params.get('N_imp1', 52)  # Number of impurities
        self.lambda_Rd1 = params.get('lambda_imp1', 18.0)  # Characteristic length
        self.V0_Rd1 = params.get('V0_imp1', 0.0)  # Disorder amplitude (meV)  
        # random typeII
        self.Nd_Rd2 = params.get('Nd_imp2', 10.0)  # Impurity density (per micron)
        self.lambda_Rd2 = params.get('lambda_imp2', 20.0)  # Characteristic length (nm)
        self.V0_Rd2 = params.get('V0_imp2', 0.0)  # Disorder amplitude (meV)
        self.a0 = params.get('a0', 10.0)  # Lattice constant (nm)
        # Nonhermitian
        self.nonH_eta = params.get('nonH_eta', 1.5e-3)
        # From file
        self.disorder_file = params.get('disorder_file', 'disorder_data.txt')  # File to read disorder from
        self.disorder_strength = params.get('disorder_strength', 1.0)  
        
        # Floquet参数
        self.max_sidebands = params['max_sidebands']
        
        # 初始化Pauli矩阵
        self._init_pauli_matrices()
        
        # 计算系统总长度
        self.total_sites = 2 * self.N_SC + self.N_junction
        
        # 初始化无序分布
        self.disorder_distribution = self._calculate_disorder_distribution()
        
        # 稀疏矩阵配置
        self.use_sparse = params.get('use_sparse', True)  # 默认启用稀疏矩阵
        self.sparse_threshold = params.get('sparse_threshold', 0.3)  # 稀疏度阈值
        self.sparse_depth_limit = params.get('sparse_depth_limit', 5)  # 最大稀疏递归深度

        # 新增缓存变量
        self.prev_sidebands = None
        self.cached_g0_inv_FKs = None
        self.cached_hop_FKs = None
        
        # 增量扩建缓存
        self.incremental_cache = {}
        
        logging.info(f"HamiltonianBuilder initialized, disorder type: {self.disorder_type}, Sparse mode: {self.use_sparse}")

    def _init_pauli_matrices(self):
        """初始化Pauli矩阵"""
        # 自旋空间
        self.sigma_0 = np.eye(2)
        self.sigma_x = np.array([[0, 1], [1, 0]])
        self.sigma_y = np.array([[0, -1j], [1j, 0]])
        self.sigma_z = np.array([[1, 0], [0, -1]])
        
        # Nambu空间
        self.tau_0 = np.eye(2)
        self.tau_x = np.array([[0, 1], [1, 0]])
        self.tau_y = np.array([[0, -1j], [1j, 0]])
        self.tau_z = np.array([[1, 0], [0, -1]])
        self.tau_plus = (self.tau_x + 1j * self.tau_y) / 2
        self.tau_minus = (self.tau_x - 1j * self.tau_y) / 2
        self.tau_e = np.array([[1, 0], [0, 0]])  # 电子块
        self.tau_h = np.array([[0, 0], [0, 1]])  # 空穴块

    def _calculate_disorder_distribution(self):
        """计算无序分布"""
        # 初始化数组
        disorder_array = np.zeros(self.total_sites)
        
        # 根据disorder类型生成无序
        if self.disorder_type == 'smooth':
            # 平滑无序
            for i in range(self.total_sites):
                if i < self.N_SC: 
                    disorder_array[i] = self.Vmax_smooth * \
                        (1 - self.Vd_smooth * np.exp(i / self.decayL_smooth))
                elif i >= self.N_junction + self.N_SC:
                    disorder_array[i] = self.Vmax_smooth * \
                        (1 - self.Vd_smooth * np.exp(-(i - self.N_junction) / self.decayL_smooth))
                else: # inside junction 
                    disorder_array[i] = self.Vmax_smooth * (1 - self.Vd_smooth)
        
        elif self.disorder_type == 'gaussian':
            # 高斯无序
            for i in range(self.total_sites):
                disorder_array[i] = self.V_gau * np.exp(-(i - self.X_gau)**2 / self.decayL_gau**2)
        
        elif self.disorder_type == 'nonhermitian':
            # 非厄米无序
            for i in range(self.total_sites):
                disorder_array[i] = -1j * self.nonH_eta
        
        elif self.disorder_type == 'random_typeI':
            # 随机类型I
            disorder_array = self._generate_random_typeI_disorder(range(self.total_sites))
        
        elif self.disorder_type == 'random_typeII':
            # 随机类型II
            disorder_array = self._generate_random_typeII_disorder(range(self.total_sites))
        
        elif self.disorder_type == 'from_file':
            # 从文件读取无序
            disorder_array = self._load_disorder_from_file()
        
        return disorder_array

    def _load_disorder_from_file(self):
        """从文件加载无序分布"""
        try:
            # 尝试从不同文件格式加载
            if self.disorder_file.endswith('.csv'):
                df = pd.read_csv(self.disorder_file)
                if 'disorder_value' in df.columns:
                    disorder_array = df['disorder_value'].values
                else:
                    # 假设第一列包含无序值
                    disorder_array = df.iloc[:, 0].values
            elif self.disorder_file.endswith('.txt') or self.disorder_file.endswith('.dat'):
                disorder_array = np.loadtxt(self.disorder_file)
            elif self.disorder_file.endswith('.npy'):
                disorder_array = np.load(self.disorder_file)
            else:
                # 尝试作为文本文件加载
                disorder_array = np.loadtxt(self.disorder_file)
            
            # 检查数组长度是否匹配总格点数
            if len(disorder_array) != self.total_sites:
                logging.warning(f"Disorder array length ({len(disorder_array)}) doesn't match total sites ({self.total_sites}). Truncating or padding.")
                if len(disorder_array) > self.total_sites:
                    disorder_array = disorder_array[:self.total_sites]
                else:
                    disorder_array = np.pad(disorder_array, (0, self.total_sites - len(disorder_array)), 'constant')
            
            logging.info(f"Loaded disorder from file: {self.disorder_file}")
            return disorder_array
            
        except Exception as e:
            logging.error(f"Failed to load disorder from file {self.disorder_file}: {str(e)}")
            return np.zeros(self.total_sites)

    def _generate_random_typeI_disorder(self, sites):
        """生成随机类型I无序"""
        disorder_array = np.zeros(self.total_sites)
        rng = np.random.default_rng()
        
        # 仅在选定区域生成杂质
        impurity_positions = rng.choice(sites, self.Num_Rd1)
        impurity_signs = np.array([(-1)**(n) for n in range(self.Num_Rd1)])
        
        # 计算未归一化势
        S = np.zeros(self.total_sites)
        for i in sites:
            for j in range(self.Num_Rd1):
                distance = abs(i - impurity_positions[j])
                S[i] += impurity_signs[j] * np.exp(-distance / self.lambda_Rd1)
        
        # 归一化
        S_avg = np.mean(S)
        S_centered = S - S_avg
        variance = np.mean(S_centered**2)
        N_V = np.sqrt(variance) / self.V0_Rd1 if variance > 0 else 1.0
        
        # 应用到无序数组
        for i in sites:
            disorder_array[i] = (self.V0_Rd1 / N_V) * S_centered[i]
        
        logging.info(f"Generated charge impurity disorder, impurities: {self.Num_Rd1}")
        return disorder_array

    def _generate_random_typeII_disorder(self, sites):
        """生成随机类型II无序"""
        disorder_array = np.zeros(self.total_sites)
        L_total_nm = (self.total_sites - 1) * self.a0
        
        # 计算选定区域中的杂质数量
        region_length = len(sites) * self.a0 / 1000  # in microns
        N_d = int(self.Nd_Rd2 * region_length)
        
        rng = np.random.default_rng()
        
        # 生成杂质位置
        min_pos = min(sites) * self.a0 if sites else 0
        max_pos = max(sites) * self.a0 if sites else 0
        impurity_positions = rng.uniform(min_pos, max_pos, N_d)
        impurity_amplitudes = rng.normal(0, 1, N_d)
        
        # 计算选定位点的无序
        for i in sites:
            x_i = i * self.a0  # 位点位置 (nm)
            potential_sum = 0.0
            for j in range(N_d):
                distance = abs(x_i - impurity_positions[j])
                potential_sum += impurity_amplitudes[j] * np.exp(-distance / self.lambda_Rd2)
            disorder_array[i] = self.V0_Rd2* potential_sum
        
        logging.info(f"Generated random amplitude disorder, impurities: {N_d}")
        return disorder_array

    def disorder_potential(self, site_index):
        """返回给定位点的无序势矩阵"""
        if 0 <= site_index < len(self.disorder_distribution):
            V_s = self.disorder_distribution[site_index]
            
            if self.disorder_type == 'nonhermitian':
                return V_s * np.kron(self.tau_0, self.sigma_0)
            else:
                return V_s * np.kron(self.tau_z, self.sigma_0)
        else:
            return np.zeros((4, 4))

    def block_hamiltonian(self, phi, region_type):
        """构建区域类型的哈密顿量块"""
        onsite = 2 * self.t - self.mu 
        H00 = onsite * np.kron(self.tau_z, self.sigma_0) + \
              self.B * np.kron(self.tau_z, self.sigma_x) * np.sqrt(self.mu_lead**2 + self.delta**2)
        H01 = -self.t * np.kron(self.tau_z, self.sigma_0) + 1j * self.alpha * np.kron(self.tau_z, self.sigma_y)
        
        if region_type == 'SC':
            H00 = H00 + self.delta * (np.exp(1j * phi) * np.kron(self.tau_plus, 1j*self.sigma_y) + 
                                 np.exp(-1j * phi) * np.kron(self.tau_minus, -1j*self.sigma_y))
            
        return H00, H01

    def construct_hamiltonian(self, phi):
        """构建SNS结的完整哈密顿量"""
        
        H00_J, H01_J = self.block_hamiltonian(0, 'NM')
        
        H00_R, H01_R = self.block_hamiltonian(phi, 'SC')
        H00_L, H01_L = self.block_hamiltonian(0, 'SC')

        # self.B = 0
        H00_R_lead, H01_R_lead = self.block_hamiltonian(phi, 'SC')
        H00_L_lead, H01_L_lead = self.block_hamiltonian(0, 'SC')
      
        return {
            'H00_L': H00_L, 'H01_L': H01_L,
            'H00_R': H00_R, 'H01_R': H01_R,
            'H00_J': H00_J, 'H01_J': H01_J,
            'H00_R_lead': H00_R_lead, 'H01_R_lead': H01_R_lead,
            'H00_L_lead': H00_L_lead, 'H01_L_lead': H01_L_lead
        }
    
    def _incremental_expand_slice_ham(self, Vbias, ham, new_max_sidebands):
        """增量扩建Floquet哈密顿量切片"""
        if self.prev_sidebands is None or self.cached_g0_inv_FKs is None or self.cached_hop_FKs is None:
            return self._full_build_slice_ham(Vbias, ham, new_max_sidebands)
            
        old_max_sidebands = self.prev_sidebands
        old_sideN = 2 * old_max_sidebands + 1
        new_sideN = 2 * new_max_sidebands + 1
        dim_old = 4 * old_sideN
        dim_new = 4 * new_sideN
        
        # 获取旧的矩阵列表
        old_g0_inv_FKs = self.cached_g0_inv_FKs
        old_hop_FKs = self.cached_hop_FKs
        
        # 新的矩阵列表
        g0_inv_FKs = []
        hop_FKs = []
        
        SNS_Len = self.N_SC * 2 + self.N_junction
        
        # 扩展每个位点的矩阵
        for site_i in range(SNS_Len):
            old_g0_inv = old_g0_inv_FKs[site_i]
            old_hop = old_hop_FKs[site_i]
            
            # 确定当前位点的哈密顿量块
            if self.N_SC <= site_i < self.N_SC + self.N_junction:
                Ham_onsite = ham['H00_J']
                Ham_hop = ham['H01_J']
                Hop_e = Ham_hop * np.kron(self.tau_e, np.ones((2, 2))) * self.v_tau
                Hop_h = -1 * Ham_hop * np.kron(self.tau_h, np.ones((2, 2))) * self.v_tau
                is_junction = True
            elif site_i < self.N_SC:
                Ham_onsite = ham['H00_L']
                Ham_hop = ham['H01_L']
                is_junction = False
            else:
                Ham_onsite = ham['H00_R']
                Ham_hop = ham['H01_R']
                is_junction = False
            
            # 扩展g0_inv_FK: 块对角矩阵，上下各增加 (new_max_sidebands - old_max_sidebands) 个块
            # 新矩阵的边带范围：[-new_max_sidebands, new_max_sidebands]
            # 旧矩阵的边带范围：[-old_max_sidebands, old_max_sidebands]
            num_bands_to_add = new_max_sidebands - old_max_sidebands
            diag_blocks = []
            
            # 上部新增的块（更负的边带）
            for band_i in range(-new_max_sidebands, -old_max_sidebands):
                diag_block = Ham_onsite - band_i * Vbias * np.eye(4)
                diag_blocks.append(diag_block)
            
            # 中间旧矩阵部分
            if sp.issparse(old_g0_inv):
                # 将旧的块对角矩阵转为块列表
                old_blocks = [old_g0_inv[i*4:(i+1)*4, i*4:(i+1)*4] for i in range(old_sideN)]
                diag_blocks.extend(old_blocks)
            else:
                for n in range(old_sideN):
                    idx = slice(n*4, (n+1)*4)
                    diag_blocks.append(old_g0_inv[idx, idx])
            
            # 下部新增的块（更正边带）
            for band_i in range(old_max_sidebands+1, new_max_sidebands+1):
                diag_block = Ham_onsite - band_i * Vbias * np.eye(4)
                diag_blocks.append(diag_block)
            
            # 构建新的块对角矩阵
            if self.use_sparse and dim_new > 20:
                g0_inv_new = block_diag(diag_blocks, format='csc')
            else:
                g0_inv_new = np.zeros((dim_new, dim_new), dtype=np.complex128)
                for i, block in enumerate(diag_blocks):
                    start_idx = i * 4
                    end_idx = (i+1) * 4
                    g0_inv_new[start_idx:end_idx, start_idx:end_idx] = block
            
            # 扩展hop_FK: 同样需要扩展
            if self.use_sparse and dim_new > 20:
                hop_new = lil_matrix((dim_new, dim_new), dtype=np.complex128)
            else:
                hop_new = np.zeros((dim_new, dim_new), dtype=np.complex128)
            
            # 将旧的跳跃矩阵放入新矩阵的中间部分
            old_start = num_bands_to_add * 4
            old_end = old_start + dim_old
            if sp.issparse(hop_new):
                hop_new[old_start:old_end, old_start:old_end] = old_hop
            else:
                hop_new[old_start:old_end, old_start:old_end] = old_hop
            
            # 对于特殊位点（结区中心），需要添加新的耦合项
            is_special_site = (site_i == (self.N_SC + self.N_junction // 2 - 1) or 
                              site_i == (self.N_SC + self.N_junction // 2))
            
            if is_special_site and is_junction:
                # 注意：新增的边带也需要耦合
                for n in range(new_sideN):
                    band_i = n - new_max_sidebands
                    start_idx = n * 4
                    end_idx = (n+1)*4
                    
                    # 上边带耦合（从当前边带到下一个边带）
                    if band_i != new_max_sidebands and Vbias >= 0:
                        next_idx = (n+1) * 4
                        if next_idx < dim_new:
                            if sp.issparse(hop_new):
                                hop_new[start_idx:end_idx, next_idx:next_idx+4] = Hop_e
                            else:
                                hop_new[start_idx:end_idx, next_idx:next_idx+4] = Hop_e
                    
                    # 下边带耦合（从当前边带到前一个边带）
                    if band_i != -new_max_sidebands and Vbias >= 0:
                        prev_idx = (n-1) * 4
                        if prev_idx >= 0:
                            if sp.issparse(hop_new):
                                hop_new[start_idx:end_idx, prev_idx:prev_idx+4] = Hop_h.conj().T
                            else:
                                hop_new[start_idx:end_idx, prev_idx:prev_idx+4] = Hop_h.conj().T
            else:
                # 非特殊位点，主对角块已经由旧矩阵填充
                pass
            
            if sp.issparse(hop_new):
                hop_new = hop_new.tocsc()
            
            g0_inv_FKs.append(g0_inv_new)
            hop_FKs.append(hop_new)
        
        return g0_inv_FKs, hop_FKs

    def build_slice_ham(self, Vbias, ham, max_sidebands):
        """构建Floquet哈密顿量切片，支持增量扩建"""
        # 检查是否可以使用增量扩建
        if self.prev_sidebands is not None and max_sidebands > self.prev_sidebands and \
                self.cached_g0_inv_FKs is not None and self.cached_hop_FKs is not None:
            logging.info(f"使用增量扩建: {self.prev_sidebands} -> {max_sidebands} 边带")
            g0_inv_FKs, hop_FKs = self._incremental_expand_slice_ham(Vbias, ham, max_sidebands)
        else:
            # 完整构建模式
            g0_inv_FKs, hop_FKs = self._full_build_slice_ham(Vbias, ham, max_sidebands)
        
        # 更新缓存
        self.prev_sidebands = max_sidebands
        self.cached_g0_inv_FKs = g0_inv_FKs
        self.cached_hop_FKs = hop_FKs
        
        return g0_inv_FKs, hop_FKs

    def _full_build_slice_ham(self, Vbias, ham, max_sidebands):
        """完整构建Floquet哈密顿量切片"""
        sideN = 2 * max_sidebands + 1
        dim = 4 * sideN
        g0_inv_FKs = []
        hop_FKs = []
        
        SNS_Len = self.N_SC * 2 + self.N_junction
        SN_Len = self.N_SC + self.N_junction
        
        for site_i in range(SNS_Len):        
            # 确定当前位点的哈密顿量块
            if self.N_SC <= site_i < SN_Len:
                Ham_onsite = ham['H00_J']
                Ham_hop = ham['H01_J']
                Hop_e = Ham_hop * np.kron(self.tau_e, np.ones((2, 2))) * self.v_tau
                Hop_h = -1 * Ham_hop * np.kron(self.tau_h, np.ones((2, 2))) * self.v_tau
                is_junction = True
            elif site_i < self.N_SC:
                Ham_onsite = ham['H00_L']
                Ham_hop = ham['H01_L']
                is_junction = False
            else:
                Ham_onsite = ham['H00_R']
                Ham_hop = ham['H01_R']
                is_junction = False
            
            # 添加无序势
            Ham_onsite = Ham_onsite + self.disorder_strength * self.disorder_potential(site_i)
            
            # 稀疏矩阵构建
            if self.use_sparse and dim > 20:
                # 构建对角块
                diag_blocks = []
                for n in range(sideN):
                    band_i = n - max_sidebands
                    diag_block = Ham_onsite - band_i * Vbias * np.eye(4)
                    diag_blocks.append(diag_block)
                
                # 创建块对角矩阵
                g0_inv_FK0 = block_diag(diag_blocks, format='csc')
                
                # 创建跳跃矩阵
                hop_FK0 = lil_matrix((dim, dim), dtype=np.complex128)
                
                # 填充主对角块
                for n in range(sideN):
                    band_i = n - max_sidebands
                    idx = slice(n * 4, (n + 1) * 4)
                    
                    # 特殊位点处理
                    is_special_site = (site_i == (self.N_SC + self.N_junction // 2 - 1)) #or 
                                      # site_i == (self.N_SC + self.N_junction // 2))
                    
                    if is_special_site and is_junction:
                        # 上边带耦合
                        if band_i != max_sidebands and Vbias >= 0:
                            next_idx = slice((n + 1) * 4, (n + 2) * 4)
                            hop_FK0[idx, next_idx] = Hop_e
                        
                        # 下边带耦合
                        if band_i != -max_sidebands and Vbias >= 0:
                            prev_idx = slice((n - 1) * 4, n * 4)
                            hop_FK0[idx, prev_idx] = Hop_h.conj().T
                    else:
                        hop_FK0[idx, idx] = Ham_hop
                        
                hop_FK0 = hop_FK0.tocsc()
            else:
                # 稠密矩阵路径
                g0_inv_FK0 = np.zeros((dim, dim), dtype=np.complex128)
                hop_FK0 = np.zeros_like(g0_inv_FK0)
                
                for n in range(sideN):
                    band_i = n - max_sidebands
                    idx = slice(n * 4, (n + 1) * 4)
                    g0_inv_FK0[idx, idx] = Ham_onsite - band_i * Vbias * np.eye(4)
                    
                    # 特殊位点处理
                    is_special_site = (site_i == (self.N_SC + self.N_junction // 2 - 1) or 
                                      site_i == (self.N_SC + self.N_junction // 2))
                    
                    if is_special_site and is_junction:
                        if band_i != max_sidebands and Vbias >= 0:
                            next_idx = slice((n + 1) * 4, (n + 2) * 4)
                            hop_FK0[idx, next_idx] = Hop_e
                        if band_i != -max_sidebands and Vbias >= 0:
                            prev_idx = slice((n - 1) * 4, n * 4)
                            hop_FK0[idx, prev_idx] = Hop_h.conj().T
                    else:
                        hop_FK0[idx, idx] = Ham_hop
            
            g0_inv_FKs.append(g0_inv_FK0)
            hop_FKs.append(hop_FK0)
        
        return g0_inv_FKs, hop_FKs

class GreenFunctionCalculator:
    def __init__(self, params):
        # 计算参数
        self.recursion_depth = params['recursion_depth']
        self.eta = params['eta']  # 虚部无穷小
        self.mu_lead = params['mu_lead']  # 引线化学势
        self.delta = params['delta']  # 超导能隙
        self.unit_factor_nA = 1  #2 * 1.6e-19 * 1e-3 / (4.13e-15) * 1e9  # 单位转换因子 (nA) (2e/h) not (2e/hbar)
        
        # 稀疏计算配置
        self.use_sparse = params.get('use_sparse', True)
        self.sparse_threshold = params.get('sparse_threshold', 0.3)
        self.sparse_depth_limit = params.get('sparse_depth_limit', 5)
        
        # 预分解器缓存
        self.factorized_cache = {}
        
        logging.info(f"GreenFunctionCalculator initialized, Sparse mode: {self.use_sparse}") 

    def fermi_dirac(self, omega):
        """费米狄拉克分布"""
        kBT = 5e-3 * self.delta  # 温度 (meV)
        return 1.0 / (1.0 + np.exp((omega - self.mu_lead) / kBT))

    def surface_gf_sc(self, omega, H00, H01, direction='left'):
        """表面格林函数计算（超导引线）"""
        E0 = omega + 1j * self.eta
        dim = H00.shape[0]
        eps = eps_s = H00.copy()
        
        if direction == 'right':
            alpha = H01
            beta = H01.conj().T
        else:  # 'left'
            alpha = H01.conj().T
            beta = H01
        
        # 递归计算
        for _ in range(self.recursion_depth):
            g = la.inv(E0 * np.eye(dim) - eps)
            eps_s = eps_s + alpha @ g @ beta
            eps = eps + alpha @ g @ beta + beta @ g @ alpha
            
            alpha_new = alpha @ g @ alpha
            beta_new = beta @ g @ beta
            
            if np.linalg.norm(alpha_new) < 1e-12:
                break
            alpha, beta = alpha_new, beta_new
        
        g_surface = la.inv(E0 * np.eye(dim) - eps_s)
        hgh = H01 @ g_surface @ H01.conj().T if direction == 'right' else H01.conj().T @ g_surface @ H01
        
        return g_surface, hgh

    def recursive_sweep(self, omega, sE_initial, range_sets, hop_FKs, g0_inv_FKs):
        """递归扫描，完全兼容稀疏和稠密矩阵，优化稀疏矩阵操作"""
        range_1, range_2, offset_L = range_sets
        sE_L0, sE_R0, sE_Lf0, sE_Rf0 = sE_initial
        
        # 获取矩阵维度
        first_mat = g0_inv_FKs[0]
        dim = first_mat.shape[0]
        
        # 创建单位矩阵（兼容稀疏和稠密）
        if sp.issparse(g0_inv_FKs[0]):
            E0_I0 = (omega + 1j * self.eta) * sp.eye(dim, format='csc', dtype=np.complex128)
        else:
            E0_I0 = (omega + 1j * self.eta) * np.eye(dim, dtype=np.complex128)
        
        # 初始化列表
        g0s_invL = []
        sELs = []
        sERs = []
        sELs_less = []
        sERs_less = []
        
        # 初始化中间变量
        GL_i = None
        GR_i = None
        GL_less_i = None
        GR_less_i = None
        
        for recur_i in range(range_1, range_2):
            recur_i_R = range_2 - 1 - recur_i + offset_L
            
            # 获取当前矩阵
            g0_inv_i = g0_inv_FKs[recur_i]
            g0_inv_iR = g0_inv_FKs[recur_i_R]
            alpha_i = hop_FKs[recur_i]
            alpha_iR = hop_FKs[recur_i_R]
            
            # 初始自能处理
            if recur_i == range_1 or recur_i_R == range_2 - 1:
                sEL_i = sE_L0
                sER_i = sE_R0
                sEL_less_i = sE_Lf0.conj().T - sE_Lf0
                sER_less_i = sE_Rf0.conj().T - sE_Rf0
            else:
                # 使用稀疏矩阵乘法避免转换
                if sp.issparse(alpha_i) and sp.issparse(GL_i):
                    sEL_i = alpha_i.conj().T @ GL_i @ alpha_i
                else:
                    # 确保矩阵为稠密后再进行运算
                    if sp.issparse(GL_i):
                        GL_i = GL_i.toarray()
                    if sp.issparse(alpha_i):
                        alpha_i = alpha_i.toarray()
                    sEL_i = alpha_i.conj().T @ GL_i @ alpha_i
                
                if sp.issparse(alpha_iR) and sp.issparse(GR_i):
                    sER_i = alpha_iR @ GR_i @ alpha_iR.conj().T
                else:
                    if sp.issparse(GR_i):
                        GR_i = GR_i.toarray()
                    if sp.issparse(alpha_iR):
                        alpha_iR = alpha_iR.toarray()
                    sER_i = alpha_iR @ GR_i @ alpha_iR.conj().T
                
                # 同样处理lesser分量
                if sp.issparse(alpha_i) and sp.issparse(GL_less_i):
                    sEL_less_i = alpha_i.conj().T @ GL_less_i @ alpha_i
                else:
                    if sp.issparse(GL_less_i):
                        GL_less_i = GL_less_i.toarray()
                    if sp.issparse(alpha_i):
                        alpha_i = alpha_i.toarray()
                    sEL_less_i = alpha_i.conj().T @ GL_less_i @ alpha_i
                
                if sp.issparse(alpha_iR) and sp.issparse(GR_less_i):
                    sER_less_i = alpha_iR @ GR_less_i @ alpha_iR.conj().T
                else:
                    if sp.issparse(GR_less_i):
                        GR_less_i = GR_less_i.toarray()
                    if sp.issparse(alpha_iR):
                        alpha_iR = alpha_iR.toarray()
                    sER_less_i = alpha_iR @ GR_less_i @ alpha_iR.conj().T
            
            # 构建逆格林函数
            gi_inv_L = E0_I0 - g0_inv_i
            gi_inv_R = E0_I0 - g0_inv_iR
            
            # 预分解矩阵以提高性能
            cache_key_L = (id(gi_inv_L), id(sEL_i))
            if cache_key_L in self.factorized_cache:
                solve_L = self.factorized_cache[cache_key_L]
                GL_i = solve_L(sp.eye(dim, format='csc', dtype=np.complex128))
            else:
                mat_to_inv = gi_inv_L - sEL_i
                if sp.issparse(mat_to_inv):
                    solve_L = factorized(mat_to_inv.tocsc())
                    GL_i = solve_L(sp.eye(dim, format='csc', dtype=np.complex128))
                    self.factorized_cache[cache_key_L] = solve_L
                else:
                    GL_i = la.inv(mat_to_inv)
            
            cache_key_R = (id(gi_inv_R), id(sER_i))
            if cache_key_R in self.factorized_cache:
                solve_R = self.factorized_cache[cache_key_R]
                GR_i = solve_R(sp.eye(dim, format='csc', dtype=np.complex128))
            else:
                mat_to_inv = gi_inv_R - sER_i
                if sp.issparse(mat_to_inv):
                    solve_R = factorized(mat_to_inv.tocsc())
                    GR_i = solve_R(sp.eye(dim, format='csc', dtype=np.complex128))
                    self.factorized_cache[cache_key_R] = solve_R
                else:
                    GR_i = la.inv(mat_to_inv)
            
            # 计算lesser分量
            if sp.issparse(GL_i):
                GL_less_i = GL_i @ sEL_less_i @ GL_i.conj().T
            else:
                GL_less_i = GL_i @ sEL_less_i @ GL_i.conj().T
            
            if sp.issparse(GR_i):
                GR_less_i = GR_i @ sER_less_i @ GR_i.conj().T
            else:
                GR_less_i = GR_i @ sER_less_i @ GR_i.conj().T
            
            # 存储结果
            sELs.append(sEL_i)
            sERs.append(sER_i)
            sELs_less.append(sEL_less_i)
            sERs_less.append(sER_less_i)
            g0s_invL.append(gi_inv_L)
        
        return g0s_invL, [sELs, sERs, sELs_less, sERs_less]

    def compute_self_energies(self, omega, lead_type, ham, max_sidebands, Vbias, N_SC, N_junction, hop_FKs, g0_inv_FKs):
        """计算自能，完全兼容稀疏和稠密矩阵"""
        sideN = 2 * max_sidebands + 1
        dim = 4 * sideN
        
        # 创建稠密矩阵
        sEL = np.zeros((dim, dim), dtype=np.complex128)
        sER = np.zeros_like(sEL)
        sEL_f0 = np.zeros_like(sEL)
        sER_f0 = np.zeros_like(sEL)
        
        for n in range(sideN):
            band_i = n - max_sidebands
            omega_shifted = omega + band_i * Vbias
            f0 = self.fermi_dirac(omega_shifted)
            a = slice(n * 4, n * 4 + 4)
            
            # 左侧超导引线
            _, sEL_block = self.surface_gf_sc(omega_shifted, ham['H00_L_lead'], ham['H01_L_lead'], direction='left')
            sEL[a, a] = sEL_block
            sEL_f0[a, a] = sEL_block * f0
            
            # 右侧超导引线
            _, sER_block = self.surface_gf_sc(omega_shifted, ham['H00_R_lead'], ham['H01_R_lead'], direction='right')
            sER[a, a] = sER_block
            sER_f0[a, a] = sER_block * f0
        
        if lead_type == 'infinite':
            return [None, None], [sEL, sER, sEL_f0, sER_f0]
        
        # 处理有限引线
        sE_initial = [sEL, sER, sEL_f0, sER_f0]
        # sE_initial = [0*sEL, 0*sER, 0*sEL_f0, 0*sER_f0]
        
        # 处理左侧引线区域
        g0_inv_Ld, sE_Ld = self.obtain_recursive_list(
            omega, 'left_lead', sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        # 处理右侧引线区域
        g0_inv_Rd, sE_Rd = self.obtain_recursive_list(
            omega, 'right_lead', sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        # 确保矩阵为稠密
        if sp.issparse(g0_inv_Ld[-1]):
            g0_inv_L = g0_inv_Ld[-1].toarray()
        else:
            g0_inv_L = g0_inv_Ld[-1]
        
        if sp.issparse(g0_inv_Rd[-1]):
            g0_inv_R = g0_inv_Rd[-1].toarray()
        else:
            g0_inv_R = g0_inv_Rd[-1]
        
        # 获取自能
        sEL_left = sE_Ld[0][-1] if not sp.issparse(sE_Ld[0][-1]) else sE_Ld[0][-1].toarray()
        sER_left = sE_Ld[1][0] if not sp.issparse(sE_Ld[1][0]) else sE_Ld[1][0].toarray()
        sEL_right = sE_Rd[0][0] if not sp.issparse(sE_Rd[0][0]) else sE_Rd[0][0].toarray()
        sER_right = sE_Rd[1][-1] if not sp.issparse(sE_Rd[1][-1]) else sE_Rd[1][-1].toarray()
        
        # 计算格林函数
        g_retard_L = la.inv(g0_inv_L - sEL_left - sER_left)
        g_retard_R = la.inv(g0_inv_R - sEL_right - sER_right)
        
        # 更新自能
        sEL = np.zeros_like(sEL)
        sER = np.zeros_like(sER)
        sEL_f0 = np.zeros_like(sEL_f0)
        sER_f0 = np.zeros_like(sER_f0)
        
        hop_LC = hop_FKs[N_SC - 1]
        hop_CR = hop_FKs[N_SC + N_junction]
        
        # 确保跳跃矩阵为稠密
        if sp.issparse(hop_LC):
            hop_LC = hop_LC.toarray()
        if sp.issparse(hop_CR):
            hop_CR = hop_CR.toarray()
        
        for n in range(sideN):
            band_i = n - max_sidebands
            omega_shifted = omega + band_i * Vbias
            f0 = self.fermi_dirac(omega_shifted)
            a = slice(n * 4, n * 4 + 4)
            
            sEL[a, a] = hop_LC[a, a].conj().T @ g_retard_L[a, a] @ hop_LC[a, a]
            sER[a, a] = hop_CR[a, a] @ g_retard_R[a, a] @ hop_CR[a, a].conj().T
            sEL_f0[a, a] = sEL[a, a] * f0
            sER_f0[a, a] = sER[a, a] * f0
        
        return [g_retard_L, g_retard_R], [sEL, sER, sEL_f0, sER_f0]

    def obtain_recursive_list(self, omega, region_type, sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs):
        """获取递归列表（保持不变）"""
        if region_type == 'left_lead':
            sE_initial = [sE * i0 for sE, i0 in zip(sE_initial, [1, 0, 1, 0])]
            range_sets = [0, N_SC, 0]
            recur_depth = N_SC
        elif region_type == 'right_lead':
            sE_initial = [sE * i0 for sE, i0 in zip(sE_initial, [0, 1, 0, 1])]
            range_sets = [N_SC + N_junction, 2 * N_SC + N_junction, N_SC + N_junction]
            recur_depth = N_SC
        elif region_type == 'junction':
            range_sets = [N_SC, N_SC + N_junction, N_SC]
            recur_depth = N_junction
        
        return self.recursive_sweep(omega, sE_initial, range_sets, hop_FKs, g0_inv_FKs)

    def _compute_current_at_omega(self, omega, lead_type, region_type, ham_builder, max_sidebands, Vbias, N_SC, N_junction, g0_inv_FKs, hop_FKs):
        """电流计算，使用传入的哈密顿量切片"""
        # 计算自能
        Gs, sE_initial = self.compute_self_energies(
            omega, lead_type, ham_builder.construct_hamiltonian(0), max_sidebands, Vbias, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        # 获取递归列表
        g0_inv, sE4s_leads = self.obtain_recursive_list(
            omega, region_type, sE_initial, N_SC, N_junction, hop_FKs, g0_inv_FKs)
        
        recur_depth = N_junction
        
        # 准备tau_z矩阵
        sideN = 2 * max_sidebands + 1
        tau_z_matrix = np.kron(np.eye(sideN), np.kron(ham_builder.tau_z, np.eye(2)))
        
        trace0 = 0
        for recur_i in range(recur_depth):
            recur_i_R = recur_depth - 1 - recur_i
            
            # 获取当前矩阵并确保为稠密
            gi_inv = g0_inv[recur_i]
            if sp.issparse(gi_inv):
                gi_inv = gi_inv.toarray()
            
            sEL_i = sE4s_leads[0][recur_i]
            if sp.issparse(sEL_i):
                sEL_i = sEL_i.toarray()
            
            sER_i = sE4s_leads[1][recur_i_R]
            if sp.issparse(sER_i):
                sER_i = sER_i.toarray()
            
            sEL_less_i = sE4s_leads[2][recur_i]
            if sp.issparse(sEL_less_i):
                sEL_less_i = sEL_less_i.toarray()
            
            sER_less_i = sE4s_leads[3][recur_i_R]
            if sp.issparse(sER_less_i):
                sER_less_i = sER_less_i.toarray()
            
            # 使用稠密矩阵计算
            G_nn_ret = la.inv(gi_inv - sEL_i - sER_i)
            G_nn_adv = G_nn_ret.conj().T
            
            sEL_nn_less = sEL_less_i
            sE_nn_less = sEL_less_i + sER_less_i
            sEL_nn = sEL_i
            
            G_nn_less = G_nn_ret @ sE_nn_less @ G_nn_adv
            temp = (G_nn_ret @ sEL_nn_less + G_nn_less @ sEL_nn.conj().T) @ tau_z_matrix
            
            trace_val = np.trace(temp).real
            trace0 += trace_val * self.unit_factor_nA
        
        return trace0

class JosephsonJunctionSolver:
    def __init__(self, params, path_manager):
        # 添加稀疏配置参数
        params.setdefault('use_sparse', True)
        params.setdefault('sparse_threshold', 0.3)
        params.setdefault('sparse_depth_limit', 5)
        
        self.params = params
        self.path_manager = path_manager
        self.output_dir = path_manager.get_raw_data_dir()
        
        # 系统参数
        self.params = params
        self.ham_builder = HamiltonianBuilder(params)
        self.gf_calculator = GreenFunctionCalculator(params)
        
        # 计算参数
        self.max_sidebands = params['max_sidebands']
        self.omega_points = params['omega_points']
        self.Vbias_points = params['Vbias_points']
        self.B_points = params.get('B_points', 0)
        self.job_parallel = params['job_parallel']
        
        # 创建输出目录
        os.makedirs(self.output_dir, exist_ok=True)
        
        # 单位转换
        self.unit_factor_nA = 2 * 1.6e-19 * 1e-3 / (4.13e-15) * 1e9  # nA
        # 自适应计算参数
        self.adaptive_iv = params.get('adaptive_iv', False)
        self.adaptive_tol = params.get('adaptive_tol', 0.01)
        self.adaptive_max_N = params.get('adaptive_max_N', 20)
        self.adaptive_method = params.get('adaptive_method', 'basic')  # 'basic', 'dynamic', 'step'

        # 哈密顿量切片缓存
        self.ham_slice_cache = {}
        
        # 生成公共时间戳
        self.common_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        logging.info("JosephsonJunctionSolver initialized")
        logging.info(f"Superconductor: {params['N_SC']} sites, Junction: {params['N_junction']} sites")
        logging.info(f"Recursion depth: {params['recursion_depth']}, Max sidebands: {params['max_sidebands']}")

    def _get_base_metadata(self):
        """返回所有计算的通用元数据参数"""
        # 基础元数据，始终包含
        metadata = {
            "N_SC": self.params['N_SC'],
            "N_junction": self.params['N_junction'],
            "delta": self.params['delta'],
            "alpha": self.params['alpha'],
            "v_tau": self.params['v_tau'],
            "B": self.params['B'],
            "Vdis":self.params['disorder_strength'],
            "max_sidebands": self.params['max_sidebands'],
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "common_timestamp": self.common_timestamp  # 添加公共时间戳
        }
        
        # 添加disorder类型
        disorder_type = self.params.get('disorder_type', 'none')
        metadata["disorder_type"] = disorder_type
        
        # 根据disorder类型添加相关参数
        if disorder_type != 'none':
            if disorder_type == 'gaussian':
                metadata.update({
                    "Vdis": self.params.get('Vdis_gau', 0.0),
                    "decayL": self.params.get('decayL_gau', 50),
                    "Xdis": self.params.get('Xdis_gau', 150)
                })
            elif disorder_type == 'smooth':
                metadata.update({
                    "decayL": self.params.get('decayL_smooth', 50),
                    "Vdis": self.params.get('Vdis_smooth', 0.0),
                    "Vd": self.params.get('Vd_smooth', 0.8)
                })
            elif disorder_type in ['random_typeI', 'random_typeII']:
                metadata.update({
                    "N_imp": self.params.get('N_imp1', 52),
                    "lambda_imp": self.params.get('lambda_imp1', 18.0),
                    "V0": self.params.get('V0_imp1', 0.0),
                    "n_d": self.params.get('Nd_imp2', 10.0),
                    "random_lambda_imp": self.params.get('lambda_imp2', 20.0),
                    "random_V0": self.params.get('V0_imp2', 0.0),
                    "a0": self.params.get('a0', 10.0)
                })
            elif disorder_type == 'nonhermitian':
                metadata.update({
                    "charge_V0": self.params.get('charge_V0', 0.5),
                    "n_d": self.params.get('n_d', 10.0)
                })
            elif disorder_type == 'from_file':
                metadata.update({
                    "disorder_file": self.params.get('disorder_file', 'disorder_data.txt')
                })
        
        # 添加自适应参数
        if self.params.get('adaptive_iv', False):
            metadata.update({
                "adaptive_method": self.params.get('adaptive_method', 'basic'),
                "adaptive_tol": self.params.get('adaptive_tol', 0.01),
                "adaptive_max_N": self.params.get('adaptive_max_N', 20)
            })
        
        return metadata

    def save_disorder_data(self):
        """保存无序数据（如果disorder类型为'from_file'则跳过）"""
        # 如果disorder类型为'from_file'，则不保存
        if self.params.get('disorder_type', 'none') == 'from_file':
            logging.info("Disorder type is 'from_file', skipping disorder data saving")
            return {
                "csv_file": None,
                "plot_file": None,
                "metadata_file": None
            }
            
        disorder_dir = os.path.join(self.output_dir, "disorder_data")
        os.makedirs(disorder_dir, exist_ok=True)
        
        # 保存图表
        plot_file = self._plot_disorder_distribution(disorder_dir)
        
        # 保存CSV
        csv_file = self._save_disorder_csv(disorder_dir)
        
        # 保存元数据
        metadata_file = self._save_disorder_metadata(disorder_dir)
        
        return {
            "csv_file": csv_file,
            "plot_file": plot_file,
            "metadata_file": metadata_file
        }

    def _plot_disorder_distribution(self, output_dir):
        """绘制无序分布并保存为PNG"""
        # 使用公共时间戳
        plot_filename = os.path.join(output_dir, f"disorder_plot_{self.common_timestamp}.png")
        
        # 创建图表
        plt.figure(figsize=(12, 6))
        plt.plot(self.ham_builder.disorder_distribution, 'b-', linewidth=1.5)
        
        # 添加区域标记
        plt.axvline(x=self.params['N_SC'], color='r', linestyle='--', alpha=0.5)
        plt.axvline(x=self.params['N_SC'] + self.params['N_junction'], color='r', linestyle='--', alpha=0.5)
        
        # 标签和标题
        plt.title(f'Disorder Potential ({self.params.get("disorder_type", "none")})', fontsize=14)
        plt.xlabel('Site Index', fontsize=12)
        plt.ylabel('Potential (meV)', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.7)
        
        # 保存图表
        plt.savefig(plot_filename, dpi=300)
        plt.close()
        logging.info(f"Disorder plot saved: {plot_filename}")
        
        return plot_filename

    def _save_disorder_csv(self, output_dir):
        """保存无序分布到CSV文件"""
        # 使用公共时间戳
        csv_filename = os.path.join(output_dir, f"disorder_data_{self.common_timestamp}.csv")
        
        # 创建CSV文件
        df = pd.DataFrame({
            'site_index': np.arange(len(self.ham_builder.disorder_distribution)),
            'disorder_value': self.ham_builder.disorder_distribution
        })
        df.to_csv(csv_filename, index=False)
        logging.info(f"Disorder data saved: {csv_filename}")
        
        return csv_filename

    def _save_disorder_metadata(self, output_dir):
        """保存无序元数据到JSON文件，仅相关参数"""
        disorder_type = self.params.get('disorder_type', 'none')
        
        # 基础元数据
        metadata = {
            "disorder_type": disorder_type,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "common_timestamp": self.common_timestamp  # 添加公共时间戳
        }
        
        # 根据disorder类型添加相关参数
        if disorder_type != 'none':
            if disorder_type == 'gaussian':
                metadata.update({
                    "Vdis": self.params.get('Vdis_gau', 0.0),
                    "decayL": self.params.get('decayL_gau', 50),
                    "Xdis": self.params.get('Xdis_gau', 150)
                })
            elif disorder_type == 'smooth':
                metadata.update({
                    "decayL": self.params.get('decayL_smooth', 50),
                    "Vdis": self.params.get('Vdis_smooth', 0.0),
                    "Vd": self.params.get('Vd_smooth', 0.8)
                })
            elif disorder_type in ['random_typeI', 'random_typeII']:
                metadata.update({
                    "N_imp": self.params.get('N_imp1', 52),
                    "lambda_imp": self.params.get('lambda_imp1', 18.0),
                    "V0": self.params.get('V0_imp1', 0.0),
                    "n_d": self.params.get('Nd_imp2', 10.0),
                    "random_lambda_imp": self.params.get('lambda_imp2', 20.0),
                    "random_V0": self.params.get('V0_imp2', 0.0),
                    "a0": self.params.get('a0', 10.0)
                })
            elif disorder_type == 'nonhermitian':
                metadata.update({
                    "charge_V0": self.params.get('charge_V0', 0.5),
                    "n_d": self.params.get('n_d', 10.0)
                })
            elif disorder_type == 'from_file':
                metadata.update({
                    "disorder_file": self.params.get('disorder_file', 'disorder_data.txt')
                })
        
        # 使用公共时间戳
        metadata_file = os.path.join(output_dir, f"disorder_metadata_{self.common_timestamp}.json")
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=4)
        
        logging.info(f"Disorder metadata saved: {metadata_file}")
        
        return metadata_file

    def run_calculation(self, calc_type):
        """运行指定计算"""
        # 保存无序数据（如果disorder类型不是'from_file'）
        if self.params.get('disorder_type', 'none') != 'none' and self.params.get('disorder_type', 'none') != 'from_file':
            self.save_disorder_data()
        
        # 保存任务元数据
        self.path_manager.save_task_metadata(self.params)
        
        # 运行计算
        if calc_type == "DC_IV":
            metadata, *data = self.compute_dc_iv()
        elif calc_type == "DC_IV_Bsweep":
            metadata, *data = self.compute_dc_iv_Bsweep()
        else:
            raise ValueError("Invalid calculation type")
        
        # 保存结果
        filepath = self.save_results(metadata, data)
        
        # 绘图
        plot_path = JJPlotter.plot_single_result(metadata, data, self.path_manager.get_plots_dir())
        
        return filepath, plot_path
    
    def save_results(self, metadata, data):
        """保存计算结果到文件"""
        calc_type = metadata["calculation"]
        # 使用公共时间戳
        common_timestamp = metadata.get("common_timestamp", datetime.now().strftime("%Y%m%d_%H%M%S"))
        filename = f"{calc_type}_{common_timestamp}.csv"
        filepath = os.path.join(self.output_dir, filename)
        
        metadata_lines = ["# Calculation Metadata"]
        for key, value in metadata.items():
            metadata_lines.append(f"# {key}: {value}")
        
        # 根据计算类型创建数据框
        if calc_type == "DC_IV":
            if len(data) == 3:  # 包含边带数
                Vbias, current, sideband = data
                df = pd.DataFrame({
                    'Bias Voltage (meV)': Vbias,
                    'Current (nA)': current,
                    'SidebandN': sideband
                })
            else:
                Vbias, current = data
                df = pd.DataFrame({
                    'Bias Voltage (meV)': Vbias,
                    'Current (nA)': current
                })
        elif calc_type == "DC_IV_Bsweep":
            B, current, sideband = data
            df = pd.DataFrame({
                'Magnetic Field': B,
                'Current (nA)': current,
                'sidebandN':sideband,
                'Fixed Bias (meV)': metadata['fixed_Vbias']
            })
            
        # 写入文件
        with open(filepath, 'w') as f:
            f.write("\n".join(metadata_lines))
            f.write("\n")
            df.to_csv(f, index=False)
        
        return filepath
    
    def tics(self, stamp=False):
        """Start timer """
        if stamp:
            print(datetime.now().strftime("Starting at %d/%m/%Y %H:%M:%S"))
        return time.time()

    def toc(self, tic, disp=True, nice_format=False, stamp=False):
        """Stop timer and display elapsed time"""
        elapsed_time = time.time() - tic
        if stamp:
            print(datetime.now().strftime("Finished at %d/%m/%Y %H:%M:%S"))
        if disp:
            if (not nice_format) or (elapsed_time < 60):
                print(f"Elapsed time: {np.round(elapsed_time,3)} seconds")
            else:
                hrs, rem = divmod(elapsed_time, 3600)
                mins, secs = divmod(rem, 60)
                print(f"Elapsed time: {int(hrs):02d}:{int(mins):02d}:{int(secs):02d}")
        return elapsed_time
        
    @staticmethod
    def generate_nonuniform_grid(
        start: float,           # 整个区域起始点
        end: float,             # 整个区域结束点
        dense_start: float,     # 期望的dense区域起点
        dense_end: float,       # 期望的dense区域终点
        space_non_dense: float = 0.05,  # 非dense区域间距 (默认0.05)
        space_ratio: float = 0.4,       # dense区域间距比例 (默认0.4)
        max_points: int = 300           # 最大总点数 (默认300)
        ) ->np.ndarray:
        """ """
        space_dense = space_non_dense * space_ratio
        
        # 调整dense区域边界
        n_left_segments = max(0, int(np.floor((dense_start - start) / space_non_dense)))
        adjusted_dense_start = start + n_left_segments * space_non_dense

        max_dense_end = end - space_non_dense  
        
        if max_dense_end > adjusted_dense_start:   # dense区域右侧至少有一个点
            min_segments = max(0, int(np.ceil((dense_end - adjusted_dense_start) / space_dense)))
            max_segments = int(np.floor((max_dense_end - adjusted_dense_start) / space_dense))
            n_dense_segments = min(min_segments, max_segments) if min_segments > 0 else max_segments
            adjusted_dense_end = adjusted_dense_start + n_dense_segments * space_dense
        else:
            n_dense_segments = 0
            adjusted_dense_end = adjusted_dense_start
        
        # 计算各区域点数
        n_left = n_left_segments
        n_dense = n_dense_segments + 1 if n_dense_segments > 0 else 0
        n_right_segments = max(0, int(np.floor((end - adjusted_dense_end) / space_non_dense)))
        n_right = n_right_segments
        
        points = []
        
        # 左侧非dense区域
        if n_left > 0:
            left_points = np.linspace(start, adjusted_dense_start, n_left + 1, endpoint=False)
            points.extend(left_points)
        
        # dense区域
        if n_dense > 0:
            dense_points = np.linspace(adjusted_dense_start, adjusted_dense_end, n_dense)
            points.extend(dense_points)
        
        # 右侧非dense区域
        if n_right > 0:
            right_start = adjusted_dense_end + space_non_dense
            right_points = np.linspace(
                right_start, 
                right_start + n_right * space_non_dense, 
                n_right + 1, 
                endpoint=True
            )
            # 确保不超过end
            right_points = right_points[right_points <= end]
            points.extend(right_points)
        
        points = np.array(points)
        points.sort()
        
        # 确保包含起点和终点
        if abs(points[0] - start) > 1e-10:
            points = np.insert(points, 0, start)
        if abs(points[-1] - end) > 1e-10:
            points = np.append(points, end)
        
        # 点数限制
        if len(points) > max_points:
            logging.warning(f"网格点数({len(points)})超过最大限制({max_points})，进行截断")
            points = points[:max_points]
        
        return points

    @staticmethod
    def generate_uniform_grid(
        start: float,           # 区域起始点
        end: float,             # 区域结束点
        spacing: float = 0.05,  # 点间距 (默认0.05)
        max_points: int = 300   # 最大总点数 (默认300)
        ) -> np.ndarray:
        """ """
        num_segments = (end - start) / spacing
        n_points = int(np.floor(num_segments)) + 1
        
        # 调整间距以满足最大点数限制
        if n_points > max_points:
            n_points = max_points
            spacing = (end - start) / (n_points - 1) if n_points > 1 else 0
        
        points = np.linspace(start, end, n_points)
        return points
        
    def generate_Vbias_vals(self):
        """Generate Vbias values using the same logic as compute_dc_iv"""
        # delta = self.params['delta']
        # Config. [0.03,3,0.03,2.1,0.03,0.4,300]
        # Config. [0.03,3,0.12,1.2,0.04,0.4,300]
        
        Vbias_max_ratio = self.params.get('Vbias_max_ratio', 2.0)
        
        if Vbias_max_ratio <= 2.2:
            return self.generate_uniform_grid(0.01, (Vbias_max_ratio+0.01),
                                              0.01,self.params['Vbias_points'])  
        else:
            # compute_dc_iv 
            return self.generate_nonuniform_grid(
                start = 0.01,   ## previous, set as 0.001
                end = Vbias_max_ratio ,
                dense_start= 0.12,
                dense_end = 1.2,
                space_non_dense =0.04,
                space_ratio= 0.25,
                max_points = self.params['Vbias_points']
            )   
            
    def compute_dc_iv(self):
        """Compute DC I-V characteristic, returns (metadata, Vbias_vals, currents, sideband_Ns)"""
        start_time = time.time()
        delta = self.params['delta']
        
        Vbias_vals= delta*self.generate_Vbias_vals()
        
        logging.info("Starting DC IV calculation")
        results = Parallel(n_jobs=self.job_parallel[0])(
            delayed(self.compute_parallel_aux)(x0, "DC_IV") for x0 in Vbias_vals #[::-1]
        )
        
        # 分离电流和边带数
        currents = [res[0] for res in results]
        sideband_Ns = [res[1] for res in results]
        
        # Prepare metadata
        metadata = self._get_base_metadata()
        metadata.update({
            "calculation": "DC_IV",
            "duration": time.time() - start_time,
            "Vbias_points": self.Vbias_points, })
        
        logging.info(f"DC IV calculation completed, duration: {metadata['duration']:.2f} sec")
        return metadata, Vbias_vals, currents, sideband_Ns
    

    def compute_parallel_aux(self, Vari_input, calc_type):
        """并行计算辅助函数（使用稀疏优化）"""
        if calc_type == "DC_IV":
            Vbias = Vari_input
            if self.params['adaptive_iv']:
                if self.params['adaptive_method'] == 'advanced':
                    return self.adaptive_current_advanced(Vbias)
            else:
                current = self.compute_current_at_bias(Vbias)
                return current, self.params['max_sidebands']
                
        elif calc_type == "DC_IV_Bsweep":
            B, Vbias = Vari_input  # 解包磁场和偏压
            # 更新磁场值
            self.ham_builder.B = B
            
            # 计算当前磁场和偏压下的电流
            if self.params['adaptive_iv']:
                current, sideband = self.compute_parallel_aux(Vbias, "DC_IV")
            else:
                current = self.compute_current_at_bias(Vbias)
                sideband = self.params['max_sidebands']
            
            return current, sideband
        
    def compute_dc_iv_Bsweep(self):
        """在固定偏压下扫描磁场计算直流电流"""
        start_time = time.time()
        Vbias = self.params['fixed_Vbias']  # 获取固定偏压值
        B_vals = self.generate_uniform_grid(
            start=0,
            end=self.params['B_max'],
            spacing=0.02,
            max_points=self.B_points
        )
        
        logging.info(f"Starting DC IV with B sweep @ Vbias={Vbias} meV")
        results = Parallel(n_jobs=self.job_parallel[0])(
            delayed(self.compute_parallel_aux)((B, Vbias), "DC_IV_Bsweep") 
            for B in B_vals
        )

        # 分离电流和边带数
        currents = [res[0] for res in results]
        sideband_Ns = [res[1] for res in results]

        
        # 准备元数据
        metadata = self._get_base_metadata()
        metadata.update({
            "calculation": "DC_IV_Bsweep",
            "duration": time.time() - start_time,
            "fixed_Vbias": Vbias,
            "B_points": self.B_points,
            "B_max": self.params['B_max']
        })
        
        logging.info(f"DC IV with B sweep completed, duration: {metadata['duration']:.2f} sec")
        
        return metadata, B_vals, currents, sideband_Ns
    
    def adaptive_current_advanced(self, Vbias):
        """改进的自适应策略（增量版本）"""
        # 获取自适应配置
        init_N, step_size, max_N = self._get_adaptive_config(Vbias)
        
        # 重置哈密顿量缓存
        self.ham_builder.prev_sidebands = None
        self.ham_builder.cached_g0_inv_FKs = None
        self.ham_builder.cached_hop_FKs = None
        
        # 初始计算
        N = init_N
        I_current = self.compute_current_at_bias(Vbias, N)
        
        # 记录收敛历史
        history = [(N, I_current)]
        
        # 自适应迭代
        for _ in range(15):  # 增加迭代次数限制
            # 检查是否达到最大边带数
            if N >= max_N:
                logging.warning(f"达到最大边带数 {max_N} @ Vbias={Vbias:.4f}Δ")
                # 返回历史中最接近收敛的结果
                return self._select_best_result(history)
            
            # 增加边带数
            new_N = N + step_size
            logging.debug(f"扩展边带数: {N} → {new_N} @ Vbias={Vbias:.4f}Δ")
            
            # 使用增量方法计算新电流
            I_next = self.compute_current_at_bias(Vbias, new_N)
            history.append((new_N, I_next))
            
            # 计算变化量
            delta_I = abs(I_next - I_current)
            rel_delta = delta_I / (abs(I_current) + 1e-6)
            
            logging.debug(f"边带数 {new_N}: 电流={I_next:.6f}, ΔI={delta_I:.6f}, 相对Δ={rel_delta:.4f}")
            
            # 收敛检查 - 同时考虑相对和绝对误差
            if rel_delta < self.params['rel_tol'] or delta_I < self.params['abs_tol']:
                logging.info(f"收敛于 {new_N} 边带 @ Vbias={Vbias:.4f}Δ, 电流={I_next:.6f}nA")
                return I_next, new_N
            
            I_current = I_next
            N = new_N
        
        # 未收敛时返回最佳结果
        logging.warning(f"达到最大迭代次数 @ Vbias={Vbias:.4f}Δ")
        return self._select_best_result(history)
    
    def compute_current_at_bias(self, Vbias, max_sidebands=None):
        """计算给定偏压下的电流（支持增量哈密顿量和缓存）"""
        if max_sidebands is None:
            max_sidebands = self.max_sidebands
            
        # 使用缓存：对于相同的Vbias和max_sidebands，避免重复构建哈密顿量切片
        cache_key = (Vbias, max_sidebands)
        if cache_key in self.ham_slice_cache:
            g0_inv_FKs, hop_FKs = self.ham_slice_cache[cache_key]
        else:
            ham = self.ham_builder.construct_hamiltonian(0)
            g0_inv_FKs, hop_FKs = self.ham_builder.build_slice_ham(Vbias, ham, max_sidebands)
            self.ham_slice_cache[cache_key] = (g0_inv_FKs, hop_FKs)
        
        N_SC = self.params['N_SC']
        N_junction = self.params['N_junction']
        
        # 生成能量积分网格
        if Vbias < 0.12:
            max_points_ = 21
        else:
            max_points_ = self.omega_points
            
        omega_values = self.generate_uniform_grid(
            start = 0 * Vbias,
            end = 1 * Vbias,
            spacing = 0.0025 * Vbias,  # 
            max_points =  max_points_
        )

        # 使用增量计算
        # 使用部分函数固定参数
        compute_func = functools.partial(
            self.gf_calculator._compute_current_at_omega,
            lead_type='finite',
            region_type='junction',
            ham_builder=self.ham_builder,
            max_sidebands=max_sidebands,
            Vbias=Vbias,
            N_SC=N_SC,
            N_junction=N_junction,
            g0_inv_FKs=g0_inv_FKs,
            hop_FKs=hop_FKs
        )
        
        # 并行计算
        results = Parallel(n_jobs=self.job_parallel[1])(
            delayed(compute_func)(omega) for omega in omega_values)
        
        return np.trapz(np.real(results), omega_values)/0.3  #/ (2 * np.pi)
    
    def _get_adaptive_config(self, Vbias):
        """获取自适应配置（添加默认配置）"""
        delta = self.params['delta']
        ratio = Vbias / delta
        
        # 查找匹配的区域配置
        for config in self.params['adaptive_regions']:
            if config[0] <= ratio < config[1]:
                return config[2], config[3], config[4]
        
        # 默认配置
        return 10, 4, 50
    
    def _select_best_result(self, history):
        """从未收敛的历史中选择最佳结果"""
        if len(history) < 2:
            return history[-1][1], history[-1][0]
        
        # 计算所有连续点之间的变化
        changes = []
        for i in range(1, len(history)):
            prev_N, prev_I = history[i-1]
            curr_N, curr_I = history[i]
            delta = abs(curr_I - prev_I)
            rel_delta = delta / (abs(prev_I) + 1e-6)
            changes.append((rel_delta, delta, i))
        
        # 找到变化最小的点
        best_idx = min(changes, key=lambda x: x[0])[2]
        best_N, best_I = history[best_idx]
        
        logging.info(f"选择最佳结果: {best_N} 边带, 电流={best_I:.6f}nA")
        return best_I, best_N

class JJPlotter:
    @staticmethod
    def plot_single_result(metadata, data, output_dir):
        """绘制单次计算结果"""
        calc_type = metadata["calculation"]
        fig = plt.figure(figsize=(10, 8))
        
        if calc_type == "DC_IV":
            if len(data) == 3:  # 包含边带数信息
                Vbias, current, sideband = data
                # 双面板图
                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10), sharex=True)
                
                # 电流-电压曲线
                ax1.plot(Vbias, current, 'r-', lw=2)
                ax1.set_ylabel('Current (nA)', fontsize=14)
                ax1.set_title('DC Current-Voltage Characteristic', fontsize=16)
                ax1.grid(True)
                
                # 边带数-电压曲线
                ax2.plot(Vbias, sideband, 'b-', lw=2)
                ax2.set_xlabel('Bias Voltage (meV)', fontsize=14)
                ax2.set_ylabel('Sideband Number', fontsize=14)
                ax2.set_title('Converged Sideband Numbers', fontsize=16)
                ax2.grid(True)
            else:
                Vbias, current = data
                plt.plot(Vbias, current, '-', lw=2, color='r')
                plt.xlabel('Bias Voltage (meV)', fontsize=14)
                plt.ylabel('Current (nA)', fontsize=14)
                plt.title('DC Current-Voltage Characteristic', fontsize=16)
                plt.grid(True)
        
        elif calc_type == "DC_IV_Bsweep":
            if len(data) == 3:  # 包含边带数信息
                B, current, sideband = data
                # 双面板图
                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10), sharex=True)
                
                # 电流-电压曲线
                ax1.plot(B, current, 'r-', lw=2, marker='.', markersize =16)
                ax1.set_ylabel('Current (nA)', fontsize=14)
                ax1.set_title('DC Current-Voltage Characteristic', fontsize=16)
                ax1.grid(True)
                
                # 边带数-电压曲线
                ax2.plot(B, sideband, 'b-', lw=2)
                ax2.set_xlabel('zeeman (meV)', fontsize=14)
                ax2.set_ylabel('Sideband Number', fontsize=14)
                ax2.set_title('Converged Sideband Numbers', fontsize=16)
                ax2.grid(True)
            else:
                B, current= data
                plt.plot(B, current, 'b-o', lw=2)
                plt.xlabel('Magnetic Field', fontsize=14)
                plt.ylabel('Current (nA)', fontsize=14)
                plt.title(f'DC Current vs Magnetic Field @ Vbias={metadata["fixed_Vbias"]} meV', fontsize=16)
                plt.grid(True)
        
        # 创建文件名并保存
        common_timestamp = metadata.get("common_timestamp", datetime.now().strftime("%Y%m%d_%H%M%S"))
        filename = f"{calc_type}_plot_{common_timestamp}.png"
        filepath = os.path.join(output_dir, filename)
        
        plt.tight_layout()
        plt.savefig(filepath, dpi=300)
        plt.close()
        
        return filepath

class PathManager:
    def __init__(self, base_dir="results", task_name=None):
        self.base_dir = base_dir
        self.task_name = task_name or datetime.now().strftime("%Y%m%d_%H%M%S")
        self.task_dir = os.path.join(self.base_dir, self.task_name)
        
        # 创建必要的目录
        self._create_directories()
    
    def _create_directories(self):
        """创建所有必要的目录"""
        os.makedirs(self.task_dir, exist_ok=True)
        os.makedirs(self.get_raw_data_dir(), exist_ok=True)
        os.makedirs(self.get_plots_dir(), exist_ok=True)
    
    def get_raw_data_dir(self):
        return os.path.join(self.task_dir, "raw_data")
    
    def get_plots_dir(self):
        return os.path.join(self.task_dir, "plots")
    
    def get_task_metadata_path(self):
        return os.path.join(self.task_dir, "task_metadata.json")
    
    def save_task_metadata(self, params):
        """保存任务元数据"""
        metadata = {
            "task_name": self.task_name,
            "start_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "parameters": self._filter_metadata(params)
        }
        
        with open(self.get_task_metadata_path(), 'w') as f:
            json.dump(metadata, f, indent=4)
        
        return self.get_task_metadata_path()
    
    def _filter_metadata(self, params):
        """过滤元数据，只保留相关参数"""
        # 创建副本
        filtered = params.copy()
        
        # 移除大数组
        if 'disorder_distribution' in filtered:
            del filtered['disorder_distribution']
        
        return filtered

def update_parameters(user_params=None):
    """确保参数正确更新"""
    # get default
    params = get_default_parameters()
    
    # get a copy
    from copy import deepcopy
    updated_params = deepcopy(params)
    
    # update it then
    if user_params is not None:
        for key, value in user_params.items():
            # 特殊处理嵌套字典
            if key == 'adaptive_regions' and isinstance(value, list):
                updated_params[key] = deepcopy(value)
            if key == 'disorder_type':
                valid_regions = ['gaussian', 'smooth', 'random_typeI', 'random_typeII', 'nonhermitian', 'none', 'from_file']
                if value not in valid_regions:
                    raise ValueError(f"Invalid disorder_type: {value}. Must be one of {valid_regions}")
            elif key in updated_params:
                updated_params[key] = value
            else:
                logging.warning(f"Ignoring unknown parameter: {key}")
    
    # recording ...
    logging.info("Final simulation parameters:")
    
    for key, value in updated_params.items():
        logging.info(f"  {key}: {value}")
    
    return updated_params

def get_default_parameters():
    """ """
    return {
        'recursion_depth': 100,
        'eta': 1e-3,
        
        'N_SC': 150,
        'N_junction': 2,
        't': 12.7,
        'delta': 0.3,
        'mu': 0.0,
        'mu_lead': 0.0,
        'B': 0,
        'alpha': 4.0,
        
        'v_tau': 0.842,        
        'max_sidebands': 10,
        'Vbias_max_ratio': 3.0,
        'fixed_Vbias': 0.1,  # 固定偏压值 (meV)
        'B_max': 0.0,       # 最大磁场强度
        
        # can set them to be the largest allowed
        'omega_points': 2000,
        'Vbias_points': 300,
        'B_points': 300,

        # disorder参数
        'disorder_type': 'from_file',
        'disorder_file': 'disorder_data_test_Nd=50_ld=10_v0=2Delta.csv',
        'disorder_strength':0.0,
        
        # adaptive params 
        'adaptive_iv': True,
        'rel_tol': 0.02,
        'abs_tol':0.01,
        'adaptive_max_N': 100,
        'adaptive_method': 'advanced', # 'dynamic'
        'adaptive_regions': [
            (0.0, 0.025, 50, 10, 220),
            (0.025, 0.05, 40, 2, 80),
            (0.05, 0.1, 38, 2, 70),    # low vbias：start_N=30, increase_N = 4, end_Nmax = 80
            (0.1, 0.2, 20, 2, 50),    
            (0.2, 0.4, 10, 2, 30),     
            (0.4, 0.7, 8, 1, 20),     
            (0.7, 1.0, 6, 1, 12),     
            (1.0, 6.0, 4, 1, 10)      
        ],   # all in ratio to self. delta
        
        'job_parallel': [101, 1],
        'output_dir': 'results'     ## should be updated every time
    }

def run_dc_iv_test(use_sparse=True, max_sidebands=5):
    """运行DC_IV测试"""
    # 获取默认参数
    params = get_default_parameters()
    
    # 设置关键参数
    params.update({
        'N_SC': 150,           # 超导区长度
        'N_junction': 2,       # 结区长度
        'delta': 0.3,          # 超导能隙 (meV)
        'alpha': 4.0,          # 自旋轨道耦合强度
        'v_tau': 0.822,         # 隧穿强度
        'B': 0.5,              # 磁场强度
        'mu_lead': 0.0,        # 引线化学势
        'mu': 0.0,             # 化学势
        'max_sidebands': max_sidebands,  # 最大边带数
        'omega_points': 501,    # 能量点数（减少以加速测试）
        'Vbias_points': 300,    # 偏压点数（减少以加速测试）
        'disorder_strength':2.0,
        
        'job_parallel': [161,1],  # 并行设置 [外层并行, 内层并行]
        'use_sparse': use_sparse,  # 是否使用稀疏矩阵
        'sparse_threshold': 0.3,  # 稀疏度阈值
        'sparse_depth_limit': 5,  # 最大稀疏递归深度
        'adaptive_iv': True,  # 禁用自适应计算
        'recursion_depth': 50, # 减少递归深度以加速测试
        'fixed_Vbias': 0.01*0.3,  # 固定偏压 0.15 meV
        'B_max': 2.0,         # 扫描磁场范围 0-1.5
        'B_points': 11,       # 磁场点数
    })
    # data set order: vtau = 0.842, B=0, 0.5, 2.0; vtau = 1.00, B=2.0, 0.5, 0.0
    
    # 创建路径管理器
    path_manager = PathManager(base_dir="gpu_results", task_name="dc_iv_test_Dec16th")
    
    # 创建求解器
    solver = JosephsonJunctionSolver(params, path_manager)
    
    try:
        # 运行DC_IV计算
        start_time = time.time()
        solver.run_calculation("DC_IV") #("DC_IV") #("DC_IV_Bsweep")
        compute_time = time.time() - start_time
        
        # 打印结果摘要
        print("\n" + "="*50)
        print(f"DC_IV计算完成，耗时: {compute_time:.2f}秒")

    except Exception as e:
        logging.error(f"DC_IV计算失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

if __name__ == "__main__":
    # 测试单个DC_IV计算
    print("运行DC_IV计算测试...")
    run_dc_iv_test(use_sparse=True, max_sidebands=15)

2026-01-23 11:28:18,342 - ERROR - Failed to load disorder from file disorder_data_test_Nd=50_ld=10_v0=2Delta.csv: [Errno 2] No such file or directory: 'disorder_data_test_Nd=50_ld=10_v0=2Delta.csv'
2026-01-23 11:28:18,343 - INFO - HamiltonianBuilder initialized, disorder type: from_file, Sparse mode: True
2026-01-23 11:28:18,344 - INFO - GreenFunctionCalculator initialized, Sparse mode: True
2026-01-23 11:28:18,344 - INFO - JosephsonJunctionSolver initialized
2026-01-23 11:28:18,345 - INFO - Superconductor: 150 sites, Junction: 2 sites
2026-01-23 11:28:18,345 - INFO - Recursion depth: 50, Max sidebands: 15
2026-01-23 11:28:18,347 - INFO - Starting DC IV calculation


运行DC_IV计算测试...


KeyboardInterrupt: 